# Swiss Law Citation Retrieval Pipeline — Notebook-Clean Edition

This version is structured for notebook execution with **explicit path config in the first cells**, original source cells for the pipeline stages, no CLI `__main__` blocks, and notebook-friendly runner cells.


## How this notebook is organized

- **Config cells** set paths and runtime parameters.
- **Utility/source cells** expose the original project modules without CLI entrypoints.
- **Stage runner cells** call functions sequentially the way a notebook should.
- **Pipeline and submission cells** are included at the end for one-shot orchestration or validation.
- **No notebook path-registry cell** is required; paths are set directly in config.


## Project Overview

# Swiss Law Citation Retrieval — Revised Pipeline
## BM25-First Architecture with LegalMALR-Adapted Multi-Agent System

---

## Quick Start

```powershell
# 1. Setup (one-time)
cd E:\swiss-law-pipeline
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
pip install -r requirements.txt

# 2. Build indexes (one-time, ~4-7 hours, CPU only)
python stage0_build_index.py

# 3. Run full pipeline on val set (~40-80 min, needs GPU)
python pipeline.py --split val

# 4. Generate test submission
python pipeline.py --split test

# 5. Validate submission
python submit.py --validate submissions/submission_test.csv
```

---

## Architecture Overview

```
Stage 0 (OFFLINE, one-time)          Stage 1 (GPU)
┌──────────────────────────┐         ┌──────────────────────────┐
│ BM25 index (171K+2M docs)│         │ Qwen3-4B Query Analysis  │
│ Citation graph (Art↔Case) │    ──→  │ Extract explicit cites    │
│ Lookup tables             │         │ Classify law domains      │
└──────────────────────────┘         │ Generate DE search queries│
                                     └───────────┬──────────────┘
                                                 │
                                     ┌───────────▼──────────────┐
                                     │ Stage 2: MAS Retrieval   │
                                     │ Planner → Agent → BM25   │
                                     │ 2-4 iterations           │
                                     │ ~200+ candidates         │
                                     └───────────┬──────────────┘
                                                 │
                                     ┌───────────▼──────────────┐
                                     │ Stage 3: Graph Expansion │
                                     │ Art. → citing BGE cases  │
                                     │ BGE → cited Art. statutes│
                                     │ CPU only, <1s            │
                                     └───────────┬──────────────┘
                                                 │
                                     ┌───────────▼──────────────┐
                                     │ Stage 4: LLM Reranker    │
                                     │ Select relevant from pool│
                                     │ + Direct citation gen    │
                                     └───────────┬──────────────┘
                                                 │
                                     ┌───────────▼──────────────┐
                                     │ Stage 5: Verify + Score  │
                                     │ Normalize citations      │
                                     │ Confidence scoring       │
                                     │ F1 threshold tuning      │
                                     └──────────────────────────┘
```

**GPU usage**: Only Qwen3-4B (Q4/NF4, ~2.8GB VRAM) loaded ONCE, serves all LLM tasks.

---

## Running Each Stage Individually

Each stage reads from the previous stage's checkpoint and can be run independently.

### Stage 0: Build Indexes (CPU, one-time)
```powershell
python stage0_build_index.py              # build everything (~4-7 hours)
python stage0_build_index.py --bm25-only  # just BM25 index (~3-5 hours)
python stage0_build_index.py --graph-only # just citation graph (~1-2 hours)
python stage0_build_index.py --lookup-only # just lookup tables (~1 minute)
```
**Outputs**: `index/bm25_v2_index.pkl`, `index/citation_graph.pkl`, `index/citation_lookup.pkl`
**Expects**: ~16GB RAM for BM25 build. CPU only.

### Stage 1: Query Analysis (GPU, ~7 min)
```powershell
python stage1_query_analysis.py                           # val set
python stage1_query_analysis.py --split test              # test set
python stage1_query_analysis.py --query "A claimant..."   # single query
python stage1_query_analysis.py --backend anthropic       # use Claude API
```
**Outputs**: `checkpoints/stage1_{split}.json`
**Expects**: GPU with Qwen3-4B, or API key for cloud backend.
**Per query**: Extracts explicit citations (regex), classifies legal domains,
identifies legal issues, generates 3-5 German search queries.

### Stage 2: Multi-Agent Sparse Retrieval (GPU+CPU, ~20 min)
```powershell
python stage2_mas_retrieval.py                    # val set
python stage2_mas_retrieval.py --split test        # test set
python stage2_mas_retrieval.py --max-iter 3       # fewer iterations (faster)
```
**Outputs**: `checkpoints/stage2_{split}.json`
**Expects**: BM25 index + Qwen3-4B + Stage 1 checkpoint.
**Per query**: Runs 2-4 iterations of Planner → Agent → BM25 loop.
Agents: Rewrite (EN→DE), Supplement (implicit conditions), Decompose (sub-issues),
Supportive (procedural), CrossRef (constitutional). Each generates DE queries → BM25.
Expected: ~200+ candidates per query.

### Stage 3: Citation Graph Expansion (CPU, ~30 sec)
```powershell
python stage3_graph_expansion.py                  # val set
python stage3_graph_expansion.py --split test     # test set
```
**Outputs**: `checkpoints/stage3_{split}.json`
**Expects**: Citation graph + Stage 2 checkpoint.
**Per query**: For each statute found, adds top 5 citing court decisions.
For each court decision, adds all cited statutes.
Expected: +5-10% recall on case law citations.

### Stage 4: LLM Reranker + Direct Gen (GPU, ~10 min)
```powershell
python stage4_llm_reranker.py                     # val set
python stage4_llm_reranker.py --split test        # test set
python stage4_llm_reranker.py --skip-direct-gen   # skip memory-based generation
```
**Outputs**: `checkpoints/stage4_{split}.json`
**Expects**: Qwen3-4B + Stage 1 + Stage 3 checkpoints.
**Per query**: LLM selects relevant citations from expanded pool (precision),
then generates additional citations from memory (recall supplement).
No chain-of-thought in output (LegalMALR finding).

### Stage 5: Verify + Score + Threshold (CPU, ~30 sec)
```powershell
python stage5_verify_and_score.py                  # val: tune threshold + submit
python stage5_verify_and_score.py --split test     # test: apply threshold + submit
python stage5_verify_and_score.py --threshold 0.20 # override threshold
```
**Outputs**: `submissions/submission_{split}.csv`, `checkpoints/stage5_{split}.json`
**Expects**: Lookup tables + Stage 4 checkpoint.
**Operations**:
  - 5A: Normalize citations, check existence, drop hallucinations
  - 5B: Confidence score = weighted sum of source signals
  - 5C: Sweep thresholds 0.05-0.95 to maximize macro-F1 (val only)
  - Expected optimal threshold: 0.15-0.30 (low = inclusive)

---

## Key Design Decisions

### Why BM25, not Dense Retrieval?
Your empirical results: BM25 R@50=0.73 vs Dense R@100=0.24.
Three reasons:
1. **Exact citation matching**: Queries contain "Art. 221 Abs. 1 StPO" — BM25 matches exactly
2. **Cross-lingual gap**: EN queries vs DE corpus — dense models lose precision
3. **High recall@50 needed**: Val queries need 10-47 citations each

### Why Multi-Agent System (MAS)?
LegalMALR shows that diverse query reformulations dramatically improve recall.
Each agent targets a different gap:
- Rewrite: vocabulary mismatch (EN→DE legal terms)
- Decompose: multi-issue queries need independent sub-queries
- Supportive: procedural articles missed by topic-focused retrieval
- CrossRef: constitutional provisions and general clauses

### Why Citation Graph?
40% of val citations are case law, but BM25 struggles with case law retrieval
(case numbers are sparse signals). The graph provides mechanical lookup:
Art. → which BGE decisions cite it → add those to pool.

---

## Expected Performance

| Component                      | Estimated Recall |
|-------------------------------|-----------------|
| BM25 baseline (R@50)          | 0.73            |
| + MAS multi-agent reformulation | +0.08-0.12      |
| + Citation graph expansion    | +0.05-0.10      |
| + Explicit citation extraction| +0.02-0.05      |
| + LLM direct generation       | +0.03-0.05      |
| **Combined recall estimate**  | **0.85-0.92**   |
| After LLM reranking (precision)| 0.65-0.75      |
| **Estimated F1**              | **0.70-0.80**   |

---

## File Structure

```
swiss-law-pipeline/
├── README.md                          # This file
├── requirements.txt                   # Python dependencies
├── pipeline.py                        # Master orchestrator (all stages)
├── submit.py                          # Competition submission generator
│
├── stage0_build_index.py              # STAGE 0: Offline index building
├── stage1_query_analysis.py           # STAGE 1: LLM query analysis
├── stage2_mas_retrieval.py            # STAGE 2: Multi-agent sparse retrieval
├── stage3_graph_expansion.py          # STAGE 3: Citation graph expansion
├── stage4_llm_reranker.py             # STAGE 4: LLM reranking + direct gen
├── stage5_verify_and_score.py         # STAGE 5: Verify + score + threshold
│
├── data/
│   ├── data_paths.py                  # Central path configuration
│   └── __init__.py
│
├── agent/
│   ├── llm_backend.py                 # Qwen3-4B singleton loader
│   ├── verifier.py                    # Citation normalization + verification
│   ├── __init__.py
│   └── prompts/
│       ├── query_analyzer_prompt.txt  # Stage 1 system prompt
│       ├── mas_prompts.py             # Stage 2 MAS agent prompts
│       ├── reranker_prompt.txt        # Stage 4A reranking prompt
│       └── direct_gen_prompt.txt      # Stage 4B direct generation prompt
│
├── retrieval/
│   ├── sparse_retriever.py            # BM25 search (primary engine)
│   ├── graph_retriever.py             # Citation graph traversal
│   ├── explicit_citations.py          # Regex citation extraction
│   └── __init__.py
│
├── scoring/
│   ├── confidence.py                  # Composite scoring + F1 optimization
│   └── __init__.py
│
├── indexing/
│   ├── build_bm25_index.py            # BM25 index builder
│   ├── build_citation_graph.py        # Citation graph builder
│   ├── build_lookup_tables.py         # Normalization table builder
│   └── __init__.py
│
├── configs/
│   └── default.json                   # Pipeline configuration
│
├── index/                             # Built indexes (gitignored)
├── checkpoints/                       # Per-stage outputs (gitignored)
├── submissions/                       # Final CSVs (gitignored)
└── models/                            # Local model files (gitignored)
```

---

## Troubleshooting

**Out of GPU memory**: Qwen3-4B NF4 needs ~2.8GB. Close other GPU apps.
If still OOM, try `--backend anthropic` to use cloud API instead.

**BM25 index takes too long**: The 2.47M court rows are the bottleneck.
Ensure you have ~16GB RAM free. Consider reducing COURT_TEXT_LIMIT in
`indexing/build_bm25_index.py`.

**Low F1 on val**: Check each stage's checkpoint to diagnose:
1. Stage 1: Are DE queries reasonable? Are domains correctly classified?
2. Stage 2: Is the candidate pool large enough (>100)? Check retrieval_log.
3. Stage 3: Are graph expansions finding new case law?
4. Stage 4: Is the reranker too aggressive (filtering too many)?
5. Stage 5: Try lowering the threshold (e.g., --threshold 0.10).

**Resuming after errors**: All stages save incrementally. Just re-run the
same command and it will skip already-processed queries.


## Config

These cells define the notebook runtime explicitly. Set the actual folders here first, then run the stage cells below.

The notebook uses visible path variables like `PROJECT_ROOT`, `RAW_DATA_DIR`, `INDEX_DIR`, `CHECKPOINTS_DIR`, and `SUBMISSIONS_DIR`. You do not need a separate notebook cell that imports or reloads a path registry.


In [ ]:
from pathlib import Path
import os
import sys
import json
import importlib

# -----------------------------------------------------------------------------
# Notebook-visible paths (edit these first)
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path("/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline").resolve()

# Raw inputs. Update this if your competition CSV/Parquet/JSONL files live elsewhere.
RAW_DATA_DIR = Path(os.environ.get("SWISS_DATA_DIR", str(PROJECT_ROOT / "data"))).resolve()

# Runtime artifact/output folders
INDEX_DIR = (PROJECT_ROOT / "index").resolve()
CHECKPOINTS_DIR = (PROJECT_ROOT / "checkpoints").resolve()
SUBMISSIONS_DIR = (PROJECT_ROOT / "submissions").resolve()
MODELS_DIR = (PROJECT_ROOT / "models").resolve()

for folder in [INDEX_DIR, CHECKPOINTS_DIR, SUBMISSIONS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Export the same paths for project modules that import data.data_paths
os.environ["SWISS_PIPELINE_ROOT"] = str(PROJECT_ROOT)
os.environ["SWISS_DATA_DIR"] = str(RAW_DATA_DIR)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT    =", PROJECT_ROOT)
print("RAW_DATA_DIR    =", RAW_DATA_DIR)
print("INDEX_DIR       =", INDEX_DIR)
print("CHECKPOINTS_DIR =", CHECKPOINTS_DIR)
print("SUBMISSIONS_DIR =", SUBMISSIONS_DIR)
print("Working dir     =", Path.cwd())


### Path behavior in this notebook

- The notebook uses the explicit path variables above.
- Original project modules remain unchanged and pick up those same folders from the environment.
- There is no separate notebook sync or reload step.


In [ ]:
# Optional dependency install for a fresh kernel:
# %pip install -r r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/requirements.txt"
#
# Project requirements (reference):
requirements_text = """# =============================================================================
# Swiss Law Pipeline — Revised (BM25-first, LegalMALR-adapted)
# Python 3.12  |  CUDA 12.1 (RTX 4050 6GB)
#
# Install:
#   python -m venv .venv
#   .venv\Scripts\activate            (Windows)
#   source .venv/bin/activate         (Linux)
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
#   pip install -r requirements.txt
# =============================================================================

# --- Retrieval (BM25 only — dense retrieval dropped) ---
rank-bm25==0.2.2

# --- LLM: Qwen3-4B 4-bit (the single GPU model for all stages) ---
transformers>=4.45.0
bitsandbytes>=0.44.0
accelerate>=1.0.0
huggingface_hub>=0.25.0
safetensors>=0.4.0

# --- PyTorch (install separately with CUDA wheel) ---
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# torch>=2.4.0

# --- LLM APIs (optional, for cloud ensemble) ---
anthropic>=0.40.0
openai>=1.50.0

# --- Data ---
pandas>=2.0.0
numpy>=1.26.0
pyarrow>=14.0.0

# --- Utilities ---
tqdm>=4.66.0
deep-translator>=1.11.0
"""
print(requirements_text)


In [ ]:
# Editable runtime settings for notebook execution
SPLIT = "test"                  # "train", "val", or "test"
BACKEND = "local"               # "local", "anthropic", or "openai"
START_STAGE = 1                 # First stage to run in pipeline mode
STAGE3_GOLD_PRIOR_MODE = "off"  # "off", "train", or "trainval"

# Optional stage-specific knobs
STAGE2_MAX_ITERATIONS = 4
STAGE4_SKIP_DIRECT_GEN = False
STAGE4B_TOP_N = None            # None -> stage default
STAGE5_THRESHOLD = None         # None -> stage default

print({
    "SPLIT": SPLIT,
    "BACKEND": BACKEND,
    "START_STAGE": START_STAGE,
    "STAGE3_GOLD_PRIOR_MODE": STAGE3_GOLD_PRIOR_MODE,
})


In [ ]:
DEFAULT_CONFIG_PATH = PROJECT_ROOT / "configs" / "default.json"
default_config = json.loads(DEFAULT_CONFIG_PATH.read_text(encoding="utf-8"))
default_config


### `configs/default.json`

```json
{
    "_comment": "Revised pipeline config — BM25-first, LegalMALR-adapted MAS architecture",

    "stage1_backend":  "local",
    "stage1_model":    null,

    "bm25_top_k":      100,
    "graph_top_k":     50,

    "mas_max_iterations": 4,
    "mas_bm25_top_k":    30,

    "reranker_backend": "local",
    "reranker_batch_size": 20,

    "confidence_weights": {
        "explicit_from_query": 0.40,
        "bm25_top10":          0.25,
        "citation_graph":      0.15,
        "llm_reranker":        0.15,
        "llm_direct_gen":      0.05
    },

    "threshold": 0.15
}

```

### Notebook execution helpers

These wrappers are the notebook-native entrypoints. They call the stage functions directly instead of relying on CLI `main` blocks.


In [ ]:
from pathlib import Path

def reload_module(module_name: str):
    module = importlib.import_module(module_name)
    return importlib.reload(module)

def run_stage0_all():
    stage0 = reload_module("stage0_build_index")
    stage0.build_bm25()
    stage0.build_citation_graph()
    stage0.build_lookup_tables()
    stage0.build_citation_signals()
    stage0.build_reference_graph_v2()
    stage0.build_gold_cocitation_prior()

def run_stage0_selected(
    bm25: bool = False,
    graph: bool = False,
    lookup: bool = False,
    signals: bool = False,
    refgraph: bool = False,
    goldprior: bool = False,
    goldprior_trainval: bool = False,
):
    stage0 = reload_module("stage0_build_index")
    if bm25:
        stage0.build_bm25()
    if graph:
        stage0.build_citation_graph()
    if lookup:
        stage0.build_lookup_tables()
    if signals:
        stage0.build_citation_signals()
    if refgraph:
        stage0.build_reference_graph_v2()
    if goldprior:
        stage0.build_gold_cocitation_prior()
    if goldprior_trainval:
        stage0.build_gold_cocitation_prior_trainval()

def run_stage1(split=None, backend=None, model=None, resume=False):
    split = SPLIT if split is None else split
    backend = BACKEND if backend is None else backend
    stage1 = reload_module("stage1_query_analysis")
    return stage1.run_batch(split, backend=backend, model=model, resume=resume)

def run_stage2(split=None, backend=None, max_iterations=None, resume=False):
    split = SPLIT if split is None else split
    backend = BACKEND if backend is None else backend
    max_iterations = STAGE2_MAX_ITERATIONS if max_iterations is None else max_iterations
    stage2 = reload_module("stage2_mas_retrieval")
    return stage2.run_batch(split, backend=backend, max_iterations=max_iterations, resume=resume)

def run_stage3(split=None, resume=False, gold_prior_mode=None, gold_prior_rules=None):
    split = SPLIT if split is None else split
    gold_prior_mode = STAGE3_GOLD_PRIOR_MODE if gold_prior_mode is None else gold_prior_mode
    stage3 = reload_module("stage3_graph_expansion")
    return stage3.run_batch(
        split,
        resume=resume,
        gold_prior_mode=gold_prior_mode,
        gold_prior_rules=gold_prior_rules,
    )

def run_stage4(split=None, backend=None, skip_direct_gen=None, resume=False):
    split = SPLIT if split is None else split
    backend = BACKEND if backend is None else backend
    skip_direct_gen = STAGE4_SKIP_DIRECT_GEN if skip_direct_gen is None else skip_direct_gen
    stage4 = reload_module("stage4_llm_reranker")
    return stage4.run_batch(
        split,
        backend=backend,
        skip_direct_gen=skip_direct_gen,
        resume=resume,
    )

def run_stage4b(split=None, top_n=None):
    split = SPLIT if split is None else split
    stage4b = reload_module("stage4b_cross_encoder")
    if top_n is None:
        return stage4b.run_batch(split)
    return stage4b.run_batch(split, top_n=top_n)

def run_stage5(split=None, threshold=None, confidence_weights=None):
    split = SPLIT if split is None else split
    threshold = STAGE5_THRESHOLD if threshold is None else threshold
    stage5 = reload_module("stage5_verify_and_score")
    if threshold is None and confidence_weights is None:
        return stage5.run_batch(split)
    return stage5.run_batch(
        split,
        threshold=threshold,
        confidence_weights=confidence_weights,
    )

def run_full_pipeline(split=None, start_stage=None, backend=None, stage3_gold_prior_mode=None):
    split = SPLIT if split is None else split
    start_stage = START_STAGE if start_stage is None else start_stage
    backend = BACKEND if backend is None else backend
    stage3_gold_prior_mode = (
        STAGE3_GOLD_PRIOR_MODE if stage3_gold_prior_mode is None else stage3_gold_prior_mode
    )
    pipe = reload_module("pipeline")
    return pipe.run_pipeline(
        split,
        start_stage=start_stage,
        backend=backend,
        stage3_gold_prior_mode=stage3_gold_prior_mode,
    )

def get_submission_path(split=None):
    split = SPLIT if split is None else split
    return SUBMISSIONS_DIR / f"submission_{split}.csv"

def get_checkpoint_path(stage_name: str, split=None):
    split = SPLIT if split is None else split
    return CHECKPOINTS_DIR / f"{stage_name}_{split}.json"

def validate_generated_submission(split=None):
    split = SPLIT if split is None else split
    submit_mod = reload_module("submit")
    return submit_mod.validate_submission(get_submission_path(split))

print("Notebook helper functions are ready.")


### Runtime artifact locations
These notebook-native helpers make the output files explicit before you run the stages.


In [ ]:
SUBMISSION_FILE = get_submission_path(SPLIT)
STAGE1_CHECKPOINT = get_checkpoint_path("stage1", SPLIT)
STAGE2_CHECKPOINT = get_checkpoint_path("stage2", SPLIT)
STAGE3_CHECKPOINT = get_checkpoint_path("stage3", SPLIT)
STAGE4_CHECKPOINT = get_checkpoint_path("stage4", SPLIT)
STAGE5_CHECKPOINT = get_checkpoint_path("stage5", SPLIT)

print("SUBMISSION_FILE  =", SUBMISSION_FILE)
print("STAGE1_CHECKPOINT =", STAGE1_CHECKPOINT)
print("STAGE5_CHECKPOINT =", STAGE5_CHECKPOINT)


## Shared Utilities

### `agent/llm_backend.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/agent/llm_backend.py"

"""
llm_backend.py -- Singleton Qwen3-4B loader for all pipeline LLM tasks.

Loads the model ONCE (~2.8GB VRAM as Q4/NF4) and provides a unified
generate() interface used by:
  - Stage 1: Query analysis
  - Stage 2: MAS agent reformulations
  - Stage 4: LLM reranking + direct citation generation

# DESIGN: One model, loaded once, serves all LLM functions.
#   The model stays in GPU memory across all stages — no loading/unloading.
#
# HOW TO RUN (standalone smoke-test):
#   cd E:\\swiss-law-pipeline
#   python agent/llm_backend.py
#   python agent/llm_backend.py --prompt "List 3 Swiss criminal law articles"
"""

import os
import sys
import json
import re
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent.parent))

# Singleton model instance
_MODEL = None
_TOKENIZER = None


def get_model_and_tokenizer():
    """
    Lazy-load Qwen3-4B with 4-bit NF4 quantization.
    Returns (model, tokenizer). Cached after first call.

    VRAM usage: ~2.7-2.8 GB with NF4 bfloat16 compute.
    """
    global _MODEL, _TOKENIZER
    if _MODEL is not None:
        return _MODEL, _TOKENIZER

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from data.data_paths import QWEN3_4B_PATH, QWEN3_4B_HF_ID

    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

    model_path = str(QWEN3_4B_PATH) if QWEN3_4B_PATH.exists() else QWEN3_4B_HF_ID
    print(f"  Loading Qwen3-4B from: {model_path}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    _TOKENIZER = AutoTokenizer.from_pretrained(model_path)
    _MODEL = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=bnb_config,
        device_map="auto",
    )
    _MODEL.eval()
    print(f"  Qwen3-4B loaded. VRAM: ~2.8GB")
    return _MODEL, _TOKENIZER


def generate(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 2048,
    temperature: float = 0.6,
    top_k: int = 20,
    top_p: float = 0.95,
    enable_thinking: bool = False,
) -> str:
    """
    Generate a response from Qwen3-4B.

    Args:
        system_prompt: System message (role definition).
        user_prompt:   User message (the actual query/task).
        max_new_tokens: Max generation length.
        temperature:   Sampling temperature (0.6 = focused but not greedy).
        top_k/top_p:   Nucleus sampling params.
        enable_thinking: If True, allows <think> blocks for chain-of-thought.
                        Stage 1 uses True, Stage 4 reranker uses False.

    Returns:
        Generated text string (with <think> blocks stripped if enable_thinking=False).
    """
    import torch

    model, tokenizer = get_model_and_tokenizer()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        out[0][input_len:],
        skip_special_tokens=True,
    )

    # Explicitly free GPU tensors so VRAM doesn't fragment across calls.
    del inputs, out
    torch.cuda.empty_cache()

    return generated


def generate_json(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 2048,
    enable_thinking: bool = False,
) -> dict:
    """
    Generate and parse JSON from Qwen3-4B.
    Handles ```json ... ``` wrapping and malformed JSON gracefully.
    On CUDA OOM: clears cache and retries once with halved max_new_tokens.
    """
    import torch
    try:
        raw = generate(system_prompt, user_prompt, max_new_tokens=max_new_tokens,
                       enable_thinking=enable_thinking)
    except RuntimeError as e:
        if "out of memory" in str(e).lower() or "CUDA" in str(e):
            print(f"\n  [OOM] CUDA out of memory — clearing cache and retrying with {max_new_tokens // 2} tokens...")
            torch.cuda.empty_cache()
            try:
                raw = generate(system_prompt, user_prompt,
                               max_new_tokens=max_new_tokens // 2,
                               enable_thinking=enable_thinking)
            except RuntimeError as e2:
                print(f"  [OOM] Retry also failed: {e2}")
                torch.cuda.empty_cache()
                return {}
        else:
            raise
    return parse_json_response(raw)


def parse_json_response(raw_text: str) -> dict:
    """Extract JSON from LLM response, handling markdown wrapping."""
    raw_text = raw_text.strip()

    # Strip ```json ... ``` wrapping
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw_text, re.DOTALL)
    if m:
        raw_text = m.group(1)
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        # Fallback: find any JSON object
        m2 = re.search(r"\{.*\}", raw_text, re.DOTALL)
        if m2:
            try:
                return json.loads(m2.group(0))
            except json.JSONDecodeError:
                pass
    return {}


def call_anthropic(system_prompt: str, user_prompt: str,
                   model: str = "claude-sonnet-4-6") -> str:
    """Optional: cloud backend for ensemble."""
    import anthropic
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model=model, max_tokens=4096,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return msg.content[0].text


def call_openai(system_prompt: str, user_prompt: str,
                model: str = "gpt-4o") -> str:
    """Optional: cloud backend for ensemble."""
    from openai import OpenAI
    client = OpenAI()
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens=4096,
    )
    return resp.choices[0].message.content


### `agent/verifier.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/agent/verifier.py"

"""
verifier.py -- Citation verification and normalization.

Validates each citation against the corpus:
  - Normalize formatting (whitespace, abbreviation spacing)
  - Check existence in statutory corpus
  - Expand article-level to Abs.-level if needed
  - Drop hallucinated case citations not in corpus (with lenient fallback)

# HOW TO RUN (standalone smoke-test):
#   cd E:\\swiss-law-pipeline
#   python agent/verifier.py
#
# EXPECTS: index/citation_lookup.pkl (built by: python stage0_build_index.py)
"""

import re
from pathlib import Path

import sys
sys.path.insert(0, str(Path(__file__).parent.parent))
from data.data_paths import LOOKUP_PKL


# Corpus uses UPPERCASE abbreviations but gold standard uses mixed-case.
# This map converts corpus forms to gold-compatible forms.
_ABBREV_CASING = {
    'STGB': 'StGB', 'STPO': 'StPO', 'STBOG': 'StBOG', 'JSTG': 'JStG',
    'JSTPO': 'JStPO', 'SCHKG': 'SchKG', 'BANKG': 'BankG', 'BANKV': 'BankV',
    'ASYLG': 'AsylG', 'VWVG': 'VwVG', 'VSTG': 'VStG', 'VSTV': 'VStV',
    'BETMG': 'BetmG', 'GSCHG': 'GSchG', 'GSCHV': 'GSchV', 'MSCHG': 'MSchG',
    'MSCHV': 'MSchV', 'STHG': 'StHG', 'PARLG': 'ParlG', 'SEBG': 'SebG',
    'FINFRAG': 'FinfraG', 'FINFRAV': 'FinfraV', 'CHEMG': 'ChemG',
    'FAMZG': 'FamZG', 'ELEG': 'EleG', 'GWG': 'GwG', 'EPG': 'EpG',
    'EPV': 'EpV', 'STAHIG': 'StAhiG', 'STROMVG': 'StromVG',
    'STROMVV': 'StromVV', 'VSTRR': 'VStrR', 'HREGV': 'HRegV',
    'GEOIG': 'GeoIG', 'WAG': 'WaG', 'PATG': 'PatG', 'PRSG': 'PrSG',
    'DESG': 'DesG', 'ARG': 'ArG', 'TWWV': 'TwwV', 'PRHG': 'PrHG',
}


def _fix_abbrev_casing(c: str) -> str:
    """Convert corpus UPPERCASE abbreviations to gold-compatible mixed-case."""
    if not c.startswith("Art."):
        return c
    parts = c.split()
    if len(parts) >= 3:
        last = parts[-1]
        fixed = _ABBREV_CASING.get(last, last)
        if fixed != last:
            parts[-1] = fixed
            return " ".join(parts)
    return c


def normalize_citation(c: str) -> str:
    """Normalize whitespace, abbreviation spacing, and casing in a citation string."""
    c = re.sub(r"\s+",    " ",     str(c or "").strip())
    c = re.sub(r"Art\.\s*",  "Art. ",  c)
    c = re.sub(r"Abs\.\s*",  "Abs. ",  c)
    c = re.sub(r"lit\.\s*",  "lit. ",  c)
    c = re.sub(r"Ziff\.\s*", "Ziff. ", c)
    c = re.sub(r"BGE\s+",    "BGE ",   c)
    c = re.sub(r"\bE\.\s*",  "E. ",    c)
    c = _fix_abbrev_casing(c.strip())
    return c.strip()


def is_law_citation(c: str) -> bool:
    return bool(re.match(r"^Art\.\s+\d", c.strip()))


def is_case_citation(c: str) -> bool:
    return bool(
        re.match(r"^BGE\s+\d+", c.strip()) or
        re.match(r"^\d+[A-Z]_\d+/\d{4}", c.strip())
    )


class Verifier:
    def __init__(self, lookup_pkl: Path = LOOKUP_PKL, caselaw_ids: set[str] | None = None):
        from indexing.build_lookup_tables import CitationLookup
        self.lookup = CitationLookup(lookup_pkl)
        self.caselaw_ids = caselaw_ids or set()

    def verify_and_normalize(
        self,
        citations: list[str],
        expand_to_abs: bool = True,
    ) -> tuple[list[str], list[str]]:
        """
        Normalize and verify a list of citations.

        Returns (verified, dropped):
          verified = citations that passed verification
          dropped  = citations that failed (hallucinations / format errors)
        """
        verified = []
        dropped  = []

        for raw in citations:
            c = normalize_citation(raw)
            if not c:
                continue

            if is_law_citation(c):
                canon = self.lookup.normalize(c)
                if canon:
                    verified.append(_fix_abbrev_casing(canon))
                elif expand_to_abs:
                    expanded = self.lookup.expand_to_abs(c)
                    if expanded and expanded != [c]:
                        verified.extend(_fix_abbrev_casing(e) for e in expanded)
                    else:
                        # Try stripping lit./Ziff. to find parent Abs. or base Art.
                        # E.g. "Art. 221 Abs. 1 lit. b StPO" -> try "Art. 221 Abs. 1 StPO"
                        #      then "Art. 221 StPO"
                        kept = False
                        m = re.match(r"(Art\.\s+\d+[a-z]?)\s+(Abs\.\s+\d+\s*)?(lit\.\s+\w+\s*)?(Ziff\.\s+\d+\s?)?(.*)", c)
                        if m:
                            law_part = m.group(5).strip() if m.group(5) else ""
                            # Try Abs. level first (strip lit./Ziff.)
                            if m.group(2):
                                abs_cite = f"{m.group(1)} {m.group(2).strip()} {law_part}".strip()
                                if self.lookup.normalize(abs_cite):
                                    verified.append(c)
                                    kept = True
                            # Try base article (strip everything)
                            if not kept:
                                base = f"{m.group(1)} {law_part}".strip()
                                if self.lookup.normalize(base):
                                    verified.append(c)
                                    kept = True
                        if not kept:
                            dropped.append(c)
                else:
                    dropped.append(c)

            elif is_case_citation(c):
                if not self.caselaw_ids or c in self.caselaw_ids:
                    verified.append(c)
                else:
                    # Lenient: keep if format looks valid (Swiss court citation patterns)
                    if re.match(r"^BGE\s+\d{2,3}\s+[IVX]+\s+\d+(\s+E\.\s+[\d\.]+)?(\s+S\.\s+\d+)?$", c):
                        verified.append(c)
                    elif re.match(r"^\d+[A-Z]_\d+/\d{4}(\s+\d{2}\.\d{2}\.\d{4})?(\s+E\.\s+[\d\.]+)?$", c):
                        verified.append(c)
                    else:
                        dropped.append(c)
            else:
                dropped.append(c)

        # Deduplicate preserving order
        seen = set()
        final = []
        for c in verified:
            if c not in seen:
                seen.add(c)
                final.append(c)

        return final, dropped


### `retrieval/bm25_artifact.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/retrieval/bm25_artifact.py"

import math
import os
import pickle
from pathlib import Path

import numpy as np
from tqdm import tqdm


ARTIFACT_FORMAT = "chunked_bm25_v1"

try:
    BM25_CHUNK_SIZE = max(1000, int(os.getenv("SWISS_BM25_CHUNK_SIZE", "10000")))
except Exception:
    BM25_CHUNK_SIZE = 10000


def get_bm25_artifact_dir(index_path: Path) -> Path:
    return index_path.with_name(f"{index_path.stem}_parts")


class ChunkedBM25Okapi:
    """Lightweight in-memory BM25 object loaded from chunked artifact files."""

    def __init__(
        self,
        *,
        doc_freqs: list[dict[str, int]],
        doc_len: np.ndarray,
        idf: dict[str, float],
        avgdl: float,
        k1: float,
        b: float,
        epsilon: float,
    ):
        self.doc_freqs = doc_freqs
        self.doc_len = np.asarray(doc_len, dtype=np.float32)
        self.idf = idf
        self.avgdl = float(avgdl)
        self.k1 = float(k1)
        self.b = float(b)
        self.epsilon = float(epsilon)
        self.corpus_size = len(self.doc_freqs)
        self._norm = self.k1 * (1.0 - self.b + self.b * self.doc_len / self.avgdl)

    def get_scores(self, query):
        score = np.zeros(self.corpus_size, dtype=np.float32)
        for q in query:
            q_freq = np.fromiter(
                (doc.get(q, 0) for doc in self.doc_freqs),
                dtype=np.float32,
                count=self.corpus_size,
            )
            score += (self.idf.get(q) or 0.0) * (
                q_freq * (self.k1 + 1.0) / (q_freq + self._norm)
            )
        return score

    def get_batch_scores(self, query, doc_ids):
        doc_ids = list(doc_ids)
        assert all(di < len(self.doc_freqs) for di in doc_ids)
        score = np.zeros(len(doc_ids), dtype=np.float32)
        doc_len = self.doc_len[doc_ids]
        norm = self.k1 * (1.0 - self.b + self.b * doc_len / self.avgdl)
        for q in query:
            q_freq = np.fromiter(
                (self.doc_freqs[di].get(q, 0) for di in doc_ids),
                dtype=np.float32,
                count=len(doc_ids),
            )
            score += (self.idf.get(q) or 0.0) * (
                q_freq * (self.k1 + 1.0) / (q_freq + norm)
            )
        return score.tolist()


def save_bm25_artifact(bm25, index_path: Path):
    index_path = Path(index_path)
    artifact_dir = get_bm25_artifact_dir(index_path)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    for old_chunk in artifact_dir.glob("doc_freqs_*.pkl"):
        old_chunk.unlink()
    for old_name in ("idf.pkl", "doc_len.npy"):
        old_path = artifact_dir / old_name
        if old_path.exists():
            old_path.unlink()

    np.save(artifact_dir / "doc_len.npy", np.asarray(bm25.doc_len, dtype=np.uint32), allow_pickle=False)

    with open(artifact_dir / "idf.pkl", "wb") as f:
        pickle.dump(bm25.idf, f, protocol=pickle.HIGHEST_PROTOCOL)

    n_docs = len(bm25.doc_freqs)
    n_chunks = math.ceil(n_docs / BM25_CHUNK_SIZE)
    for chunk_idx, start in enumerate(
        tqdm(
            range(0, n_docs, BM25_CHUNK_SIZE),
            total=n_chunks,
            desc="  Saving BM25 chunks",
            ncols=80,
            ascii=True,
        )
    ):
        chunk = bm25.doc_freqs[start:start + BM25_CHUNK_SIZE]
        chunk_path = artifact_dir / f"doc_freqs_{chunk_idx:05d}.pkl"
        with open(chunk_path, "wb") as f:
            pickle.dump(chunk, f, protocol=pickle.HIGHEST_PROTOCOL)

    manifest = {
        "format": ARTIFACT_FORMAT,
        "artifact_dir": artifact_dir.name,
        "doc_len_file": "doc_len.npy",
        "idf_file": "idf.pkl",
        "doc_freq_prefix": "doc_freqs_",
        "doc_freq_suffix": ".pkl",
        "n_chunks": n_chunks,
        "chunk_size": BM25_CHUNK_SIZE,
        "corpus_size": bm25.corpus_size,
        "avgdl": float(bm25.avgdl),
        "k1": float(bm25.k1),
        "b": float(bm25.b),
        "epsilon": float(getattr(bm25, "epsilon", 0.25)),
    }
    with open(index_path, "wb") as f:
        pickle.dump(manifest, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_bm25_artifact(index_path: Path):
    index_path = Path(index_path)
    with open(index_path, "rb") as f:
        obj = pickle.load(f)

    if not isinstance(obj, dict) or obj.get("format") != ARTIFACT_FORMAT:
        return obj

    artifact_dir = index_path.parent / obj["artifact_dir"]

    doc_len = np.load(artifact_dir / obj["doc_len_file"], allow_pickle=False)
    with open(artifact_dir / obj["idf_file"], "rb") as f:
        idf = pickle.load(f)

    doc_freqs: list[dict[str, int]] = []
    for chunk_idx in tqdm(
        range(obj["n_chunks"]),
        total=obj["n_chunks"],
        desc="  Loading BM25 chunks",
        ncols=80,
        ascii=True,
    ):
        chunk_path = artifact_dir / f"{obj['doc_freq_prefix']}{chunk_idx:05d}{obj['doc_freq_suffix']}"
        with open(chunk_path, "rb") as f:
            chunk = pickle.load(f)
        doc_freqs.extend(chunk)

    return ChunkedBM25Okapi(
        doc_freqs=doc_freqs,
        doc_len=doc_len,
        idf=idf,
        avgdl=obj["avgdl"],
        k1=obj["k1"],
        b=obj["b"],
        epsilon=obj["epsilon"],
    )


### `retrieval/explicit_citations.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/retrieval/explicit_citations.py"

"""
explicit_citations.py -- Extract citation strings directly from query text.

Many val queries contain literal citation strings like:
  "Art. 221 Abs. 1 lit. b StPO"
  "BGE 137 IV 122 E. 6.2"
  "1B_210/2023 E. 4.1"

These are FREE RECALL — they are derived from literal query references and
should ALWAYS be included in the candidate pipeline regardless of retrieval scores.

# HOW TO RUN (standalone test):
#   cd E:\\swiss-law-pipeline
#   python retrieval/explicit_citations.py
"""

import re


# Common alternate abbreviations seen in multilingual Swiss citations.
# We normalize these to the corpus' canonical abbreviations so downstream
# verification and retrieval can keep them instead of dropping them.
_ALT_ABBREV_TO_CANON = {
    "CC": "ZGB",
    "CO": "OR",
    "LP": "SchKG",
    "CPC": "ZPO",
    "LDIP": "IPRG",
    "LFors": "GestG",
    "CPP": "StPO",
    "CP": "StGB",
    "LTF": "BGG",
    "PA": "VwVG",
}

_SINGLE_STAT_PATTERN = re.compile(
    r"Art\.\s+"
    r"(\d+[a-z]?(?:\s+Abs\.\s+\d+[a-z]?)?(?:\s+lit\.\s+[a-z])?)"
    r"\s+(?:of\s+the\s+)?"
    r"([A-Z][A-Za-z0-9]{1,14})",
    re.UNICODE,
)

_MULTI_STAT_PATTERN = re.compile(
    r"Art\.\s+"
    r"("
    r"\d+[a-z]?(?:\s+Abs\.\s+\d+[a-z]?)?(?:\s+lit\.\s+[a-z])?"
    r"(?:\s*(?:,|and|und|or|oder)\s*"
    r"\d+[a-z]?(?:\s+Abs\.\s+\d+[a-z]?)?(?:\s+lit\.\s+[a-z])?)+"
    r")"
    r"\s+(?:of\s+the\s+)?"
    r"([A-Z][A-Za-z0-9]{1,14})",
    re.UNICODE,
)

_ARTICLE_PART_SPLIT = re.compile(r"\s*(?:,|and|und|or|oder)\s*", re.UNICODE)


def _canon_law_abbrev(abbrev: str) -> str:
    return _ALT_ABBREV_TO_CANON.get(abbrev, abbrev)


def _norm_statute_ref(article_part: str, law_abbrev: str) -> str:
    article_part = re.sub(r"\s+", " ", article_part.strip())
    law_abbrev = _canon_law_abbrev(law_abbrev.strip())
    return f"Art. {article_part} {law_abbrev}".strip()


def extract_explicit_citations(query_text: str) -> set[str]:
    """
    Extract all citation strings from query text using regex patterns.

    Returns a set of citation strings derived from literal references in the
    query text. We lightly normalize whitespace and common multilingual law
    abbreviations (e.g. CC -> ZGB, CO -> OR), and expand shared-law patterns
    such as "Art. 38 and 39 CO" into individual citations.
    """
    citations = set()

    # ── Shared-law statutory refs: Art. 38 and 39 CO -> 2 citations ────────
    for m in _MULTI_STAT_PATTERN.finditer(query_text):
        raw_articles, raw_law = m.group(1), m.group(2)
        for art_part in _ARTICLE_PART_SPLIT.split(raw_articles):
            art_part = art_part.strip()
            if art_part:
                citations.add(_norm_statute_ref(art_part, raw_law))

    # ── Statutory: Art. X [Abs. Y [lit. z]] LAW ──────────────────────────
    # Matches: Art. 221 Abs. 1 lit. b StPO, Art. 44 ATSG, Art. 125 of the CC
    for m in _SINGLE_STAT_PATTERN.finditer(query_text):
        citations.add(_norm_statute_ref(m.group(1), m.group(2)))

    # ── BGE decisions: BGE XXX II/III/IV/V YYY E. Z.Z ───────────────────
    # Matches: BGE 137 IV 122 E. 6.2
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"BGE\s+\d+\s+[IVX]+\s+\d+\s+E\.\s+[\d.]+",
            query_text,
        )
    )

    # ── BGE decisions without Erwägung: BGE 137 IV 122 ──────────────────
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"BGE\s+\d+\s+[IVX]+\s+\d+",
            query_text,
        )
    )

    # ── Numbered cases: 1B_210/2023 E. 4.1 ──────────────────────────────
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"\d+[A-Z]_\d+/\d{4}\s+E\.\s+[\d.]+",
            query_text,
        )
    )

    # ── Numbered cases without Erwägung: 1B_210/2023 ────────────────────
    citations.update(
        m.group(0).strip()
        for m in re.finditer(
            r"\d+[A-Z]_\d+/\d{4}",
            query_text,
        )
    )

    return citations


### `retrieval/graph_retriever.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/retrieval/graph_retriever.py"

"""
graph_retriever.py -- Citation graph traversal retrieval.

Given candidate statutory articles -> find case law that cites them.
Given candidate case citations   -> find statutory articles they cite.

This is the CRITICAL bridge between statutory and case law retrieval.
40% of val citations are court decisions, but train data is 98.8% statutory.
The citation graph mechanically bridges this gap.

# HOW TO RUN (requires index/citation_graph.pkl):
#   cd E:\\swiss-law-pipeline
#   python retrieval/graph_retriever.py
#   python retrieval/graph_retriever.py --article "Art. 221 StPO"
#
# EXPECTS: index/citation_graph.pkl (built by: python stage0_build_index.py)
"""

from pathlib import Path

import sys
sys.path.insert(0, str(Path(__file__).parent.parent))
from data.data_paths import CITATION_GRAPH_PKL
from indexing.build_citation_graph import CitationGraph


class GraphRetriever:
    def __init__(self, graph_pkl: Path = CITATION_GRAPH_PKL):
        print("  Loading citation graph ...")
        self.graph = CitationGraph(graph_pkl)
        print(f"    {len(self.graph.article_to_cases):,} articles with case links")
        print(f"    {len(self.graph.case_to_articles):,} cases with article links")

    def articles_to_cases(
        self,
        article_citations: list[str],
        top_k_per_article: int = 20,
    ) -> list[tuple[str, float]]:
        """
        Given statutory articles, return citing case law.
        Score = sum of (1/rank) across articles (RRF-style).
        """
        scores: dict[str, float] = {}
        for art in article_citations:
            cases = self.graph.get_citing_cases(art, top_n=top_k_per_article)
            for rank, case in enumerate(cases, 1):
                scores[case] = scores.get(case, 0.0) + 1.0 / (60 + rank)
        return sorted(scores.items(), key=lambda x: -x[1])

    def cases_to_articles(self, case_citations: list[str]) -> list[str]:
        """Given case citations, return statutory articles they cite."""
        found: set[str] = set()
        for case in case_citations:
            arts = self.graph.get_cited_articles(case)
            found.update(arts)
        return list(found)

    def bidirectional_expand(
        self,
        seed_articles: list[str],
        seed_cases: list[str],
        depth: int = 1,
    ) -> tuple[list[tuple[str, float]], list[str]]:
        """
        Expand seeds bidirectionally up to `depth` hops.
        Returns (expanded_cases_with_scores, expanded_articles).
        """
        all_cases:    dict[str, float] = {}
        all_articles: set[str]         = set(seed_articles)

        current_articles = list(seed_articles)
        current_cases    = list(seed_cases)

        for _ in range(depth):
            new_cases = self.articles_to_cases(current_articles)
            for c, s in new_cases:
                all_cases[c] = max(all_cases.get(c, 0.0), s)

            new_articles = self.cases_to_articles(
                current_cases + [c for c, _ in new_cases]
            )
            all_articles.update(new_articles)

            current_articles = new_articles
            current_cases    = [c for c, _ in new_cases[:20]]

        return (
            sorted(all_cases.items(), key=lambda x: -x[1]),
            list(all_articles),
        )


### `retrieval/sparse_retriever.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/retrieval/sparse_retriever.py"

"""
sparse_retriever.py -- BM25 retrieval over the combined law+court index.

This is the PRIMARY retrieval engine in the revised pipeline.
Dense retrieval is dropped entirely — BM25 gives 3x better recall (R@50=0.73
vs R@100=0.24 for dense) on this cross-lingual legal task.

Features:
  - PMI-based law abbreviation boosting
  - Multi-query union with max-score fusion
  - np.argpartition for O(n) top-K
  - Separate law/court/combined search modes

# HOW TO RUN (standalone smoke-test):
#   cd E:\\swiss-law-pipeline
#   python retrieval/sparse_retriever.py
#   python retrieval/sparse_retriever.py --query "Untersuchungshaft Kollusionsgefahr StPO"
#
# EXPECTS: index/bm25_v2_index.pkl + index/bm25_v2_ids.pkl
#   (built by: python stage0_build_index.py)
"""

import re
import pickle
import json
from pathlib import Path

import numpy as np

import sys
sys.path.insert(0, str(Path(__file__).parent.parent))
from data.data_paths import STATUTORY_BM25_PKL, STATUTORY_BM25_IDS, TOKEN_LAW_FREQ
from retrieval.bm25_artifact import load_bm25_artifact

_TOKEN_RE = re.compile(r"[^\w\d]+", re.UNICODE)


def tokenise(text: str) -> list[str]:
    """Tokenize text for BM25 queries. Keeps short tokens if numeric (article numbers)."""
    if not text:
        return []
    return [t for t in _TOKEN_RE.split(text.lower()) if len(t) >= 2 or t.isdigit()]


class SparseRetriever:
    """
    Wraps the combined BM25 index (law + court).
    Exposes search_statutory(), search_caselaw(), search_combined().
    """

    def __init__(self):
        print("  Loading combined BM25 index ...")
        self._bm25 = load_bm25_artifact(STATUTORY_BM25_PKL)

        with open(STATUTORY_BM25_IDS, "rb") as f:
            meta = pickle.load(f)

        self._ids:     list[str] = meta["citation_canon"]
        self._n_law:   int       = meta["n_law"]
        self._n_court: int       = meta["n_court"]
        print(f"    {self._n_law:,} law  +  {self._n_court:,} court  =  {len(self._ids):,} total")

        # Build id->index mapping for fast rank lookup
        self._id_to_idx: dict[str, int] = {c: i for i, c in enumerate(self._ids)}

        # PMI token->law map for query enhancement
        self._token_law_freq: dict = {}
        self._token_total:    dict = {}
        if TOKEN_LAW_FREQ.exists():
            with open(TOKEN_LAW_FREQ, encoding="utf-8") as f:
                raw = json.load(f)
            self._token_law_freq = {t: v for t, v in raw.items()}
            self._token_total    = {t: sum(v.values()) for t, v in self._token_law_freq.items()}
            print(f"    PMI token->law map: {len(self._token_law_freq):,} tokens")

    # ------------------------------------------------------------------
    # Public search API
    # ------------------------------------------------------------------

    def search_statutory(self, queries: list[str], top_k: int = 100) -> list[tuple[str, float]]:
        """Search law articles only. Returns [(citation, score), ...]."""
        return self._search(queries, top_k, law_only=True)

    def search_caselaw(self, queries: list[str], top_k: int = 100) -> list[tuple[str, float]]:
        """Search court considerations only. Returns [(citation, score), ...]."""
        return self._search(queries, top_k, court_only=True)

    def search_combined(self, queries: list[str], top_k: int = 200) -> list[tuple[str, float]]:
        """Search across the full combined index (law + court)."""
        return self._search(queries, top_k)

    def get_bm25_rank(self, query_tokens: list[str], citation: str) -> int:
        """Get the BM25 rank of a specific citation for given query tokens. -1 if not found."""
        idx = self._id_to_idx.get(citation, -1)
        if idx < 0:
            return -1
        scores = self._bm25.get_scores(query_tokens)
        rank = (scores > scores[idx]).sum() + 1
        return int(rank)

    # ------------------------------------------------------------------
    # Internal
    # ------------------------------------------------------------------

    def _enhance_tokens(self, tokens: list[str], law_boost: int = 10,
                        top_n_laws: int = 5) -> list[str]:
        """Append predicted law abbreviations (PMI) to query tokens."""
        if not self._token_law_freq:
            return tokens

        scores: dict[str, float] = {}
        for t in set(tokens):
            if t not in self._token_law_freq:
                continue
            idf = self._bm25.idf.get(t, 0.0)
            if idf < 1.0:
                continue
            total = self._token_total.get(t, 1)
            for ab, cnt in self._token_law_freq[t].items():
                scores[ab] = scores.get(ab, 0.0) + (cnt / total) * idf

        predicted = sorted(scores.items(), key=lambda x: -x[1])[:top_n_laws]
        enhanced = list(dict.fromkeys(tokens))
        for ab, _ in predicted:
            enhanced.extend([ab] * law_boost)
        return enhanced

    def _search(self, queries: list[str], top_k: int,
                law_only: bool = False, court_only: bool = False) -> list[tuple[str, float]]:
        """Core search: runs each query, takes max score per doc across queries."""
        scores: dict[str, float] = {}

        for q in queries:
            toks = tokenise(q)
            if not toks:
                continue
            toks = self._enhance_tokens(toks)

            raw = self._bm25.get_scores(toks)

            # Restrict to relevant slice
            if law_only:
                raw[self._n_law:] = 0.0
            elif court_only:
                raw[:self._n_law] = 0.0

            k = min(top_k, int((raw > 0).sum()), len(raw))
            if k == 0:
                continue
            top_idx = np.argpartition(raw, -k)[-k:]
            for i in top_idx:
                if raw[i] <= 0:
                    continue
                c = self._ids[i]
                scores[c] = max(scores.get(c, 0.0), float(raw[i]))

        return sorted(scores.items(), key=lambda x: -x[1])[:top_k]


### `scoring/confidence.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/scoring/confidence.py"

"""
confidence.py -- Composite confidence scoring and F1 threshold optimization.

Revised scoring system with:
  - Signal-counting approach (how many independent sources found this citation)
  - Tiered LLM reranker scores (tier 3 = directly applicable, tier 2 = related)
  - Multi-signal bonus for citations found by 3+ independent signals
  - BM25 initial and MAS agent signals now contribute
  - **Continuous BM25 score**: per-query normalized BM25 relevance (0-1) adds
    up to 0.20, creating a ranking gradient within same-signal-count groups.
  - **Cross-encoder score** (Strategy C): when available, replaces the binary
    llm_reranker/tier3 signals with a continuous Qwen3-Reranker-4B P(yes)
    probability × 0.18. Falls back to binary signals for unscored citations.

Signal weights (additive, max ~1.0):
  - Explicit in query text:     0.40
  - BM25 continuous score:      0.00-0.20 (normalized per query)
  - BM25 initial / MAS agents:  0.10
  - Citation graph:              0.10
  - Cross-encoder score:        0.00-0.18 (replaces binary reranker when available)
  - LLM reranker tier 3:        0.25  (fallback when no CE score)
  - LLM reranker tier 2:        0.15  (fallback when no CE score)
  - LLM direct generation:      0.05
  - LLM stage1 candidate:       0.05
  - Procedural (stage1):        0.05
  - Multi-signal bonus (3+):    0.10

# HOW TO RUN (standalone unit-test):
#   cd E:\\swiss-law-pipeline
#   python scoring/confidence.py
"""

import numpy as np


DEFAULT_WEIGHTS = {
    "explicit_from_query":    0.40,
    "bm25_top10":             0.15,   # binary: in BM25 top-10 yes/no
    "bm25_initial":           0.10,
    "citation_graph":         0.10,
    "llm_reranker_tier3":     0.25,
    "llm_reranker":           0.15,
    "llm_direct_gen":         0.05,
    "llm_stage1":             0.05,
    "llm_stage1_procedural":  0.05,
    "co_citation":            0.03,
}

# Additional continuous BM25 gradient (on top of binary signals)
# Lower weight than before — this adds ranking within same-signal groups
# without inflating all scores and pushing the threshold too high.
BM25_CONTINUOUS_WEIGHT = 0.10

# Cross-encoder weight (Strategy C): continuous P(yes) score × this weight
# replaces binary llm_reranker/tier3 signals. Validated at 0.18 on all 10
# val queries: 0.5400 → 0.6520 macro-F1.
CROSS_ENCODER_WEIGHT = 0.18

# Signals that count as "independent" for the multi-signal bonus
_INDEPENDENT_SIGNALS = {
    "explicit_from_query", "bm25_top10", "bm25_initial",
    "citation_graph", "llm_reranker", "llm_reranker_tier3",
    "llm_direct_gen", "llm_stage1",
}

MULTI_SIGNAL_BONUS = 0.10  # added when 3+ independent signals agree


def score_citation(cite: str, sources: dict[str, set[str]],
                   weights: dict[str, float] | None = None,
                   bm25_scores: dict[str, float] | None = None) -> float:
    """
    Compute composite confidence score for a single citation.

    Scoring approach: additive weights + continuous BM25 score + graduated
    multi-signal bonus. The key discriminators are:
      1. NUMBER of independent signals (gold avg 3-4, noise avg 1-2)
      2. Continuous BM25 relevance score (creates ranking gradient within
         same-signal-count groups — this is what separates 2-signal gold
         from 2-signal noise)

    Args:
        cite:           Citation string.
        sources:        Dict mapping source_name -> set of citations from that source.
        weights:        Override default weights.
        bm25_scores:    Dict mapping citation -> normalized BM25 score (0-1).
                        If provided, replaces binary bm25_top10 with continuous signal.

    Returns:
        Float score in [0, 1].
    """
    w = weights if weights is not None else DEFAULT_WEIGHTS

    score = 0.0
    active_signals = set()

    for source_name, weight in w.items():
        if cite in sources.get(source_name, set()):
            # If citation has tier3, don't also add the base llm_reranker weight
            if source_name == "llm_reranker" and cite in sources.get("llm_reranker_tier3", set()):
                continue  # tier3 supersedes base reranker
            score += weight
            active_signals.add(source_name)

    # Also add MAS agent signals as bm25_initial weight if they contributed
    for mas_source in ["mas_rewrite", "mas_supplement", "mas_decompose",
                       "mas_supportive", "mas_crossref"]:
        if cite in sources.get(mas_source, set()):
            if "bm25_initial" not in active_signals:
                score += w.get("bm25_initial", 0.10)
                active_signals.add("bm25_initial")
            break

    # Continuous BM25 score: normalized per-query BM25 relevance (0.0 to 0.20)
    # This creates the ranking gradient that binary signals can't provide.
    # A gold citation at BM25 rank #2 gets ~0.18, while noise at rank #40 gets ~0.05.
    if bm25_scores is not None:
        bm25_norm = bm25_scores.get(cite, 0.0)
        if bm25_norm > 0:
            score += bm25_norm * BM25_CONTINUOUS_WEIGHT
            active_signals.add("bm25_hit")

    # Graduated multi-signal bonus: the more independent signals agree,
    # the more likely this is a true gold citation.
    independent_count = len(active_signals & _INDEPENDENT_SIGNALS)
    if independent_count >= 4:
        score += 0.20  # very high confidence
    elif independent_count >= 3:
        score += 0.10  # high confidence
    elif independent_count >= 2:
        score += 0.05  # moderate confidence

    return min(score, 1.0)


def score_all_citations(
    all_citations: list[str],
    sources: dict[str, set[str]],
    weights: dict[str, float] | None = None,
    bm25_scores: dict[str, float] | None = None,
) -> list[tuple[str, float]]:
    """
    Score all citations and return sorted (citation, score) list.
    """
    scored = [
        (cite, score_citation(cite, sources, weights, bm25_scores=bm25_scores))
        for cite in all_citations
    ]
    return sorted(scored, key=lambda x: -x[1])


def compute_f1(predicted: list[str], gold: set[str]) -> float:
    """Compute F1 for a single query (case-insensitive matching)."""
    if not predicted and not gold:
        return 1.0
    if not predicted or not gold:
        return 0.0
    gold_lower = {c.lower() for c in gold}
    pred_lower = {c.lower() for c in predicted}
    tp = len(pred_lower & gold_lower)
    p = tp / len(pred_lower)
    r = tp / len(gold_lower)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def optimize_threshold(
    scored_predictions: dict[str, list[tuple[str, float]]],
    val_gold: dict[str, set[str]],
    threshold_range: tuple[float, float, float] = (0.05, 0.95, 0.01),
) -> tuple[float, float]:
    """
    Sweep thresholds on val set to find the one maximizing macro-F1.
    """
    lo, hi, step = threshold_range
    thresholds = np.arange(lo, hi + step, step)

    best_f1, best_thresh = 0.0, lo

    for thresh in thresholds:
        f1s = []
        for qid, scored in scored_predictions.items():
            preds = [c for c, s in scored if s >= thresh]
            gold = val_gold.get(qid, set())
            if gold:
                f1s.append(compute_f1(preds, gold))

        macro = sum(f1s) / len(f1s) if f1s else 0.0
        if macro > best_f1:
            best_f1 = macro
            best_thresh = float(thresh)

    return best_thresh, best_f1


## Stage 0 — Offline Index Building

One-time index construction and precomputation.

### `stage0_build_index.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage0_build_index.py"

"""
stage0_build_index.py -- STAGE 0: Offline Index Building (CPU only, one-time)

Builds all indexes needed by the pipeline:
  0A. BM25 combined index over 171K laws + ~2M court considerations
  0B. Citation graph: Art. <-> Court decision bidirectional links
  0C. Citation lookup tables: normalization, existence checks, Abs. expansion
  0D. Citation signal cards: deterministic statute + case signal artifacts
  0E. Typed reference graph v2: statute/case edge sets + reverse views
  0F. Train-only gold co-citation prior for conservative expansion
  0F*. Optional train+val co-citation prior for competition-mode runs

This runs ONCE and takes ~3-5 hours total (dominated by BM25 indexing).
All work is CPU-only — no GPU needed.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python stage0_build_index.py              # build everything
#   python stage0_build_index.py --bm25-only  # only BM25 index
#   python stage0_build_index.py --graph-only # only citation graph
#   python stage0_build_index.py --lookup-only # only lookup tables
#   python stage0_build_index.py --signals-only # only citation signal cards
#   python stage0_build_index.py --refgraph-only # only reference graph v2
#   python stage0_build_index.py --goldprior-only # only train-only gold prior
#   python stage0_build_index.py --goldprior-trainval-only # only train+val gold prior
#
# ─── PREREQUISITES ──────────────────────────────────────────────────────────
#   data/corpus.parquet              (171K law articles)
#   data/court_considerations.csv    (2.47M court rows)
#   data/laws_knowledge_base.jsonl   (KB metadata)
#   data/train.csv + data/val.csv    (for document expansion + eval)
#
# ─── OUTPUTS ────────────────────────────────────────────────────────────────
#   index/bm25_v2_index.pkl          (~2-4 GB, combined BM25 index)
#   index/bm25_v2_ids.pkl            (citation IDs aligned to BM25 index)
#   index/citation_graph.pkl         (Art. <-> Case citation links)
#   index/citation_lookup.pkl        (normalization + existence tables)
#   index/citation_signal_lookup.pkl (deterministic signal lookup)
#   index/reference_graph_v2.pkl     (typed statute/case reference graph)
#   index/gold_cocitation_prior_train.pkl (train-only pair prior)
#   index/gold_cocitation_prior_trainval.pkl (optional competition-mode pair prior)
#   data/query_translations_trainval.json  (cached EN->DE translations)
#   data/token_law_freq.json         (PMI token->law frequency map)
#   data/statute_signals.jsonl       (deterministic statute signal cards)
#   data/case_signals.jsonl          (deterministic case signal cards)
#
# ─── EXPECTED TIMING ────────────────────────────────────────────────────────
#   BM25 index:      ~3-5 hours (reads 2.47M court rows, builds tokens)
#   Citation graph:  ~1-2 hours (scans 2.47M court texts for Art. refs)
#   Lookup tables:   ~1 minute  (reads corpus.parquet only)
#   Total:           ~4-7 hours one-time. After that, pipeline runs fast.
#
# ─── MEMORY ─────────────────────────────────────────────────────────────────
#   BM25 build needs ~16 GB RAM (2.15M tokenized docs in memory).
#   Citation graph needs ~4 GB RAM (chunked CSV reading).
#   Lookup tables need ~1 GB RAM.
"""

import argparse
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
from data.data_paths import (
    STATUTORY_BM25_PKL, CITATION_GRAPH_PKL, LOOKUP_PKL,
    CITATION_SIGNAL_LOOKUP_PKL, REFERENCE_GRAPH_V2_PKL,
    GOLD_COCITATION_PRIOR_TRAIN_PKL, GOLD_COCITATION_PRIOR_TRAINVAL_PKL,
    INDEX_DIR,
)

INDEX_DIR.mkdir(parents=True, exist_ok=True)


def build_bm25():
    """0A. Build combined BM25 index (law + court). ~3-5 hours."""
    print("\n" + "=" * 70)
    print("  STAGE 0A: Building BM25 Index")
    print("=" * 70)
    t0 = time.time()

    # The existing build_bm25_index.py is a complete standalone script.
    # It loads data, translates queries, builds docs, builds BM25, evaluates recall.
    # We import and run it directly.
    import indexing.build_bm25_index  # noqa — runs on import (script-style)

    elapsed = time.time() - t0
    print(f"\n  BM25 index built in {elapsed/60:.1f} min")
    print(f"  Output: {STATUTORY_BM25_PKL}")


def build_citation_graph():
    """0B. Build Art. <-> Case citation graph. ~1-2 hours."""
    print("\n" + "=" * 70)
    print("  STAGE 0B: Building Citation Graph")
    print("=" * 70)
    t0 = time.time()

    from indexing.build_citation_graph import build_citation_graph as _build
    _build()

    elapsed = time.time() - t0
    print(f"\n  Citation graph built in {elapsed/60:.1f} min")
    print(f"  Output: {CITATION_GRAPH_PKL}")


def build_lookup_tables():
    """0C. Build citation normalization + lookup tables. ~1 minute."""
    print("\n" + "=" * 70)
    print("  STAGE 0C: Building Lookup Tables")
    print("=" * 70)
    t0 = time.time()

    from indexing.build_lookup_tables import build_lookup_tables as _build
    _build()

    elapsed = time.time() - t0
    print(f"\n  Lookup tables built in {elapsed:.1f}s")
    print(f"  Output: {LOOKUP_PKL}")


def build_citation_signals():
    """0D. Build deterministic citation signal cards."""
    print("\n" + "=" * 70)
    print("  STAGE 0D: Building Citation Signals")
    print("=" * 70)
    t0 = time.time()

    from stage0d_build_citation_signals import build_all as _build
    _build()

    elapsed = time.time() - t0
    print(f"\n  Citation signals built in {elapsed/60:.1f} min")
    print(f"  Output: {CITATION_SIGNAL_LOOKUP_PKL}")


def build_reference_graph_v2():
    """0E. Build typed reference graph v2."""
    print("\n" + "=" * 70)
    print("  STAGE 0E: Building Reference Graph V2")
    print("=" * 70)
    t0 = time.time()

    from stage0e_build_reference_graph_v2 import build_all as _build
    _build()

    elapsed = time.time() - t0
    print(f"\n  Reference graph v2 built in {elapsed/60:.1f} min")
    print(f"  Output: {REFERENCE_GRAPH_V2_PKL}")


def build_gold_cocitation_prior():
    """0F. Build train-only gold co-citation prior."""
    print("\n" + "=" * 70)
    print("  STAGE 0F: Building Gold Co-Citation Prior (Train Only)")
    print("=" * 70)
    t0 = time.time()

    from stage0f_build_gold_cocitation_prior import build_prior as _build
    _build()

    elapsed = time.time() - t0
    print(f"\n  Gold co-citation prior built in {elapsed:.1f}s")
    print(f"  Output: {GOLD_COCITATION_PRIOR_TRAIN_PKL}")


def build_gold_cocitation_prior_trainval():
    """0F*. Build optional train+val gold co-citation prior."""
    print("\n" + "=" * 70)
    print("  STAGE 0F*: Building Gold Co-Citation Prior (Train+Val)")
    print("=" * 70)
    t0 = time.time()

    from stage0f_build_gold_cocitation_prior import build_prior as _build
    from data.data_paths import TRAIN_CSV, VAL_CSV
    _build(
        csv_paths=[TRAIN_CSV, VAL_CSV],
        out_pkl=GOLD_COCITATION_PRIOR_TRAINVAL_PKL,
        prior_name="trainval",
    )

    elapsed = time.time() - t0
    print(f"\n  Gold co-citation train+val prior built in {elapsed:.1f}s")
    print(f"  Output: {GOLD_COCITATION_PRIOR_TRAINVAL_PKL}")


### `indexing/build_bm25_index.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/indexing/build_bm25_index.py"

"""
build_bm25_index.py -- Production BM25 index builder for Swiss law + court retrieval.

Builds a COMBINED index over:
  - Law articles (~171k from corpus.parquet)
  - Court considerations (~1.98M unique citations from court_considerations.csv)

The court source file has 2.47M rows but only 1.98M unique citations — some
citations appear multiple times with different paragraph texts.  These are
deduplicated by grouping on citation and concatenating all texts (up to
COURT_TEXT_LIMIT chars) into one document per unique citation.

Evaluates recall on the FULL gold set (law + court) for both train and val.

Outputs:
  index/bm25_v2_index.pkl   -- BM25Okapi index (law + court)
  index/bm25_v2_ids.pkl     -- citation list aligned to index
  data/query_translations_trainval.json
  data/token_law_freq.json

# HOW TO RUN (~3-5 hours for full 2.15M doc index, needs ~16GB RAM):
#   cd E:\swiss-law-pipeline
#   python indexing/build_bm25_index.py
#
# PowerShell:
#   cd E:\swiss-law-pipeline
#   python indexing/build_bm25_index.py
#   python indexing/build_bm25_index.py | Tee-Object -FilePath index/bm25_build.log
#
# To skip train eval and just build the index (faster):
#   Set EVAL_TRAIN_SAMPLE = 0  in the PATHS/GLOBALS section below, then run.
"""

import sys, os, re, json, pickle, math, random, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
from rank_bm25 import BM25Okapi
from tqdm import tqdm


# ==============================================================================
# ENV / IO SAFETY
# ==============================================================================

os.environ["PYTHONIOENCODING"] = "utf-8"


def safe_reconfigure():
    for stream_name in ("stdout", "stderr"):
        stream = getattr(sys, stream_name, None)
        if stream is not None and hasattr(stream, "reconfigure"):
            try:
                stream.reconfigure(encoding="utf-8", errors="replace")
            except Exception:
                pass


safe_reconfigure()


# ==============================================================================
# PATHS / GLOBALS
# ==============================================================================

import sys as _sys
_sys.path.insert(0, str(Path(__file__).parent.parent))
from data.data_paths import (
    DATA, INDEX_DIR, CHECKPOINTS_DIR,
    LAW_CORPUS_PARQUET, COURT_CSV, KB_JSONL,
    STATUTORY_BM25_PKL, STATUTORY_BM25_IDS,
    QUERY_TRANSLATIONS_JSON, TOKEN_LAW_FREQ,
    TRAIN_CSV, VAL_CSV,
)
from retrieval.bm25_artifact import save_bm25_artifact
INDEX_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR   = INDEX_DIR   # evaluation txt files go here too

_TOKEN_RE = re.compile(r"[^\w\d]+", re.UNICODE)

TOPK = (30, 50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000)

# BM25 config (fixed best)
K_F11  = 1.2
BEST_B = 0.75
K_F1   = 100   # candidate pool size for F1 / pipeline

# Train evaluation: set to None to evaluate all 1139 queries (slow ~9h),
# set to an int for a random sample, or 0 to skip (index-build only).
EVAL_TRAIN_SAMPLE = 0

RANDOM_SEED = 42

try:
    TRANSLATION_WORKERS = max(1, min(16, int(os.getenv("SWISS_TRANSLATION_WORKERS", "8"))))
except Exception:
    TRANSLATION_WORKERS = 8

try:
    TRANSLATION_SAVE_EVERY = max(1, int(os.getenv("SWISS_TRANSLATION_SAVE_EVERY", "50")))
except Exception:
    TRANSLATION_SAVE_EVERY = 50

TRANSLATION_MODE = os.getenv("SWISS_QUERY_TRANSLATION_MODE", "auto").strip().lower()
if TRANSLATION_MODE not in {"auto", "force", "off"}:
    TRANSLATION_MODE = "auto"

try:
    TRANSLATION_WARN_LIMIT = max(0, int(os.getenv("SWISS_TRANSLATION_WARN_LIMIT", "3")))
except Exception:
    TRANSLATION_WARN_LIMIT = 3


# ==============================================================================
# HELPERS
# ==============================================================================

def tokenise(text: str) -> list[str]:
    if not text:
        return []
    return [t for t in _TOKEN_RE.split(text.lower().strip()) if len(t) >= 2 or t.isdigit()]


def is_law_citation(c: str) -> bool:
    return bool(c and re.match(r"^Art\.\s+\d", c.strip()))


def parse_gold_law(raw: str) -> list[str]:
    """Return only law-article citations (Art. X ...) from semicolon-separated string."""
    if not isinstance(raw, str):
        return []
    return [c.strip() for c in raw.split(";") if c.strip() and is_law_citation(c.strip())]


def parse_gold_all(raw: str) -> list[str]:
    """Return ALL citations (law + court) from semicolon-separated string."""
    if not isinstance(raw, str):
        return []
    return [c.strip() for c in raw.split(";") if c.strip()]


def build_query_tokens(query_text: str, de_text: str | None = None) -> list[str]:
    """Union of English + German-translated tokens (deduplicated, stable order)."""
    en = tokenise(query_text)
    if not de_text:
        return en

    seen = set()
    combined = []
    for t in en + tokenise(de_text):
        if t not in seen:
            seen.add(t)
            combined.append(t)
    return combined


def safe(text: str) -> str:
    """Encode to ASCII, replacing non-ASCII chars -- safe for any terminal."""
    return text.encode("ascii", "replace").decode("ascii")


def sep(title: str = ""):
    print("\n" + "=" * 70)
    if title:
        print(f"  {safe(title)}")
        print("=" * 70)


def hr():
    print("-" * 70)


_TRANSLATOR_LOCAL = threading.local()
_TRANSLATION_DISABLED = threading.Event()

_DE_MARKERS = {
    "der", "die", "das", "und", "ist", "nicht", "mit", "von", "für", "fuer",
    "welche", "welcher", "welches", "welchen", "wird", "wurde", "eine", "einer",
    "einem", "eines", "einen", "im", "am", "des", "dem", "den", "zur", "zum",
    "über", "ueber", "nach", "bei", "als", "auch", "sich", "hat", "haben",
    "vorliegend", "frage", "prüfen", "pruefen", "sind", "kann", "gegen", "wegen",
}
_EN_MARKERS = {
    "the", "and", "is", "are", "of", "for", "what", "which", "with", "from", "to",
    "under", "whether", "can", "does", "should", "would", "where", "when", "who",
    "contract", "liability", "court", "law", "claim", "issue", "facts",
}


def _get_google_translator():
    translator = getattr(_TRANSLATOR_LOCAL, "google_de", None)
    if translator is None:
        from deep_translator import GoogleTranslator
        translator = GoogleTranslator(source="en", target="de")
        _TRANSLATOR_LOCAL.google_de = translator
    return translator


def should_translate_query(text: str) -> bool:
    if TRANSLATION_MODE == "off":
        return False
    if TRANSLATION_MODE == "force":
        return True

    # Auto mode: only translate queries that show clear English signal.
    if any(ch in text for ch in "äöüÄÖÜß"):
        return False

    toks = tokenise(text)
    if not toks:
        return False

    de_score = sum(1 for t in toks if t in _DE_MARKERS)
    en_score = sum(1 for t in toks if t in _EN_MARKERS)

    if de_score >= max(2, en_score + 1):
        return False
    return en_score > de_score


def is_fatal_translation_error(message: str) -> bool:
    msg = message.lower()
    return (
        "nameresolutionerror" in msg
        or "failed to resolve" in msg
        or "getaddrinfo failed" in msg
        or "temporary failure in name resolution" in msg
    )


def translate_to_de(text: str) -> str | None:
    if _TRANSLATION_DISABLED.is_set():
        return None
    return _get_google_translator().translate(text[:4999])


def save_translation_cache(cache_file: Path, cache: dict[str, str]):
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def compute_f1(predicted: list[str], gold_set: set[str]) -> float:
    if not predicted and not gold_set:
        return 1.0
    if not predicted or not gold_set:
        return 0.0

    tp = sum(1 for c in predicted if c in gold_set)
    p = tp / len(predicted)
    r = tp / len(gold_set)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def oracle_f1(ranked: list[str], gold_set: set[str]) -> tuple[float, int]:
    """Best achievable F1 at any cutoff K -- upper bound."""
    if not gold_set:
        return 0.0, 1

    best_f1, best_k, tp = 0.0, 1, 0
    for k, cit in enumerate(ranked, 1):
        if cit in gold_set:
            tp += 1
        p = tp / k
        r = tp / len(gold_set)
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        if f1 > best_f1:
            best_f1, best_k = f1, k
    return best_f1, best_k


# ==============================================================================
# 1. LOAD DATA
# ==============================================================================
sep("LOADING DATA")

# --- Law corpus ---
law_corpus = pd.read_parquet(LAW_CORPUS_PARQUET)
print(f"  Law corpus: {len(law_corpus):,} articles")

# --- Court corpus (court_considerations.csv: 2.47M rows, ~1.98M unique citations) ---
# Multiple rows per citation (different paragraph texts) → concatenate into one doc.
COURT_TEXT_LIMIT = 8000   # max chars of combined text per citation

court_corpus_path = COURT_CSV
if not court_corpus_path.exists():
    print(f"  WARNING: Court corpus not found at {court_corpus_path}")
    court_corpus = pd.DataFrame(columns=["citation", "text"])
else:
    print("  Reading court_considerations.csv (2.47M rows) ...")
    raw_court = pd.read_csv(court_corpus_path)
    raw_court["citation"] = raw_court["citation"].fillna("").str.strip()
    raw_court["text"]     = raw_court["text"].fillna("")
    raw_court = raw_court[raw_court["citation"] != ""].reset_index(drop=True)
    print(f"    Raw rows: {len(raw_court):,}  |  Unique citations: {raw_court['citation'].nunique():,}")

    # Deduplicate: group by citation, concatenate all paragraph texts
    print("  Deduplicating: concatenating texts per citation ...")
    def _join_texts(texts):
        combined = " ".join(t for t in texts if t)
        return combined[:COURT_TEXT_LIMIT]

    court_corpus = (
        raw_court.groupby("citation", sort=False)["text"]
        .apply(_join_texts)
        .reset_index()
    )
    court_corpus.columns = ["citation", "text"]
    del raw_court  # free ~500MB RAM
    print(f"  Court corpus after dedup: {len(court_corpus):,} unique citations")

# --- Knowledge base (law metadata) ---
print("  Loading KB JSONL ...")
kb_map: dict[str, dict] = {}
with open(KB_JSONL, encoding="utf-8") as f:
    for line in f:
        try:
            rec = json.loads(line)
            canon = rec.get("citation_canon", "")
            if canon:
                kb_map[canon] = rec
        except Exception:
            pass
print(f"  KB records: {len(kb_map):,}")

law_name_en_map: dict[str, str] = {}
for rec in kb_map.values():
    abbrev = rec.get("law", {}).get("law_abbreviation") or ""
    name_en = rec.get("law", {}).get("law_name_en") or ""
    if abbrev and name_en and abbrev not in law_name_en_map:
        law_name_en_map[abbrev] = name_en
print(f"  English law names loaded: {len(law_name_en_map):,}")

# --- Train / val ---
train = pd.read_csv(TRAIN_CSV)
val   = pd.read_csv(VAL_CSV)
print(f"  Train: {len(train):,} queries  |  Val: {len(val):,} queries")

# --- Build unified corpus_lower_map (law + court) ---
# Maps citation.lower() -> citation (original case preserved)
_corpus_lower_map: dict[str, str] = {}
for c in law_corpus["citation_canon"].tolist():
    _corpus_lower_map[c.lower()] = c
for c in court_corpus["citation"].tolist():
    _corpus_lower_map[c.lower()] = c


# ==============================================================================
# 2. QUERY TRANSLATIONS (cached)
# ==============================================================================
sep("LOADING QUERY TRANSLATIONS")

_cache_file = QUERY_TRANSLATIONS_JSON
query_translations: dict[str, str] = {}

if _cache_file.exists():
    with open(_cache_file, encoding="utf-8") as f:
        query_translations = json.load(f)
    print(f"  Loaded {len(query_translations):,} cached translations.")

all_queries = pd.concat([train, val], ignore_index=True)
missing = [r["query_id"] for _, r in all_queries.iterrows() if r["query_id"] not in query_translations]

if missing:
    missing_rows = all_queries[all_queries["query_id"].isin(missing)][["query_id", "query"]]
    translatable_rows = []
    skipped_auto = 0
    for qid, query_text in missing_rows.itertuples(index=False, name=None):
        if should_translate_query(query_text):
            translatable_rows.append((qid, query_text))
        else:
            skipped_auto += 1

    text_to_qids: dict[str, list[str]] = defaultdict(list)
    for qid, query_text in translatable_rows:
        text_to_qids[query_text].append(qid)

    jobs = list(text_to_qids.items())
    if TRANSLATION_MODE == "auto":
        print(
            f"  Auto mode skipped {skipped_auto:,} queries that already look German."
        )

    print(
        f"  Translating {len(translatable_rows):,} new queries across "
        f"{len(jobs):,} unique texts ..."
    )

    if jobs:
        print(
            f"  Workers: {TRANSLATION_WORKERS}  |  "
            f"Autosave every {TRANSLATION_SAVE_EVERY} completed texts"
        )
        pending_saves = 0
        failed = 0
        warning_samples: list[str] = []
        fatal_error = None
        _TRANSLATION_DISABLED.clear()

        with ThreadPoolExecutor(max_workers=TRANSLATION_WORKERS) as executor:
            future_to_job = {
                executor.submit(translate_to_de, query_text): (query_text, qids)
                for query_text, qids in jobs
            }
            for future in tqdm(
                as_completed(future_to_job),
                total=len(jobs),
                desc="  Translating",
                ncols=80,
                file=sys.stdout,
                ascii=True,
            ):
                _, qids = future_to_job[future]
                try:
                    de = future.result()
                except Exception as e:
                    failed += 1
                    msg = safe(str(e))
                    if len(warning_samples) < TRANSLATION_WARN_LIMIT:
                        warning_samples.append(msg)
                    if fatal_error is None and is_fatal_translation_error(msg):
                        fatal_error = msg
                        _TRANSLATION_DISABLED.set()
                    de = None

                if de:
                    for qid in qids:
                        query_translations[qid] = de

                pending_saves += 1
                if pending_saves >= TRANSLATION_SAVE_EVERY:
                    save_translation_cache(_cache_file, query_translations)
                    pending_saves = 0

        save_translation_cache(_cache_file, query_translations)
        print("  Saved updated translation cache.")

        if failed:
            print(f"  WARNING: {failed:,} translation requests failed.")
            if fatal_error:
                print("  Translation backend appears unavailable; remaining requests were skipped early.")
            for i, msg in enumerate(warning_samples, 1):
                print(f"    Sample {i}: {msg[:220]}")
    else:
        print("  No queries need external translation in this run.")
else:
    print(f"  All {len(all_queries):,} queries already translated.")


# ==============================================================================
# 3. DOCUMENT EXPANSION FROM TRAINING QUERIES (law + court gold)
# ==============================================================================
sep("BUILDING DOCUMENT EXPANSION TOKENS")

_all_labeled = pd.concat([train, val], ignore_index=True)
_n_queries = len(_all_labeled)

# Count how many distinct queries contain each token (for IDF filter)
_tok_query_count: dict[str, set] = defaultdict(set)
for _, r in _all_labeled.iterrows():
    # Use ALL gold (law + court) for expansion eligibility
    if not parse_gold_all(str(r.get("gold_citations", "") or "")):
        continue
    de = query_translations.get(r["query_id"])
    for t in set(build_query_tokens(r["query"], de)):
        _tok_query_count[t].add(r["query_id"])


def _exp_idf(tok: str) -> float:
    df = len(_tok_query_count.get(tok, set()))
    return math.log((_n_queries + 1) / (df + 1)) if df > 0 else 0.0


# Map each cited article/court to the query tokens from queries that cited it
_cit_tok_counter: dict[str, Counter] = {}
_cit_query_count: dict[str, set] = defaultdict(set)

for _, r in _all_labeled.iterrows():
    # Include ALL gold (law + court) for expansion
    gold_all = parse_gold_all(str(r.get("gold_citations", "") or ""))
    if not gold_all:
        continue

    de = query_translations.get(r["query_id"])
    toks = build_query_tokens(r["query"], de)

    for c_raw in gold_all:
        c_norm = _corpus_lower_map.get(c_raw.lower())
        if not c_norm:
            continue
        if c_norm not in _cit_tok_counter:
            _cit_tok_counter[c_norm] = Counter()
        _cit_tok_counter[c_norm].update(toks)
        _cit_query_count[c_norm].add(r["query_id"])

# Keep top-50 discriminative tokens per article
_MIN_QUERY_CITATIONS = 1
_MIN_EXPANSION_IDF   = 2.0

citation_expansion: dict[str, list[str]] = {}
for c, cnt in _cit_tok_counter.items():
    if len(_cit_query_count.get(c, set())) < _MIN_QUERY_CITATIONS:
        continue
    filtered = [t for t, _ in cnt.most_common(100) if _exp_idf(t) >= _MIN_EXPANSION_IDF][:50]
    if filtered:
        citation_expansion[c] = filtered

print(f"  Articles/courts with expansion tokens: {len(citation_expansion):,}")
n_law_exp   = sum(1 for c in citation_expansion if is_law_citation(c))
n_court_exp = len(citation_expansion) - n_law_exp
print(f"    Law: {n_law_exp:,}  |  Court: {n_court_exp:,}")


# ==============================================================================
# 4. BUILD DOCUMENTS
# ==============================================================================
sep("BUILDING DOCUMENT CORPUS (law + court)")


def make_law_doc(row: pd.Series, kb_rec: dict | None, expansion_toks: list[str] | None = None) -> str:
    """
    Law document:
      English: law abbreviation + English law name + group/subgroup/topic
      German:  provision_keywords (x3) + law_keywords (x2) + heading + article text
      Expansion: training query tokens for gold articles (docT5Query-style)
    """
    parts = []

    abbrev = str(row.get("law_abbrev", "") or "")
    name_en = law_name_en_map.get(abbrev, "")
    if abbrev:
        parts.append(abbrev)
    if name_en:
        parts.append(name_en)

    for field in ("group", "subgroup", "article_topic"):
        val_f = str(row.get(field, "") or "")
        if val_f and val_f not in parts:
            parts.append(val_f)

    if kb_rec:
        law_info = kb_rec.get("law", {})
        name_en_kb = law_info.get("law_name_en", "")
        if name_en_kb and name_en_kb != name_en:
            parts.append(name_en_kb)

        sem = kb_rec.get("semantic", {})
        ptype = sem.get("provision_type", "")
        if ptype:
            parts.append(ptype)

        kws = kb_rec.get("keywords", {})
        prov_kws = kws.get("provision_keywords_de") or []
        law_kws  = kws.get("law_keywords_de") or []

        if isinstance(prov_kws, list) and prov_kws:
            kw_str = " ".join(prov_kws)
            parts += [kw_str, kw_str, kw_str]  # x3 boost

        if isinstance(law_kws, list) and law_kws:
            lk_str = " ".join(law_kws)
            parts += [lk_str, lk_str]  # x2 boost

        heading = kb_rec.get("structure", {}).get("context_heading_title", "") or ""
        if heading:
            parts.append(heading)

    german_title = str(row.get("title", "") or "")
    if german_title:
        parts.append(german_title)

    text = str(row.get("text", "") or "")
    if text:
        parts.append(text)

    if expansion_toks:
        parts.append(" ".join(expansion_toks))

    return " ".join(parts)


def make_court_doc(row: pd.Series, expansion_toks: list[str] | None = None) -> str:
    """
    Court document:
      Citation string repeated for token signal (case number is key retrieval signal)
      Combined consideration text (already truncated to COURT_TEXT_LIMIT at load time)
      Expansion tokens from training queries that cited this court entry
    """
    parts = []

    citation = str(row.get("citation", "") or "")
    if citation:
        # Repeat citation tokens to boost case number matching
        parts += [citation, citation]

    text = str(row.get("text", "") or "")
    if text:
        parts.append(text)

    if expansion_toks:
        parts.append(" ".join(expansion_toks))

    return " ".join(parts)


print("  Building law document texts ...")
law_docs = []
n_law_expanded = 0

for _, row in tqdm(
    law_corpus.iterrows(),
    total=len(law_corpus),
    desc="  Law docs",
    ncols=80,
    file=sys.stdout,
    ascii=True,
    mininterval=30,
):
    kb_rec = kb_map.get(row["citation_canon"])
    exp = citation_expansion.get(row["citation_canon"])
    if exp:
        n_law_expanded += 1
    law_docs.append(make_law_doc(row, kb_rec, expansion_toks=exp))

print(f"  Built {len(law_docs):,} law docs  ({n_law_expanded:,} with expansion)")

print("  Building court document texts ...")
court_docs = []
n_court_expanded = 0

for _, row in tqdm(
    court_corpus.iterrows(),
    total=len(court_corpus),
    desc="  Court docs",
    ncols=80,
    file=sys.stdout,
    ascii=True,
    mininterval=30,
):
    exp = citation_expansion.get(row["citation"])
    if exp:
        n_court_expanded += 1
    court_docs.append(make_court_doc(row, expansion_toks=exp))

print(f"  Built {len(court_docs):,} court docs  ({n_court_expanded:,} with expansion)")

# Combine: law first, then court
all_docs = law_docs + court_docs
bm25_ids = law_corpus["citation_canon"].tolist() + court_corpus["citation"].tolist()
assert len(all_docs) == len(bm25_ids)

id_to_row: dict[str, int] = {c: i for i, c in enumerate(bm25_ids)}
print(f"\n  Combined corpus: {len(bm25_ids):,} documents")

print("  Tokenising ...")
tokenised = [
    tokenise(d)
    for d in tqdm(all_docs, desc="  Tokenising", ncols=80, file=sys.stdout, ascii=True, mininterval=30)
]
avg_len = sum(len(t) for t in tokenised) / len(tokenised)
print(f"  Avg tokens per doc: {avg_len:.0f}")


# ==============================================================================
# 5. TOKEN->LAW MAP (for PMI query enhancement)
# ==============================================================================
sep("BUILDING TOKEN->LAW MAP")

token_law_freq: dict[str, Counter] = defaultdict(Counter)

for _, row in tqdm(
    train.iterrows(),
    total=len(train),
    desc="  Mapping",
    ncols=80,
    file=sys.stdout,
    ascii=True,
    mininterval=1,
):
    gold = parse_gold_law(str(row.get("gold_citations", "") or ""))
    if not gold:
        continue

    abbrevs = {c.split()[-1].lower() for c in gold}
    toks = build_query_tokens(row["query"], query_translations.get(row["query_id"]))

    for t in set(toks):
        for ab in abbrevs:
            token_law_freq[t][ab] += 1

_token_total_count: dict[str, int] = {t: sum(c.values()) for t, c in token_law_freq.items()}
print(f"  Token->law map: {len(token_law_freq):,} unique tokens")


# ==============================================================================
# 6. BUILD BM25 INDEX
# ==============================================================================
sep(f"BUILDING BM25 INDEX  (k1={K_F11}, b={BEST_B})")

bm25 = BM25Okapi(
    tqdm(tokenised, desc="  BM25 index", ncols=80, file=sys.stdout, ascii=True, mininterval=5),
    k1=K_F11,
    b=BEST_B,
)
print("  Index built.")


# ==============================================================================
# 7. QUERY ENHANCEMENT
# ==============================================================================
def predict_law_abbrevs_pmi(tokens: list[str], bm25_idf: dict,
                            top_n: int = 5, min_idf: float = 1.0) -> list[tuple[str, float]]:
    """PMI-normalised law prediction: P(law|token) * IDF(token)."""
    scores: dict[str, float] = {}

    for t in set(tokens):
        if t not in token_law_freq:
            continue

        idf = bm25_idf.get(t, 0.0)
        if idf < min_idf:
            continue

        total = _token_total_count.get(t, 1)
        for ab, cnt in token_law_freq[t].items():
            scores[ab] = scores.get(ab, 0.0) + (cnt / total) * idf

    return sorted(scores.items(), key=lambda x: -x[1])[:top_n]


def enhance_tokens(tokens: list[str], top_n_laws: int = 5,
                   law_boost: int = 10) -> list[str]:
    base = list(dict.fromkeys(tokens))
    predicted = predict_law_abbrevs_pmi(tokens, bm25.idf, top_n=top_n_laws, min_idf=1.0)
    for ab, _ in predicted:
        base.extend([ab] * law_boost)
    return base


# ==============================================================================
# 8. QUERY ROW BUILDERS
# ==============================================================================
def build_rows(df: pd.DataFrame) -> list[dict]:
    """Build query rows with FULL gold (law + court), filtered to what's in the corpus."""
    rows = []
    for _, row in df.iterrows():
        qid = row["query_id"]
        gold_raw = parse_gold_all(str(row.get("gold_citations", "") or ""))
        gold = [_corpus_lower_map.get(c.lower(), c) for c in gold_raw]
        gold_in  = [c for c in gold if c in id_to_row]
        gold_mis = [c for c in gold if c not in id_to_row]

        gold_law_raw = parse_gold_law(str(row.get("gold_citations", "") or ""))
        gold_law = [_corpus_lower_map.get(c.lower(), c) for c in gold_law_raw]
        gold_law_in = [c for c in gold_law if c in id_to_row]

        de = query_translations.get(qid)
        toks = build_query_tokens(row["query"], de)

        rows.append({
            "query_id":    qid,
            "query":       row["query"],
            "gold":        gold_in,           # ALL gold in corpus (law+court)
            "gold_set":    set(gold_in),
            "gold_law":    gold_law_in,       # law-only gold in corpus
            "gold_law_set": set(gold_law_in),
            "gold_missing": gold_mis,         # gold not in corpus
            "n_total_gold": len(gold_raw),
            "tokens":      toks,
        })
    return rows


# ==============================================================================
# 9. SAVE INDEX  (saved before evaluation so the index is never lost)
# ==============================================================================
sep("SAVING INDEX")

out_index  = STATUTORY_BM25_PKL
out_ids    = STATUTORY_BM25_IDS
out_lawmap = TOKEN_LAW_FREQ

save_bm25_artifact(bm25, out_index)

with open(out_ids, "wb") as f:
    pickle.dump(
        {
            "citation_canon": bm25_ids,
            "n_law":   len(law_corpus),
            "n_court": len(court_corpus),
            "best_k1": K_F11,
            "best_b":  BEST_B,
            "best_f1_k": K_F1,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

with open(_cache_file, "w", encoding="utf-8") as f:
    json.dump(query_translations, f, ensure_ascii=False, indent=2)

with open(out_lawmap, "w", encoding="utf-8") as f:
    json.dump({t: dict(cnt) for t, cnt in token_law_freq.items()}, f, ensure_ascii=False)

print(f"  Saved: {out_index}")
print(f"  Saved: {out_ids}")
print(f"  Saved: {_cache_file.name}  ({len(query_translations):,} translations)")
print(f"  Saved: {out_lawmap.name}  ({len(token_law_freq):,} tokens)")


# ==============================================================================
# 10. EVALUATION
# ==============================================================================
sep("EVALUATION")


def evaluate(label: str, rows: list[dict]) -> dict:
    """Evaluate recall@K (over full gold: law+court) for each query."""
    results = {}

    for vr in tqdm(rows, desc=f"  Eval {label}", ncols=80, file=sys.stdout, ascii=True, mininterval=5):
        if not vr["gold"]:
            continue

        toks = enhance_tokens(vr["tokens"])
        scores = bm25.get_scores(toks)
        rank = scores.argsort()[::-1]
        c2rank = {bm25_ids[i]: r + 1 for r, i in enumerate(rank)}

        rec_at = {}
        rec_law_at = {}
        for k in TOPK:
            top_k = set(bm25_ids[i] for i in rank[:k])
            rec_at[k]     = sum(1 for c in vr["gold"]     if c in top_k) / len(vr["gold"])
            if vr["gold_law"]:
                rec_law_at[k] = sum(1 for c in vr["gold_law"] if c in top_k) / len(vr["gold_law"])
            else:
                rec_law_at[k] = 0.0

        results[vr["query_id"]] = {
            "recall_at":     rec_at,
            "recall_law_at": rec_law_at,
            "gold_count":    len(vr["gold"]),
            "gold_law_count": len(vr["gold_law"]),
            "gold_ranks":    {c: c2rank.get(c, -1) for c in vr["gold"]},
            "gold_missing":  vr["gold_missing"],
        }

    return results


def print_eval(label: str, results: dict, per_query: bool = True, save_path: Path | None = None):
    print(f"\n  -- {safe(label)} --")

    # --- Combined (law+court) recall ---
    print(f"\n  COMBINED GOLD RECALL (law + court):")
    header = f"  {'QID':<12}" + "".join(f"  R@{k:>4}" for k in TOPK) + f"  {'gold':>5}  {'law':>5}"

    macro_all  = {k: [] for k in TOPK}
    macro_law  = {k: [] for k in TOPK}
    rows_out = []

    for qid, r in sorted(results.items()):
        row_str = f"  {qid:<12}"
        for k in TOPK:
            row_str += f"  {r['recall_at'][k]:>6.3f}"
            macro_all[k].append(r["recall_at"][k])
            macro_law[k].append(r["recall_law_at"][k])
        row_str += f"  {r['gold_count']:>5}  {r['gold_law_count']:>5}"
        rows_out.append(row_str)

    if per_query:
        print(header)
        hr()
        for row_str in rows_out:
            print(row_str)
        hr()

    avg_all_str = (
        f"  {'MACRO AVG':<12}"
        + "".join(f"  {(sum(macro_all[k]) / len(macro_all[k]) if macro_all[k] else 0.0):>6.3f}" for k in TOPK)
    )
    avg_law_str = (
        f"  {'(law only)':<12}"
        + "".join(f"  {(sum(macro_law[k]) / len(macro_law[k]) if macro_law[k] else 0.0):>6.3f}" for k in TOPK)
    )
    print(avg_all_str)
    print(avg_law_str)

    if save_path is not None:
        with open(save_path, "w", encoding="utf-8") as fout:
            fout.write(f"{label}\n")
            fout.write(header + "\n")
            fout.write("-" * 90 + "\n")
            for row_str in rows_out:
                fout.write(row_str + "\n")
            fout.write("-" * 90 + "\n")
            fout.write(avg_all_str + "\n")
            fout.write(avg_law_str + "\n")
        print(f"  Per-query results saved: {save_path}")

    return {
        "all": {k: (sum(macro_all[k]) / len(macro_all[k]) if macro_all[k] else 0.0) for k in TOPK},
        "law": {k: (sum(macro_law[k]) / len(macro_law[k]) if macro_law[k] else 0.0) for k in TOPK},
    }


# --- Build val rows ---
val_rows = build_rows(val)

# --- Build train rows (sample or all) ---
if EVAL_TRAIN_SAMPLE is not None:
    rng = random.Random(RANDOM_SEED)
    train_sample = train.sample(n=min(EVAL_TRAIN_SAMPLE, len(train)), random_state=RANDOM_SEED)
    train_eval_rows = build_rows(train_sample)
    train_label = f"TRAIN (sample n={len(train_eval_rows)})"
else:
    train_eval_rows = build_rows(train)
    train_label = f"TRAIN (all n={len(train_eval_rows)})"

# --- Evaluate ---
val_results = evaluate("val", val_rows)
val_avgs    = print_eval("VAL -- combined law+court recall", val_results, per_query=True,
                         save_path=OUT_DIR / "bm25_val_recall.txt")

train_results = evaluate(train_label, train_eval_rows)
train_avgs    = print_eval(f"TRAIN -- combined law+court recall", train_results, per_query=True,
                           save_path=OUT_DIR / "bm25_train_recall.txt")


# ==============================================================================
# 10. MISSED CITATIONS ANALYSIS (val)
# ==============================================================================
sep("MISSED CITATIONS (val, combined gold)")

for vr in sorted(val_rows, key=lambda x: val_results.get(x["query_id"], {}).get("recall_at", {}).get(1500, 1.0)):
    res = val_results.get(vr["query_id"])
    if not res:
        continue

    r_100  = res["recall_at"].get(100, 0.0)
    r_1500 = res["recall_at"].get(1500, 0.0)
    if r_1500 >= 1.0:
        continue

    print(
        f"\n  {vr['query_id']}  "
        f"R@100={r_100:.3f}  R@1500={r_1500:.3f}  "
        f"gold={res['gold_count']}(law={res['gold_law_count']})"
    )
    for c in vr["gold"]:
        rank = res["gold_ranks"].get(c, -1)
        is_law = " [law]" if is_law_citation(c) else " [court]"
        marker = " <<MISSED@1500>>" if rank < 0 or rank > 1500 else ""
        print(f"    rank={rank:>7}  {c}{is_law}{marker}")
    if vr["gold_missing"]:
        print(f"  Not in corpus ({len(vr['gold_missing'])}): {vr['gold_missing'][:5]}")


# ==============================================================================
# 11. F1 @ K_F1 (val, law-only gold -- for submission quality)
# ==============================================================================
sep(f"F1 @ K={K_F1} (val, law-only gold)")

print(f"  Scoring val queries (top {K_F1}) ...")
val_f1_law, val_f1_oracle = [], []

for vr in tqdm(val_rows, desc="  F1 val", ncols=80, file=sys.stdout, ascii=True):
    if not vr["gold_law"]:
        continue
    toks = enhance_tokens(vr["tokens"])
    raw_scores = bm25.get_scores(toks)
    order = raw_scores.argsort()[::-1][:3000]
    ranked = [bm25_ids[i] for i in order]
    topk   = ranked[:K_F1]

    f1_law = compute_f1(topk, vr["gold_law_set"])
    orc, _ = oracle_f1(ranked, vr["gold_law_set"])
    val_f1_law.append(f1_law)
    val_f1_oracle.append(orc)

macro_f1_law    = sum(val_f1_law) / len(val_f1_law) if val_f1_law else 0.0
macro_f1_oracle = sum(val_f1_oracle) / len(val_f1_oracle) if val_f1_oracle else 0.0

print(f"""
  Macro F1@{K_F1} (law-only gold):   {macro_f1_law:.4f}
  Oracle upper bound (law):          {macro_f1_oracle:.4f}
""")


# ==============================================================================
# 12. FINAL SUMMARY
# ==============================================================================
sep("FINAL SUMMARY")

print(f"""
  Config: BM25Okapi k1={K_F11:.2f}, b={BEST_B:.2f}  |  PMI law_boost=10  |  K_pool={K_F1}
  Index:  {len(law_corpus):,} law  +  {len(court_corpus):,} court  =  {len(bm25_ids):,} total docs

  VAL Combined Recall (law + court gold):
  {'K':>6}  {'All':>8}  {'Law':>8}
  {'-' * 28}""")

for k in TOPK:
    print(f"  {k:>6}  {val_avgs['all'][k]:>8.4f}  {val_avgs['law'][k]:>8.4f}")

print(f"""
  TRAIN Combined Recall ({train_label}):
  {'K':>6}  {'All':>8}  {'Law':>8}
  {'-' * 28}""")

for k in TOPK:
    print(f"  {k:>6}  {train_avgs['all'][k]:>8.4f}  {train_avgs['law'][k]:>8.4f}")

print(f"""
  Macro F1@{K_F1} (val, law-only):  {macro_f1_law:.4f}
  Oracle upper bound (val, law):    {macro_f1_oracle:.4f}

  To use in downstream pipeline:
    index  -> index/bm25_v2_index.pkl
    ids    -> index/bm25_v2_ids.pkl  (first {len(law_corpus):,} = law, rest = court)
""")

print("Done.")


### `indexing/build_citation_graph.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/indexing/build_citation_graph.py"

"""
build_citation_graph.py -- Production-grade bidirectional citation graph builder.

Scans ALL 2.47M court consideration rows ONCE and builds FIVE structures:

  Direct citation graph (Art. ↔ Case, Case ↔ Case):
    article_to_cases:    Art. X → {case_cite: count}   (which cases cite this article?)
    case_to_articles:    case_cite → [Art. X, ...]      (which articles does this case cite?)
    case_to_cases:       case_cite → {other: count}     (which other cases does it cite?)
    article_corpus_freq: Art. X → total citation count  (importance ranking)

  Co-citation graph (symmetric, any ↔ any):
    cocitation:          cite_A → {cite_B: count}       (appear together in same text)
    citation_doc_freq:   cite → num_rows_it_appears_in  (for PMI normalisation)

  Sources extracted from TEXT COLUMN in three languages:
    German:   Art. 221 Abs. 1 StPO,  § 12 Abs. 2 VwVG
    French:   art. 221 al. 1 let. b CPP  → normalised to German form
    Italian:  art. 221 cpv. 1 lett. b CPP → normalised to German form
    Cases:    BGE 139 IV 175 E. 3.1,  1B_210/2023 E. 4.1

  Also ingests train.csv gold citations as co-citation signal.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python indexing/build_citation_graph.py           # full build (~1-2 hours)
#   python indexing/build_citation_graph.py --test    # smoke test only (instant)
#
# ─── VALIDATION CHECKLIST ───────────────────────────────────────────────────
#   After build, check these numbers in the output:
#     ✓ "Rows with Art. refs"       → expect ~50% (40% DE + 10% FR/IT)
#     ✓ "Unique articles in graph"  → expect 10K-50K+
#     ✓ "Case-to-case links"        → expect 100K+
#     ✓ "Co-citation pairs kept"    → expect 500K-5M
#     ✓ Top co-cited pairs should include BGG procedural clusters
#     ✗ If any near zero → a regex is broken
"""

import re
import pickle
import argparse
from itertools import combinations
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
from tqdm import tqdm

import sys
sys.path.insert(0, str(Path(__file__).parent.parent))
from data.data_paths import COURT_CSV, CITATION_GRAPH_PKL, TRAIN_CSV

INDEX_DIR = CITATION_GRAPH_PKL.parent
INDEX_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 200_000

# Co-citation tuning
MAX_CITES_PER_ROW = 30   # cap per row to avoid O(n²) explosion on outliers
MIN_CO_OCCURRENCE  = 3   # prune pairs seen fewer than this many times

def count_csv_rows(path: Path) -> int:
    """Fast newline-based row count for progress reporting (minus header)."""
    total_lines = 0
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            total_lines += block.count(b"\n")
    return max(0, total_lines - 1)


# =============================================================================
# CITATION EXTRACTION PATTERNS
# =============================================================================

# 1a. German statutory: Art. 221 Abs. 1 lit. b StPO
ART_PATTERN = re.compile(
    r"Art\.\s+"
    r"(\d+[a-z]?)"
    r"(?:\s+Abs\.\s+(\d+))?"
    r"(?:\s+lit\.\s+([a-z]))?"
    r"(?:\s+Ziff\.\s+(\d+))?"
    r"\s+([A-Z][A-Za-z]{1,14})",
    re.UNICODE
)

# 1b. French/Italian statutory → normalised to German form
FR_IT_ART_PATTERN = re.compile(
    r"(?<![A-Za-z])art\.\s+"
    r"(\d+[a-z]?)"
    r"(?:\s+(?:al\.|cpv\.)\s+(\d+))?"
    r"(?:\s+(?:let\.|lett\.)\s+([a-z]))?"
    r"(?:\s+(?:ch\.|n[o°]\s*\.?)\s+(\d+))?"
    r"\s+([A-Z][A-Za-z]{1,14})",
    re.UNICODE
)

# French/Italian → German abbreviation mapping
FR_IT_TO_DE: dict[str, str] = {
    "CPP": "StPO",   "CP": "StGB",    "LCR": "SVG",     "DPA": "VStrR",
    "LStup": "BetmG",
    "CC": "ZGB",     "CO": "OR",      "LP": "SchKG",     "CPC": "ZPO",
    "LDIP": "IPRG",  "LFors": "GestG",
    "Cst": "BV",     "Cost": "BV",    "CEDH": "EMRK",   "CEDU": "EMRK",
    "LTF": "BGG",    "LTAF": "VGG",
    "PA": "VwVG",    "LAT": "RPG",    "LPE": "USG",
    "LCart": "KG",   "LAsi": "AsylG", "LEI": "AIG",
    "LPGA": "ATSG",  "LAI": "IVG",    "LAMal": "KVG",
    "LACI": "AVIG",  "LAVS": "AHVG",  "LPP": "BVG",
    "LAINF": "UVG",  "LAA": "UVG",
    "LIFD": "DBG",   "LHID": "StHG",  "TVA": "MWSTG",   "LTVA": "MWSTG",
    "LFINMA": "FINMAG", "LEtr": "AuG",
}

# 2. BGE: BGE 137 IV 122 E. 6.2
BGE_PATTERN = re.compile(
    r"BGE\s+(\d{2,3})\s+([IVX]+)\s+(\d+)"
    r"(?:\s+E\.\s+(\d+(?:\.\d+)*))?",
    re.UNICODE
)

# 3. Numbered cases: 1B_210/2023 E. 4.1
NUMBERED_CASE_PATTERN = re.compile(
    r"(\d+[A-Z]_\d+/\d{4})"
    r"(?:\s+E\.\s+(\d+(?:\.\d+)*))?",
    re.UNICODE
)

# 4. Section sign: § 12 Abs. 2 VwVG → Art. 12 Abs. 2 VwVG
SECTION_PATTERN = re.compile(
    r"§\s*(\d+[a-z]?)"
    r"(?:\s+Abs\.\s+(\d+))?"
    r"\s+([A-Z][A-Za-z]{1,14})",
    re.UNICODE
)

ART_BLACKLIST = frozenset({
    "Nr", "Abs", "Lit", "Ziff", "Bst", "Art", "Rz", "Fn",
    "Bd", "Aufl", "Vgl", "Der", "Die", "Das", "Ein", "Eine",
    "Siehe", "Und", "Oder", "Mit", "Aus", "Auf", "Bei",
    "Dazu", "Zur", "Zum", "Vom", "Vor", "Nach", "Satz",
    "Ingress", "Titel", "Kapitel", "Abschnitt",
})

FR_IT_WORD_BLACKLIST = frozenset({
    "et", "de", "du", "des", "la", "le", "les", "un", "une", "ou",
    "si", "se", "sa", "son", "ses", "ce", "cf", "al", "ss", "ch",
    "par", "no", "cpv", "let", "lett", "alinea", "chiffre",
    "del", "dei", "degli", "della", "delle",
    *[w.lower() for w in ART_BLACKLIST],
})

# =============================================================================
# NORMALIZATION HELPERS
# =============================================================================

def normalize_art_match(m) -> str | None:
    art_num, abs_num, lit, ziff, abbrev = (
        m.group(1), m.group(2), m.group(3), m.group(4), m.group(5)
    )
    if abbrev in ART_BLACKLIST or len(abbrev) < 2:
        return None
    parts = [f"Art. {art_num}"]
    if abs_num: parts.append(f"Abs. {abs_num}")
    if lit:     parts.append(f"lit. {lit}")
    if ziff:    parts.append(f"Ziff. {ziff}")
    parts.append(abbrev)
    return " ".join(parts)


def normalize_fr_it_match(m) -> str | None:
    art_num, abs_num, lit, ziff, abbrev = (
        m.group(1), m.group(2), m.group(3), m.group(4), m.group(5)
    )
    if abbrev.lower() in FR_IT_WORD_BLACKLIST or len(abbrev) < 2:
        return None
    de_abbrev = FR_IT_TO_DE.get(abbrev, abbrev)
    if de_abbrev in ART_BLACKLIST:
        return None
    parts = [f"Art. {art_num}"]
    if abs_num: parts.append(f"Abs. {abs_num}")
    if lit:     parts.append(f"lit. {lit}")
    if ziff:    parts.append(f"Ziff. {ziff}")
    parts.append(de_abbrev)
    return " ".join(parts)


def normalize_section_match(m) -> str | None:
    num, abs_num, abbrev = m.group(1), m.group(2), m.group(3)
    if abbrev in ART_BLACKLIST:
        return None
    parts = [f"Art. {num}"]
    if abs_num: parts.append(f"Abs. {abs_num}")
    parts.append(abbrev)
    return " ".join(parts)


# =============================================================================
# EXTRACTION
# =============================================================================

def extract_all_citations(text: str) -> dict:
    """
    Extract ALL citation types from a court text (DE/FR/IT).
    All article keys are in German canonical form.
    Returns {'articles': [...], 'bge_cases': [...], 'numbered_cases': [...]}.
    """
    articles      = set()
    bge_cases     = set()
    numbered_cases = set()

    for m in ART_PATTERN.finditer(text):
        n = normalize_art_match(m)
        if n: articles.add(n)

    for m in FR_IT_ART_PATTERN.finditer(text):
        n = normalize_fr_it_match(m)
        if n: articles.add(n)

    for m in SECTION_PATTERN.finditer(text):
        n = normalize_section_match(m)
        if n: articles.add(n)

    for m in BGE_PATTERN.finditer(text):
        vol, div, page, erw = m.group(1), m.group(2), m.group(3), m.group(4)
        base = f"BGE {vol} {div} {page}"
        bge_cases.add(f"{base} E. {erw}" if erw else base)

    for m in NUMBERED_CASE_PATTERN.finditer(text):
        case_num, erw = m.group(1), m.group(2)
        numbered_cases.add(f"{case_num} E. {erw}" if erw else case_num)

    return {
        "articles":       list(articles),
        "bge_cases":      list(bge_cases),
        "numbered_cases": list(numbered_cases),
    }


# =============================================================================
# GRAPH BUILDER  (single CSV scan → five structures)
# =============================================================================

def build_citation_graph():
    # ── Direct citation structures ───────────────────────────────────────────
    article_to_cases:    dict[str, Counter] = defaultdict(Counter)
    case_to_articles:    dict[str, set]     = defaultdict(set)
    case_to_cases:       dict[str, Counter] = defaultdict(Counter)
    article_global_freq: Counter            = Counter()

    # ── Co-citation structures ───────────────────────────────────────────────
    cocitation:       dict[str, Counter] = defaultdict(Counter)
    citation_doc_freq: Counter           = Counter()

    print(f"Scanning {COURT_CSV} ...")
    print("  Extracting: statutes, BGE citations, numbered cases, and co-citation pairs")

    total_input_rows = count_csv_rows(COURT_CSV)
    print(f"  Total rows: {total_input_rows:,}")

    chunk_iter = pd.read_csv(
        COURT_CSV, chunksize=CHUNKSIZE, encoding="utf-8",
        usecols=["citation", "text"]
    )

    total_rows      = 0
    total_art_links = 0
    total_case_links = 0
    rows_with_art   = 0
    rows_with_case  = 0
    total_pairs     = 0

    pbar = tqdm(
        total=total_input_rows,
        desc="  Rows scanned",
        unit="row",
        unit_scale=True,
        ncols=80,
        file=sys.stdout,
        ascii=True,
        mininterval=5,
    )

    for chunk in chunk_iter:
        scanned_rows = len(chunk)
        pbar.update(scanned_rows)
        chunk = chunk.dropna(subset=["citation", "text"])
        chunk["citation"] = chunk["citation"].str.strip()
        chunk["text"]     = chunk["text"].astype(str)

        for _, row in chunk.iterrows():
            case_cite  = row["citation"]
            total_rows += 1

            extracted = extract_all_citations(row["text"])

            # ── Direct citation links ────────────────────────────────────────
            if extracted["articles"]:
                rows_with_art += 1
                for art in extracted["articles"]:
                    article_to_cases[art][case_cite] += 1
                    case_to_articles[case_cite].add(art)
                    article_global_freq[art] += 1
                total_art_links += len(extracted["articles"])

            other_cases = extracted["bge_cases"] + extracted["numbered_cases"]
            if other_cases:
                rows_with_case += 1
                for other in other_cases:
                    if other != case_cite:
                        case_to_cases[case_cite][other] += 1
                total_case_links += len(other_cases)

            # ── Co-citation: all pairs in this row ───────────────────────────
            all_cites = set(extracted["articles"])
            all_cites.update(extracted["bge_cases"])
            all_cites.update(extracted["numbered_cases"])
            all_cites.add(case_cite)  # include the row's own citation

            for c in all_cites:
                citation_doc_freq[c] += 1

            # Cap per row to prevent O(n²) explosion
            cites_list = sorted(all_cites)
            if len(cites_list) > MAX_CITES_PER_ROW:
                arts  = [c for c in cites_list if c.startswith("Art.")]
                rest  = [c for c in cites_list if not c.startswith("Art.")]
                cites_list = (arts[:MAX_CITES_PER_ROW // 2]
                              + rest[:MAX_CITES_PER_ROW // 2])

            for a, b in combinations(cites_list, 2):
                cocitation[a][b] += 1
                cocitation[b][a] += 1
                total_pairs += 1

    pbar.close()

    # ── Add train.csv gold co-citation signal ────────────────────────────────
    print(f"\n  Adding train.csv gold co-citation signal ...")
    train_pairs_added = 0
    try:
        train_df = pd.read_csv(TRAIN_CSV, usecols=["gold_citations"])
        for raw in train_df["gold_citations"].dropna():
            cites = [c.strip() for c in str(raw).split(";") if c.strip()]
            # Weight train signal: each gold pair gets +10 (equivalent to 10 rows)
            for a, b in combinations(sorted(set(cites)), 2):
                cocitation[a][b] += 10
                cocitation[b][a] += 10
                citation_doc_freq[a] = citation_doc_freq.get(a, 0) + 1
                citation_doc_freq[b] = citation_doc_freq.get(b, 0) + 1
                train_pairs_added += 1
        print(f"    {len(train_df):,} queries, {train_pairs_added:,} co-citation pairs added")
    except Exception as e:
        print(f"    WARNING: could not read train.csv: {e}")

    # ── Prune co-citation pairs below threshold ───────────────────────────────
    print(f"  Pruning co-citation pairs with count < {MIN_CO_OCCURRENCE} ...")
    pruned_cocitation = {}
    kept = pruned = 0
    for cite_a, neighbors in cocitation.items():
        good = {b: c for b, c in neighbors.items() if c >= MIN_CO_OCCURRENCE}
        if good:
            pruned_cocitation[cite_a] = good
            kept    += len(good)
        pruned += len(neighbors) - len(good)
    del cocitation

    # ── Serialise both graphs into one pickle ────────────────────────────────
    graph = {
        # Direct citation
        "article_to_cases":    {art: dict(cases) for art, cases in article_to_cases.items()},
        "case_to_articles":    {case: sorted(arts) for case, arts in case_to_articles.items()},
        "case_to_cases":       {case: dict(others) for case, others in case_to_cases.items()},
        "article_corpus_freq": dict(article_global_freq),
        # Co-citation
        "cocitation":          pruned_cocitation,
        "citation_doc_freq":   dict(citation_doc_freq),
    }

    with open(CITATION_GRAPH_PKL, "wb") as f:
        pickle.dump(graph, f, protocol=pickle.HIGHEST_PROTOCOL)

    # ── Diagnostics ───────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  CITATION GRAPH BUILD COMPLETE")
    print(f"{'='*60}")
    print(f"  Rows scanned:              {total_rows:>12,}")
    print(f"  Rows with Art. refs:       {rows_with_art:>12,}  ({100*rows_with_art/max(total_rows,1):.1f}%)")
    print(f"  Rows with case refs:       {rows_with_case:>12,}  ({100*rows_with_case/max(total_rows,1):.1f}%)")
    print(f"  Total Art.<->Case links:   {total_art_links:>12,}")
    print(f"  Total Case<->Case links:   {total_case_links:>12,}")
    print(f"  Unique articles in graph:  {len(graph['article_to_cases']):>12,}")
    print(f"  Unique cases with Art.:    {len(graph['case_to_articles']):>12,}")
    print(f"  Unique cases with Case.:   {len(graph['case_to_cases']):>12,}")
    print(f"  Co-citation pairs (total): {total_pairs:>12,}")
    print(f"  Co-citation pairs kept:    {kept:>12,}  (pruned {pruned:,})")
    print(f"  Unique citations in co-g:  {len(pruned_cocitation):>12,}")

    print(f"\n  Top 20 most-cited articles:")
    for art, freq in article_global_freq.most_common(20):
        n_cases = len(graph["article_to_cases"].get(art, {}))
        print(f"    {freq:>6} total, {n_cases:>5} unique cases:  {art}")

    print(f"\n  Top 20 co-cited pairs:")
    all_pairs = [
        (a, b, c)
        for a, nbrs in pruned_cocitation.items()
        for b, c in nbrs.items()
        if a < b
    ]
    all_pairs.sort(key=lambda x: -x[2])
    for a, b, cnt in all_pairs[:20]:
        print(f"    {cnt:>6}×  {a}  ↔  {b}")

    abbrev_counts: Counter = Counter()
    for art in graph["article_to_cases"]:
        abbrev_counts[art.split()[-1]] += 1
    print(f"\n  Top 20 law abbreviations:")
    for abbrev, count in abbrev_counts.most_common(20):
        print(f"    {count:>6} articles:  {abbrev}")

    print(f"\nSaved: {CITATION_GRAPH_PKL}")
    return graph


# =============================================================================
# RUNTIME INTERFACE
# =============================================================================

class CitationGraph:
    """
    Runtime interface: direct citation graph + co-citation graph.
    All loaded from a single citation_graph.pkl.
    """

    def __init__(self, graph_pkl: Path = CITATION_GRAPH_PKL):
        with open(graph_pkl, "rb") as f:
            data = pickle.load(f)
        self.article_to_cases    = data["article_to_cases"]
        self.case_to_articles    = data["case_to_articles"]
        self.case_to_cases       = data.get("case_to_cases", {})
        self.article_corpus_freq = data.get("article_corpus_freq", {})
        self.cocitation          = data.get("cocitation", {})
        self.citation_doc_freq   = data.get("citation_doc_freq", {})

    # ── Direct citation lookups ───────────────────────────────────────────────

    def get_citing_cases(self, art_citation: str, top_n: int = 50) -> list[str]:
        """Article → top citing cases. Fuzzy: strips Abs./lit. if exact fails."""
        key   = art_citation.strip()
        cases = self.article_to_cases.get(key, {})
        if not cases:
            parent = re.sub(r"\s+Abs\.\s+\d+", "", key)
            parent = re.sub(r"\s+lit\.\s+\w+", "", parent)
            if parent != key:
                cases = self.article_to_cases.get(parent, {})
        return [c for c, _ in sorted(cases.items(), key=lambda x: -x[1])[:top_n]] if cases else []

    def get_cited_articles(self, case_citation: str) -> list[str]:
        """Case → articles it cites. Fuzzy: strips E. suffix if exact fails."""
        key    = case_citation.strip()
        result = self.case_to_articles.get(key, [])
        if not result:
            base = re.sub(r"\s+E\.\s+[\d.]+$", "", key)
            if base != key:
                result = self.case_to_articles.get(base, [])
        return result

    def get_related_cases(self, case_citation: str, top_n: int = 20) -> list[str]:
        """Case → other cases it references (case-to-case graph)."""
        key    = case_citation.strip()
        others = self.case_to_cases.get(key, {})
        if not others:
            base = re.sub(r"\s+E\.\s+[\d.]+$", "", key)
            if base != key:
                others = self.case_to_cases.get(base, {})
        return [c for c, _ in sorted(others.items(), key=lambda x: -x[1])[:top_n]]

    def get_article_importance(self, art_citation: str) -> int:
        return self.article_corpus_freq.get(art_citation.strip(), 0)

    # ── Co-citation lookups ───────────────────────────────────────────────────

    def get_co_cited(self, citation: str, top_n: int = 10,
                     min_count: int = 5) -> list[tuple[str, int]]:
        """
        Return top citations that co-occur with this one in court texts.
        Returns [(companion, count), ...] sorted descending by count.
        """
        key       = citation.strip()
        neighbors = self.cocitation.get(key, {})
        filtered  = [(c, n) for c, n in neighbors.items() if n >= min_count]
        return sorted(filtered, key=lambda x: -x[1])[:top_n]

    def get_co_cited_pmi(self, citation: str, top_n: int = 10,
                         total_rows: int = 2_476_315) -> list[tuple[str, float]]:
        """
        Co-citations ranked by PMI — emphasises pairs that are specifically
        associated, not just ubiquitous (e.g. Art. 66 BGG appears everywhere
        but has low PMI with rare articles).
        PMI(A,B) = log( P(A,B) / P(A) / P(B) )
        """
        import math
        key      = citation.strip()
        freq_a   = max(self.citation_doc_freq.get(key, 1), 1)
        neighbors = self.cocitation.get(key, {})
        scored   = []
        for companion, co_count in neighbors.items():
            freq_b = max(self.citation_doc_freq.get(companion, 1), 1)
            pmi    = math.log(max(co_count * total_rows / (freq_a * freq_b), 1e-10))
            scored.append((companion, pmi))
        return sorted(scored, key=lambda x: -x[1])[:top_n]

    def expand_pool_cocitation(self, candidate_pool: list[str],
                               top_n_per_cite: int = 5,
                               min_count: int = 10) -> set[str]:
        """
        Expand a candidate pool using co-citation.
        For each citation, add its top co-cited companions.
        """
        expanded = set(candidate_pool)
        for cite in candidate_pool:
            for companion, _ in self.get_co_cited(cite, top_n=top_n_per_cite,
                                                   min_count=min_count):
                expanded.add(companion)
        return expanded


# =============================================================================
# SMOKE TEST
# =============================================================================

def run_smoke_test():
    import sys
    out = open(sys.stdout.fileno(), mode="w", encoding="utf-8", closefd=False)

    out.write("Smoke test: citation extraction (DE + FR + IT) + co-citation pairs\n\n")

    test_texts = [
        ("BGE 137 IV 122 E. 6.2",
         "Gemaess Art. 221 Abs. 1 StPO ist Untersuchungshaft zulaessig. "
         "Wie in BGE 139 IV 175 E. 3.1 festgehalten. "
         "Vgl. Art. 5 Ziff. 1 EMRK und Art. 31 Abs. 1 BV "
         "sowie Urteil 1B_210/2023 E. 4.1."),
        ("9C_492/2014 E. 2.1",
         "Art. 44 ATSG sieht vor (BGE 137 V 210 E. 3.4). "
         "Art. 28 Abs. 1 IVG. Siehe auch § 12 Abs. 2 VwVG."),
        ("1B_210/2023 E. 4.1",
         "Conformement a l'art. 221 al. 1 let. b CPP, la detention provisoire "
         "ne peut etre ordonnee que si le prevenu est fortement soupconne. "
         "Cf. art. 10 al. 2 Cst. et art. 5 CEDH. Voir aussi BGE 139 IV 270 E. 3.1 "
         "et arret 1B_536/2018 E. 5.1."),
        ("1B_195/2022 E. 2.2.1",
         "Ai sensi dell'art. 221 cpv. 1 lett. b CPP il carcere preventivo "
         "puo essere ordinato solo se vi e il rischio di fuga. "
         "Cfr. art. 10 cpv. 2 Cost. e art. 5 CEDU."),
    ]

    for case_cite, text in test_texts:
        out.write(f"  Case: {case_cite}\n")
        e = extract_all_citations(text)
        out.write(f"    Articles:       {sorted(e['articles'])}\n")
        out.write(f"    BGE cases:      {sorted(e['bge_cases'])}\n")
        out.write(f"    Numbered cases: {sorted(e['numbered_cases'])}\n")

        # Co-citation pairs from this single row
        all_cites = set(e["articles"] + e["bge_cases"] + e["numbered_cases"])
        all_cites.add(case_cite)
        pairs = list(combinations(sorted(all_cites), 2))
        out.write(f"    Co-cite pairs:  {len(pairs)} pairs from {len(all_cites)} citations\n\n")

    # ── Assertions ────────────────────────────────────────────────────────────
    e1 = extract_all_citations(test_texts[0][1])
    assert "Art. 221 Abs. 1 StPO"  in e1["articles"], "FAIL: Art. 221 Abs. 1 StPO (DE)"
    assert "Art. 31 Abs. 1 BV"     in e1["articles"], "FAIL: Art. 31 Abs. 1 BV (DE)"
    assert any("Art. 5" in a and "EMRK" in a for a in e1["articles"]), "FAIL: Art. 5 EMRK (DE)"
    assert any("139 IV 175" in c   for c in e1["bge_cases"]),  "FAIL: BGE 139 IV 175"
    assert any("1B_210/2023" in c  for c in e1["numbered_cases"]), "FAIL: 1B_210/2023"

    e2 = extract_all_citations(test_texts[1][1])
    assert "Art. 44 ATSG"        in e2["articles"], "FAIL: Art. 44 ATSG"
    assert "Art. 28 Abs. 1 IVG"  in e2["articles"], "FAIL: Art. 28 Abs. 1 IVG"
    assert "Art. 12 Abs. 2 VwVG" in e2["articles"], "FAIL: § -> Art. 12 VwVG"
    assert any("137 V 210" in c  for c in e2["bge_cases"]), "FAIL: BGE 137 V 210"

    e3 = extract_all_citations(test_texts[2][1])
    assert "Art. 221 Abs. 1 lit. b StPO" in e3["articles"], "FAIL: FR CPP -> StPO"
    assert any("Art. 10" in a and "BV"   in a for a in e3["articles"]), "FAIL: FR Cst -> BV"
    assert any("Art. 5"  in a and "EMRK" in a for a in e3["articles"]), "FAIL: FR CEDH -> EMRK"
    assert any("139 IV 270" in c  for c in e3["bge_cases"]), "FAIL: BGE 139 IV 270 (FR)"
    assert any("1B_536/2018" in c for c in e3["numbered_cases"]), "FAIL: 1B_536 (FR)"

    e4 = extract_all_citations(test_texts[3][1])
    assert "Art. 221 Abs. 1 lit. b StPO" in e4["articles"], "FAIL: IT CPP -> StPO"
    assert any("Art. 10" in a and "BV"   in a for a in e4["articles"]), "FAIL: IT Cost -> BV"
    assert any("Art. 5"  in a and "EMRK" in a for a in e4["articles"]), "FAIL: IT CEDU -> EMRK"

    # Co-citation: verify pairs are generated
    all_cites_t1 = set(e1["articles"] + e1["bge_cases"] + e1["numbered_cases"])
    pairs_t1 = list(combinations(sorted(all_cites_t1), 2))
    assert len(pairs_t1) >= 6, f"FAIL: expected >=6 co-cite pairs, got {len(pairs_t1)}"

    out.write("  ALL ASSERTIONS PASSED\n")


### `indexing/build_lookup_tables.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/indexing/build_lookup_tables.py"

"""
build_lookup_tables.py -- Stage 0C: Build citation normalization & lookup tables.

Creates:
  - citation_set: all valid citation strings (for existence verification)
  - article_to_abs: Art. 975 ZGB -> [Art. 975 Abs. 1 ZGB, Art. 975 Abs. 2 ZGB, ...]
  - abs_to_article: Art. 975 Abs. 1 ZGB -> Art. 975 ZGB
  - sr_to_abbrev / abbrev_to_sr: bidirectional SR number <-> abbreviation mapping

Output: index/citation_lookup.pkl

# HOW TO RUN:
#   cd E:\swiss-law-pipeline
#   python indexing/build_lookup_tables.py
#
# PowerShell:
#   cd E:\swiss-law-pipeline
#   python indexing/build_lookup_tables.py
"""

import re, pickle
from pathlib import Path
from collections import defaultdict

import pandas as pd

import sys
sys.path.insert(0, str(Path(__file__).parent.parent))
from data.data_paths import LAW_CORPUS_PARQUET, LOOKUP_PKL

INDEX_DIR = LOOKUP_PKL.parent
INDEX_DIR.mkdir(parents=True, exist_ok=True)

_ART_RE = re.compile(r"^Art\.\s+(\d+)(?:\s+Abs\.\s+(\d+))?(?:\s+lit\.\s+(\w+))?\s+(.+)$")


def normalize_citation(c: str) -> str:
    c = re.sub(r"\s+", " ", str(c or "").strip())
    c = re.sub(r"Art\.\s*",  "Art. ",  c)
    c = re.sub(r"Abs\.\s*",  "Abs. ",  c)
    c = re.sub(r"lit\.\s*",  "lit. ",  c)
    c = re.sub(r"Ziff\.\s*", "Ziff. ", c)
    return c.strip()


def build_lookup_tables():
    corpus = pd.read_parquet(LAW_CORPUS_PARQUET, columns=["citation_canon", "title"])
    corpus = corpus.rename(columns={"citation_canon": "citation"})
    if "title" not in corpus.columns:
        corpus["title"] = ""
    corpus["citation"] = corpus["citation"].fillna("").str.strip()
    corpus = corpus[corpus["citation"] != ""]

    citation_set = set(corpus["citation"].tolist())

    # citation_canon -> normalized form
    citation_lower_map = {c.lower(): c for c in citation_set}

    # Group Abs. variants under parent article
    article_to_abs: dict[str, list[str]] = defaultdict(list)
    abs_to_article:  dict[str, str]      = {}

    for cite in citation_set:
        m = _ART_RE.match(cite)
        if m:
            art_num = m.group(1)
            abs_num = m.group(2)
            abbrev  = m.group(4).strip()
            parent  = f"Art. {art_num} {abbrev}"

            if abs_num:
                article_to_abs[parent].append(cite)
                abs_to_article[cite] = parent

    # SR number <-> abbreviation mapping
    # SR numbers are numeric abbreviations like "210", "311.0" etc.
    sr_to_abbrev: dict[str, str] = {}
    abbrev_to_sr: dict[str, str] = {}

    _SR_RE = re.compile(r"^Art\.\s+\d+\S*\s+(\d[\d\.]+)$")  # numeric SR suffix
    _AB_RE = re.compile(r"^Art\.\s+\d+\S*\s+([A-Z]\S+)$")   # text abbreviation

    # Build from title field if available
    if "title" in corpus.columns:
        for _, row in corpus.iterrows():
            cite  = row["citation"]
            title = str(row.get("title", "") or "")
            # Title often contains the full law name with SR number
            sr_m = re.search(r"\bSR\s+(\d[\d\.]+)", title)
            ab_m = _AB_RE.match(cite)
            if sr_m and ab_m:
                sr   = sr_m.group(1)
                abbr = ab_m.group(1)
                sr_to_abbrev[sr]   = abbr
                abbrev_to_sr[abbr] = sr

    lookup = {
        "citation_set":      citation_set,
        "citation_lower_map": citation_lower_map,
        "article_to_abs":    dict(article_to_abs),
        "abs_to_article":    abs_to_article,
        "sr_to_abbrev":      sr_to_abbrev,
        "abbrev_to_sr":      abbrev_to_sr,
    }

    with open(LOOKUP_PKL, "wb") as f:
        pickle.dump(lookup, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"  Citation set:     {len(citation_set):,}")
    print(f"  Parent articles:  {len(article_to_abs):,}")
    print(f"  Abs. entries:     {len(abs_to_article):,}")
    print(f"  SR mappings:      {len(sr_to_abbrev):,}")
    print(f"Saved: {LOOKUP_PKL}")


class CitationLookup:
    """Runtime interface for citation lookup tables."""

    def __init__(self, lookup_pkl: Path = LOOKUP_PKL):
        with open(lookup_pkl, "rb") as f:
            data = pickle.load(f)
        self.citation_set      = data["citation_set"]
        self.lower_map         = data["citation_lower_map"]
        self.article_to_abs    = data["article_to_abs"]
        self.abs_to_article    = data["abs_to_article"]
        self.sr_to_abbrev      = data["sr_to_abbrev"]
        self.abbrev_to_sr      = data["abbrev_to_sr"]

    def exists(self, cite: str) -> bool:
        return cite in self.citation_set or cite.lower() in self.lower_map

    def normalize(self, cite: str) -> str | None:
        """Return canonical form of citation, or None if not found."""
        if cite in self.citation_set:
            return cite
        return self.lower_map.get(cite.lower())

    def expand_to_abs(self, article_cite: str) -> list[str]:
        """Art. 975 ZGB -> [Art. 975 Abs. 1 ZGB, Art. 975 Abs. 2 ZGB, ...]"""
        return self.article_to_abs.get(article_cite, [article_cite])

    def collapse_to_article(self, abs_cite: str) -> str:
        """Art. 975 Abs. 1 ZGB -> Art. 975 ZGB"""
        return self.abs_to_article.get(abs_cite, abs_cite)


### `stage0d_build_citation_signals.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage0d_build_citation_signals.py"

"""
stage0d_build_citation_signals.py -- STAGE 0D: Build deterministic citation signals.

This v2 skeleton emits nested signal cards grouped by the actual structure
types present in the raw Swiss law and court datasets.

Outputs:
  - data/citation_signal_schema_v2.json
  - data/statute_signals.jsonl
  - data/case_signals.jsonl
  - index/citation_signal_lookup.pkl
"""

from __future__ import annotations

import argparse
import json
import pickle
import re
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pandas as pd

sys.path.insert(0, str(Path(__file__).parent))

from data.data_paths import (
    CASE_SIGNALS_JSONL,
    CITATION_GRAPH_PKL,
    CITATION_SIGNAL_LOOKUP_PKL,
    CITATION_SIGNAL_SCHEMA_JSON,
    COURT_CSV,
    KB_JSONL,
    LAW_CORPUS_PARQUET,
    LAWS_DE_CSV,
    LOOKUP_PKL,
    STATUTE_SIGNALS_JSONL,
)
from indexing.build_citation_graph import extract_all_citations


SCHEMA_VERSION = "citation-signals-v2"
CASE_CHUNKSIZE = 200_000
TEXT_SNIPPET_CHARS = 420
GRAPH_NEIGHBOR_LIMIT = 8
KEYWORD_LIMIT = 8
ENUM_ITEM_LIMIT = 8

_TOKEN_RE = re.compile(r"[^\w\d]+", re.UNICODE)
_ART_PARSE_RE = re.compile(
    r"^Art\.\s+(?P<art>\d+)(?P<art_suffix>[a-z]?)"
    r"(?:\s+Abs\.\s+(?P<abs>\d+))?"
    r"(?:\s+lit\.\s+(?P<lit>[a-z]))?"
    r"(?:\s+Ziff\.\s+(?P<ziff>\d+))?"
    r"\s+(?P<law>.+)$",
    re.IGNORECASE,
)
_BGE_RE = re.compile(
    r"^(?P<root>BGE\s+(?P<volume>\d{2,3})\s+(?P<div>[IVX]+)\s+(?P<page>\d+))"
    r"(?:\s+E\.?\s+(?P<path>.+))?$"
)
_NUM_CASE_RE = re.compile(
    r"^(?P<root>(?P<family>\d+[A-Z])_(?P<serial>\d+)/(?P<year>\d{4}))"
    r"(?:\s+E\.?\s+(?P<path>.+))?$"
)
_LEGACY_CASE_RE = re.compile(
    r"^(?P<prefix>[A-Z0-9][A-Z0-9.]*)\s+"
    r"(?P<serial>\d+/\d{2,4})"
    r"(?:\s+(?P<date>\d{2}\.\d{2}\.\d{4}))?"
    r"(?:\s+E\.?\s+(?P<path>.+))?$"
)
_HEADING_PATTERNS = [
    re.compile(r"^(?P<number>\d+(?:\.\d+)?)\.\s*(?P<kind>Abschnitt|Teil|Titel|Kapitel)\s*:?\s*(?P<title>.+)$"),
    re.compile(r"^(?P<number>[IVXLC]+)\.\s*(?P<kind>Abschnitt|Teil|Titel|Kapitel)\s*:?\s*(?P<title>.+)$"),
    re.compile(r"^(?P<number>[A-Za-zÄÖÜäöüß]+)\s+(?P<kind>Abschnitt|Teil|Titel|Kapitel)\s*:?\s*(?P<title>.+)$"),
]
_LETTER_ENUM_RE = re.compile(r"(?:^|[;\n])\s*([a-z])\.\s+([^;\n]+)")
_NUMBER_ENUM_RE = re.compile(r"(?:^|\n)\s*(\d+)\s+([^\n]+)")
_CONDITIONAL_RE = re.compile(r"\b(wenn|falls|sofern|soweit)\b[^.]{0,220}", re.IGNORECASE)
_EXCEPTION_RE = re.compile(r"\b(ausgenommen|vorbehaltlich|jedoch|es sei denn)\b[^.]{0,220}", re.IGNORECASE)
_DATE_LIKE_PATH_RE = re.compile(r"^\d{1,2}\.\d{1,2}\.\d{4}$")

_STOPWORDS = {
    "aber", "als", "also", "am", "an", "auch", "auf", "aus", "bei", "bereits",
    "bis", "dabei", "damit", "dann", "darauf", "dass", "dem", "den", "der",
    "des", "die", "dies", "diese", "dieser", "dieses", "doch", "dort", "durch",
    "ein", "eine", "einer", "eines", "er", "es", "falls", "ferner", "für",
    "gegen", "gemäss", "hat", "haben", "hier", "hin", "hinsichtlich", "im",
    "in", "indem", "insbesondere", "ist", "jede", "jeder", "jedoch", "kann",
    "kein", "keine", "können", "laut", "mit", "nach", "nicht", "noch", "nun",
    "oder", "ohne", "sich", "sie", "sind", "soweit", "sowie", "und", "unter",
    "vom", "von", "vor", "war", "werden", "wird", "wie", "wir", "zu",
    "zum", "zur", "zwar", "über", "art", "abs", "lit", "ziff", "bge", "e",
    "the", "and", "for", "with", "from", "that", "this", "into", "their",
    "would", "such", "only", "under", "while", "which", "whether", "where",
}


def build_signal_schema() -> dict[str, Any]:
    """Return the grouped v2 schema contract for signal-card outputs."""
    return {
        "schema_version": SCHEMA_VERSION,
        "top_level_fields": {
            "citation_canon": "Canonical citation ID used by the pipeline",
            "citation_type": "One of: statute, case",
            "language": "Primary text language of the retained excerpt",
            "identity_structure": "Canonical identity, aliases, and stable IDs",
            "heading_hierarchy_structure": "Law-title and chapter/section hierarchy cues",
            "article_family_structure": "Parent-child article grouping",
            "article_variant_structure": "Paragraph / lit. / Ziff. specificity",
            "intra_article_structure": "Enumerations, conditions, and exceptions inside statute text",
            "case_tree_structure": "Case root and consideration-path hierarchy",
            "case_family_structure": "Case-family prior such as 4A, 6B, 2C",
            "bge_division_structure": "BGE-specific volume/division/page structure",
            "legacy_case_structure": "Legacy non-modern case formats and dates",
            "reference_graph_structure": "Outgoing refs, graph neighbors, and co-citation cues",
            "fragment_aggregation_structure": "Duplicate-row / multi-fragment aggregation metadata",
            "normalization_structure": "Normalization rules and ambiguity flags",
            "retrieval_views": "Ready-to-use compact retrieval strings",
        },
    }


def _clean_text(text: Any, limit: int | None = None) -> str:
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    return text[:limit] if limit is not None else text


def _unique_keep_order(values: list[Any]) -> list[Any]:
    seen = set()
    out = []
    for value in values:
        if value in (None, "", [], {}):
            continue
        if value not in seen:
            seen.add(value)
            out.append(value)
    return out


def _norm_num(value: Any) -> str | None:
    if value is None:
        return None
    if isinstance(value, float):
        if pd.isna(value):
            return None
        if value.is_integer():
            return str(int(value))
        return str(value)
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return None
    if text.endswith(".0") and text[:-2].isdigit():
        return text[:-2]
    return text


def _law_id_from_parts(law_abbrev: str | None, sr_number: str | None, fallback: str | None) -> str | None:
    return law_abbrev or sr_number or fallback


def _slugify(text: str) -> str:
    text = re.sub(r"[^A-Za-z0-9]+", "_", str(text or "").strip())
    return re.sub(r"_+", "_", text).strip("_").lower()


def _top_keywords(text: str, top_k: int = KEYWORD_LIMIT) -> list[str]:
    tokens = []
    for tok in _TOKEN_RE.split(text.lower()):
        if len(tok) < 4:
            continue
        if tok.isdigit():
            continue
        if tok in _STOPWORDS:
            continue
        tokens.append(tok)
    counts = Counter(tokens)
    return [tok for tok, _ in counts.most_common(top_k)]


def _extract_heading_info(title_de: str) -> dict[str, Any]:
    title_de = _clean_text(title_de)
    if not title_de:
        return {
            "law_title_raw": None,
            "law_title_clean": None,
            "heading_path_de": [],
            "heading_types": [],
            "heading_numbers": [],
            "heading_leaf_de": None,
            "heading_depth": 0,
        }

    base, _, heading_tail = title_de.partition(" - ")
    heading_tail = _clean_text(heading_tail)
    heading_path = []
    heading_types = []
    heading_numbers = []
    heading_leaf = None

    if heading_tail:
        heading_leaf = heading_tail
        heading_path.append(heading_tail)
        for pattern in _HEADING_PATTERNS:
            m = pattern.match(heading_tail)
            if m:
                heading_types.append(m.group("kind"))
                heading_numbers.append(m.group("number"))
                heading_leaf = _clean_text(m.group("title"))
                if heading_leaf:
                    heading_path.append(heading_leaf)
                break

    return {
        "law_title_raw": title_de,
        "law_title_clean": base or title_de,
        "heading_path_de": _unique_keep_order(heading_path),
        "heading_types": heading_types,
        "heading_numbers": heading_numbers,
        "heading_leaf_de": heading_leaf,
        "heading_depth": len(_unique_keep_order(heading_path)),
    }


def _extract_enum_items(text: str) -> tuple[list[dict[str, str]], list[dict[str, str]], list[str], list[str]]:
    letter_items = [
        {"label": label, "text": _clean_text(value, limit=240)}
        for label, value in _LETTER_ENUM_RE.findall(text)
    ][:ENUM_ITEM_LIMIT]
    number_items = [
        {"label": label, "text": _clean_text(value, limit=240)}
        for label, value in _NUMBER_ENUM_RE.findall(text)
    ][:ENUM_ITEM_LIMIT]
    conds = [_clean_text(m.group(0), limit=240) for m in _CONDITIONAL_RE.finditer(text)][:6]
    exceptions = [_clean_text(m.group(0), limit=240) for m in _EXCEPTION_RE.finditer(text)][:6]
    return letter_items, number_items, conds, exceptions


def _split_sentences(text: str, limit: int = 6) -> list[str]:
    raw = re.split(r"(?<=[\.\!\?])\s+", text)
    out = []
    for sent in raw:
        sent = _clean_text(sent)
        if sent:
            out.append(sent[:240])
        if len(out) >= limit:
            break
    return out


def _parse_article_parts(citation: str) -> dict[str, Any]:
    m = _ART_PARSE_RE.match(citation.strip())
    if not m:
        return {
            "article_number": None,
            "article_number_base": None,
            "article_suffix": None,
            "paragraph_number": None,
            "letter_item": None,
            "ziff_number": None,
            "law_id_from_citation": None,
            "parse_ok": False,
        }
    art_num = m.group("art")
    art_suffix = m.group("art_suffix") or None
    paragraph = m.group("abs") or None
    letter_item = m.group("lit") or None
    ziff = m.group("ziff") or None
    return {
        "article_number": art_num + (art_suffix or ""),
        "article_number_base": art_num,
        "article_suffix": art_suffix,
        "paragraph_number": paragraph,
        "letter_item": letter_item,
        "ziff_number": ziff,
        "law_id_from_citation": _clean_text(m.group("law")),
        "parse_ok": True,
    }


def _match_keys_for_statute(citation: str, parent_article: str, law_id: str | None) -> list[str]:
    keys = [
        citation,
        citation.lower(),
        citation.replace(".", ""),
        citation.replace(" ", ""),
        parent_article,
    ]
    if law_id:
        keys.extend([law_id, law_id.lower()])
    return _unique_keep_order([_clean_text(k) for k in keys if _clean_text(k)])


def _classify_case_citation(citation: str) -> dict[str, Any]:
    citation = citation.strip()
    m = _BGE_RE.match(citation)
    if m:
        path = _clean_text(m.group("path")) or None
        return {
            "case_root": m.group("root"),
            "consideration_path": path,
            "consideration_segments": path.split(".") if path else [],
            "consideration_depth": len(path.split(".")) if path else 0,
            "is_bge": True,
            "case_family_code": f"BGE_{m.group('div')}",
            "case_year": None,
            "case_serial": m.group("page"),
            "bge_volume": m.group("volume"),
            "bge_division": m.group("div"),
            "bge_page": m.group("page"),
            "is_legacy_case": False,
            "legacy_prefix": None,
            "legacy_date": None,
            "legacy_format_pattern": None,
            "parse_mode": "bge",
        }

    m = _NUM_CASE_RE.match(citation)
    if m:
        path = _clean_text(m.group("path")) or None
        return {
            "case_root": m.group("root"),
            "consideration_path": path,
            "consideration_segments": path.split(".") if path else [],
            "consideration_depth": len(path.split(".")) if path else 0,
            "is_bge": False,
            "case_family_code": m.group("family"),
            "case_year": m.group("year"),
            "case_serial": m.group("serial"),
            "bge_volume": None,
            "bge_division": None,
            "bge_page": None,
            "is_legacy_case": False,
            "legacy_prefix": None,
            "legacy_date": None,
            "legacy_format_pattern": None,
            "parse_mode": "modern",
        }

    m = _LEGACY_CASE_RE.match(citation)
    if m:
        root_parts = [m.group("prefix"), m.group("serial")]
        if m.group("date"):
            root_parts.append(m.group("date"))
        root = " ".join(root_parts)
        path = _clean_text(m.group("path")) or None
        return {
            "case_root": root,
            "consideration_path": path,
            "consideration_segments": path.split(".") if path else [],
            "consideration_depth": len(path.split(".")) if path else 0,
            "is_bge": False,
            "case_family_code": m.group("prefix"),
            "case_year": None,
            "case_serial": m.group("serial"),
            "bge_volume": None,
            "bge_division": None,
            "bge_page": None,
            "is_legacy_case": True,
            "legacy_prefix": m.group("prefix"),
            "legacy_date": m.group("date"),
            "legacy_format_pattern": "prefix serial [date] E.path",
            "parse_mode": "legacy",
        }

    return {
        "case_root": citation,
        "consideration_path": None,
        "consideration_segments": [],
        "consideration_depth": 0,
        "is_bge": citation.startswith("BGE "),
        "case_family_code": None,
        "case_year": None,
        "case_serial": None,
        "bge_volume": None,
        "bge_division": None,
        "bge_page": None,
        "is_legacy_case": False,
        "legacy_prefix": None,
        "legacy_date": None,
        "legacy_format_pattern": None,
        "parse_mode": "fallback",
    }


def _match_keys_for_case(citation: str, case_root: str, case_family: str | None) -> list[str]:
    keys = [
        citation,
        citation.lower(),
        case_root,
        case_root.lower(),
        citation.replace(" ", ""),
    ]
    if case_family:
        keys.extend([case_family, case_family.lower()])
    return _unique_keep_order([_clean_text(k) for k in keys if _clean_text(k)])


def _extract_law_id_from_article(citation: str) -> str | None:
    parsed = _parse_article_parts(citation)
    return parsed["law_id_from_citation"]


def _load_kb_records() -> dict[str, dict[str, Any]]:
    kb_map: dict[str, dict[str, Any]] = {}
    if not KB_JSONL.exists():
        return kb_map
    with open(KB_JSONL, encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except Exception:
                continue
            canon = str(rec.get("citation_canon") or "").strip()
            if canon:
                kb_map[canon] = rec
    return kb_map


def _load_lookup_lower_map() -> dict[str, str]:
    if not LOOKUP_PKL.exists():
        return {}
    with open(LOOKUP_PKL, "rb") as f:
        lookup = pickle.load(f)
    return lookup.get("citation_lower_map", {}) or {}


def _load_graph_data() -> dict[str, Any]:
    if not CITATION_GRAPH_PKL.exists():
        return {}
    with open(CITATION_GRAPH_PKL, "rb") as f:
        graph = pickle.load(f)
    return graph if isinstance(graph, dict) else {}


def _canonicalize_refs(refs: list[str], lower_map: dict[str, str], self_citation: str | None = None) -> list[str]:
    out = []
    for ref in refs:
        ref = _clean_text(ref)
        if not ref:
            continue
        canon = lower_map.get(ref.lower(), ref)
        if self_citation and canon == self_citation:
            continue
        out.append(canon)
    return _unique_keep_order(out)


def _build_statute_aux(raw_laws: pd.DataFrame) -> dict[str, Any]:
    counts = raw_laws["citation"].value_counts().to_dict()
    exact_dups = (
        raw_laws.groupby("citation")[["text", "title"]]
        .apply(lambda g: int(g.duplicated().sum()))
        .to_dict()
    )
    titles = (
        raw_laws.groupby("citation")["title"]
        .apply(lambda s: _unique_keep_order([_clean_text(v) for v in s.tolist()]))
        .to_dict()
    )
    return {
        "row_count": counts,
        "exact_duplicate_count": exact_dups,
        "titles": titles,
    }


def _build_case_fragment_counts(chunksize: int = CASE_CHUNKSIZE) -> dict[str, int]:
    counts: Counter[str] = Counter()
    for chunk in pd.read_csv(COURT_CSV, usecols=["citation"], chunksize=chunksize, encoding="utf-8"):
        counts.update(_clean_text(c) for c in chunk["citation"].fillna("").astype(str) if _clean_text(c))
    return dict(counts)


def _load_law_meta(kb_map: dict[str, dict[str, Any]]) -> tuple[dict[str, dict[str, Any]], pd.DataFrame]:
    corpus = pd.read_parquet(LAW_CORPUS_PARQUET)
    law_meta: dict[str, dict[str, Any]] = {}

    for row in corpus.itertuples(index=False):
        cite = str(getattr(row, "citation_canon", "") or "").strip()
        kb_rec = kb_map.get(cite, {})
        law = kb_rec.get("law", {})
        law_abbrev = str(getattr(row, "law_abbrev", "") or law.get("law_abbreviation") or "").strip() or None
        sr_number = str(law.get("sr_number") or "").strip() or None
        fallback = str(law.get("law_identifier_from_citation") or "").strip() or None
        law_id = _law_id_from_parts(law_abbrev, sr_number, fallback)
        if not law_id:
            continue
        if law_id not in law_meta:
            law_meta[law_id] = {
                "law_id": law_id,
                "law_abbrev": law_abbrev,
                "sr_number": sr_number,
                "law_name_de": str(law.get("law_title_clean") or getattr(row, "law_full_name", "") or "").strip() or None,
                "law_name_en": str(law.get("law_name_en") or "").strip() or None,
                "group": str(getattr(row, "group", "") or "").strip() or None,
                "subgroup": str(getattr(row, "subgroup", "") or "").strip() or None,
            }
    return law_meta, corpus


def _build_statute_views(record: dict[str, Any]) -> dict[str, str | None]:
    identity = record["identity_structure"]
    heading = record["heading_hierarchy_structure"]
    article_variant = record["article_variant_structure"]
    intra = record["intra_article_structure"]
    refs = record["reference_graph_structure"]
    excerpt = record["retrieval_views"]["debug_excerpt_de"]

    lexical_signature = _clean_text(
        " ".join(
            str(x) for x in [
                record["citation_canon"],
                identity.get("law_id"),
                heading.get("law_title_clean"),
                heading.get("heading_leaf_de"),
                article_variant.get("granularity_level"),
            ] if x
        ),
        limit=260,
    )
    keyword_signature_de = _clean_text(
        " ".join(intra.get("enumeration_trigger_terms_de", []) + refs.get("law_ids_referenced", [])),
        limit=260,
    )
    keyword_signature_en = _clean_text(
        " ".join(_unique_keep_order([
            identity.get("law_name_en"),
            record["normalization_structure"].get("law_id_normalized_to"),
        ])),
        limit=220,
    )
    signal_text_de = _clean_text(
        " | ".join(
            str(x) for x in [
                "STATUTE",
                record["citation_canon"],
                identity.get("law_id"),
                heading.get("law_title_clean"),
                heading.get("heading_leaf_de"),
                " ".join(refs.get("law_ids_referenced", [])),
                " ".join(record["intra_article_structure"].get("enumeration_trigger_terms_de", [])[:4]),
            ] if x
        ),
        limit=900,
    )
    signal_text_en = _clean_text(
        " | ".join(
            str(x) for x in [
                "STATUTE",
                record["citation_canon"],
                identity.get("law_id"),
                identity.get("law_name_en"),
                article_variant.get("granularity_level"),
            ] if x
        ),
        limit=700,
    )
    bm25_view = _clean_text(" ".join([signal_text_de, lexical_signature, excerpt or ""]), limit=1400)
    reranker_view = _clean_text(
        " | ".join(
            str(x) for x in [
                record["citation_canon"],
                heading.get("law_title_clean"),
                heading.get("heading_leaf_de"),
                excerpt,
                " ".join(refs.get("outgoing_article_refs", [])[:8]),
            ] if x
        ),
        limit=1200,
    )
    return {
        "signal_text_de": signal_text_de,
        "signal_text_en": signal_text_en,
        "lexical_signature_de": lexical_signature or None,
        "keyword_signature_de": keyword_signature_de or None,
        "keyword_signature_en": keyword_signature_en or None,
        "bm25_view_de": bm25_view or None,
        "reranker_view_de": reranker_view or None,
        "debug_excerpt_de": excerpt or None,
    }


def _build_case_views(record: dict[str, Any]) -> dict[str, str | None]:
    case_tree = record["case_tree_structure"]
    refs = record["reference_graph_structure"]
    case_family = record["case_family_structure"]
    excerpt = record["retrieval_views"]["debug_excerpt_de"]

    lexical_signature = _clean_text(
        " ".join(
            str(x) for x in [
                record["citation_canon"],
                case_tree.get("case_root"),
                case_family.get("case_family_code"),
                " ".join(case_family.get("family_domain_prior_de", [])),
            ] if x
        ),
        limit=260,
    )
    keyword_signature_de = _clean_text(
        " ".join(refs.get("law_ids_referenced", []) + refs.get("outgoing_article_refs", [])[:5]),
        limit=260,
    )
    keyword_signature_en = _clean_text(" ".join(case_family.get("family_domain_prior_en", [])), limit=220)
    signal_text_de = _clean_text(
        " | ".join(
            str(x) for x in [
                "CASE",
                record["citation_canon"],
                case_family.get("case_family_code"),
                " ".join(case_family.get("family_domain_prior_de", [])),
                " ".join(refs.get("outgoing_article_refs", [])[:8]),
                excerpt,
            ] if x
        ),
        limit=1000,
    )
    signal_text_en = _clean_text(
        " | ".join(
            str(x) for x in [
                "CASE",
                record["citation_canon"],
                case_family.get("case_family_code"),
                " ".join(case_family.get("family_domain_prior_en", [])),
                "referenced laws",
                " ".join(refs.get("law_ids_referenced", [])[:8]),
            ] if x
        ),
        limit=700,
    )
    bm25_view = _clean_text(" ".join([signal_text_de, lexical_signature, excerpt or ""]), limit=1500)
    reranker_view = _clean_text(
        " | ".join(
            str(x) for x in [
                record["citation_canon"],
                case_tree.get("case_root"),
                case_tree.get("consideration_path"),
                excerpt,
                " ".join(refs.get("outgoing_article_refs", [])[:8]),
                " ".join(refs.get("outgoing_case_refs", [])[:6]),
            ] if x
        ),
        limit=1300,
    )
    return {
        "signal_text_de": signal_text_de,
        "signal_text_en": signal_text_en,
        "lexical_signature_de": lexical_signature or None,
        "keyword_signature_de": keyword_signature_de or None,
        "keyword_signature_en": keyword_signature_en or None,
        "bm25_view_de": bm25_view or None,
        "reranker_view_de": reranker_view or None,
        "debug_excerpt_de": excerpt or None,
    }


def build_statute_signals(limit: int | None = None) -> dict[str, Any]:
    kb_map = _load_kb_records()
    lower_map = _load_lookup_lower_map()
    graph = _load_graph_data()
    law_meta, corpus = _load_law_meta(kb_map)
    raw_laws = pd.read_csv(LAWS_DE_CSV, usecols=["citation", "text", "title"])
    statute_aux = _build_statute_aux(raw_laws)

    article_to_cases = graph.get("article_to_cases", {}) or {}
    article_corpus_freq = graph.get("article_corpus_freq", {}) or {}
    cocitation = graph.get("cocitation", {}) or {}

    parent_to_children: dict[str, list[str]] = defaultdict(list)
    for row in corpus.itertuples(index=False):
        cite = str(getattr(row, "citation_canon", "") or "").strip()
        parent = str(getattr(row, "citation_parent_raw", "") or "").strip() or cite
        parent_to_children[parent].append(cite)

    lookup = {
        "schema_version": SCHEMA_VERSION,
        "citation_type": {},
        "statute_parent": {},
        "statute_law_id": {},
        "statute_must_match_paragraph": {},
        "parent_to_statute_children": defaultdict(list),
    }

    count = 0
    STATUTE_SIGNALS_JSONL.parent.mkdir(parents=True, exist_ok=True)
    with open(STATUTE_SIGNALS_JSONL, "w", encoding="utf-8") as out:
        for row in corpus.itertuples(index=False):
            citation_canon = str(getattr(row, "citation_canon", "") or "").strip()
            if not citation_canon:
                continue

            kb_rec = kb_map.get(citation_canon, {})
            law = kb_rec.get("law", {})
            references = kb_rec.get("references", {})
            parsed = _parse_article_parts(citation_canon)
            parent_article = str(getattr(row, "citation_parent_raw", "") or "").strip() or citation_canon
            children = _unique_keep_order(parent_to_children.get(parent_article, []))
            siblings = [c for c in children if c != citation_canon]

            law_abbrev = str(getattr(row, "law_abbrev", "") or law.get("law_abbreviation") or "").strip() or None
            sr_number = str(law.get("sr_number") or "").strip() or None
            fallback = parsed["law_id_from_citation"] or str(law.get("law_identifier_from_citation") or "").strip() or None
            law_id = _law_id_from_parts(law_abbrev, sr_number, fallback)
            law_name_de = str(law.get("law_title_clean") or getattr(row, "law_full_name", "") or "").strip() or None
            law_name_en = str(law.get("law_name_en") or "").strip() or None
            title_de = str(getattr(row, "title", "") or "").strip() or None
            heading_info = _extract_heading_info(title_de or law_name_de or "")

            text = (kb_rec.get("content", {}).get("text_clean_de") or getattr(row, "text", "") or "")
            text = str(text or "")
            excerpt = _clean_text(text, limit=TEXT_SNIPPET_CHARS)
            letter_items, number_items, conds, exceptions = _extract_enum_items(text)
            sentences = _split_sentences(text)

            citation_info = kb_rec.get("citation", {})
            raw_variants = []
            raw_variants.extend(citation_info.get("citation_aliases") or [])
            raw_variants.extend(citation_info.get("citation_patterns") or [])
            raw_variants.extend([citation_canon, parent_article])
            raw_variants = _unique_keep_order([_clean_text(v) for v in raw_variants if _clean_text(v)])

            extracted_refs = extract_all_citations(text) if text else {"articles": [], "bge_cases": [], "numbered_cases": []}
            outgoing_article_refs = []
            outgoing_article_refs.extend(references.get("references_out_resolved") or [])
            outgoing_article_refs.extend(references.get("references_out_raw") or [])
            outgoing_article_refs.extend(extracted_refs.get("articles", []))
            outgoing_article_refs = _canonicalize_refs(outgoing_article_refs, lower_map, self_citation=citation_canon)

            outgoing_case_refs = []
            outgoing_case_refs.extend(extracted_refs.get("bge_cases", []))
            outgoing_case_refs.extend(extracted_refs.get("numbered_cases", []))
            outgoing_case_refs = _canonicalize_refs(outgoing_case_refs, lower_map, self_citation=citation_canon)

            law_ids_referenced = _unique_keep_order([_extract_law_id_from_article(c) for c in outgoing_article_refs])
            cited_by_cases_top = [
                case for case, _ in sorted((article_to_cases.get(citation_canon, {}) or {}).items(), key=lambda x: (-x[1], x[0]))[:GRAPH_NEIGHBOR_LIMIT]
            ]
            co_cited_with_top = [
                cite for cite, _ in sorted((cocitation.get(citation_canon, {}) or {}).items(), key=lambda x: (-x[1], x[0]))[:GRAPH_NEIGHBOR_LIMIT]
            ]
            enumeration_trigger_terms = _unique_keep_order(_top_keywords(" ".join([i["text"] for i in letter_items + number_items] + conds + exceptions), top_k=6))
            normalized_to = law_id or parsed["law_id_from_citation"]
            ambiguous_tokens = []
            if normalized_to and (" " in normalized_to or "-" in normalized_to or re.match(r"^\d", normalized_to)):
                ambiguous_tokens.append(normalized_to)

            record = {
                "schema_version": SCHEMA_VERSION,
                "citation_canon": citation_canon,
                "citation_type": "statute",
                "language": "de",
                "identity_structure": {
                    "citation_raw_variants": raw_variants,
                    "citation_match_keys": _match_keys_for_statute(citation_canon, parent_article, law_id),
                    "source_dataset": "laws_de.csv",
                    "stable_group_id": _slugify(f"{law_id}_{parsed['article_number_base'] or parsed['article_number'] or citation_canon}"),
                    "stable_leaf_id": _slugify(citation_canon),
                    "law_id": law_id,
                    "law_abbrev": law_abbrev,
                    "sr_number": sr_number,
                    "law_name_de": law_name_de,
                    "law_name_en": law_name_en,
                },
                "heading_hierarchy_structure": heading_info,
                "article_family_structure": {
                    "parent_article_canon": parent_article,
                    "child_article_canon": citation_canon,
                    "article_number": parsed["article_number"],
                    "article_number_base": parsed["article_number_base"],
                    "article_children": children,
                    "article_siblings": siblings,
                    "has_paragraph_children": any(" Abs. " in c for c in children),
                    "paragraph_count": sum(1 for c in children if " Abs. " in c),
                },
                "article_variant_structure": {
                    "article_suffix": parsed["article_suffix"],
                    "paragraph_number": parsed["paragraph_number"] or _norm_num(getattr(row, "article_para", None)),
                    "letter_item": parsed["letter_item"],
                    "ziff_number": parsed["ziff_number"],
                    "must_match_paragraph": bool(parsed["paragraph_number"] or " Abs. " in citation_canon),
                    "must_match_letter_item": bool(parsed["letter_item"] or parsed["ziff_number"]),
                    "granularity_level": "ziff" if parsed["ziff_number"] else "letter_item" if parsed["letter_item"] else "paragraph" if parsed["paragraph_number"] or " Abs. " in citation_canon else "article",
                },
                "intra_article_structure": {
                    "has_lettered_enumeration": bool(letter_items),
                    "has_numbered_enumeration": bool(number_items),
                    "enumeration_items_de": letter_items + number_items,
                    "enumeration_trigger_terms_de": enumeration_trigger_terms,
                    "sentence_spans": sentences,
                    "conditional_clauses_de": conds,
                    "exception_clauses_de": exceptions,
                },
                "case_tree_structure": {},
                "case_family_structure": {},
                "bge_division_structure": {},
                "legacy_case_structure": {},
                "reference_graph_structure": {
                    "outgoing_article_refs": outgoing_article_refs,
                    "outgoing_case_refs": outgoing_case_refs,
                    "incoming_ref_count": int(article_corpus_freq.get(citation_canon, 0)),
                    "cited_by_cases_top": cited_by_cases_top,
                    "co_cited_with_top": co_cited_with_top,
                    "law_ids_referenced": law_ids_referenced,
                    "graph_degree": int(len(cited_by_cases_top) + len(outgoing_article_refs) + len(co_cited_with_top)),
                    "pmi_neighbors": [],
                },
                "fragment_aggregation_structure": {
                    "row_count_for_citation": int(statute_aux["row_count"].get(citation_canon, 1)),
                    "exact_duplicate_count": int(statute_aux["exact_duplicate_count"].get(citation_canon, 0)),
                    "source_fragments_count": int(statute_aux["row_count"].get(citation_canon, 1)),
                    "merged_text_length": len(text),
                    "fragment_positions": [],
                    "duplicate_titles": statute_aux["titles"].get(citation_canon, []),
                    "aggregation_strategy": "citation-level merge over laws_de rows",
                },
                "normalization_structure": {
                    "normalized_from_raw": citation_canon,
                    "normalization_rules_applied": ["collapse_whitespace", "canonicalize_outgoing_refs", "derive_parent_article"],
                    "date_like_e_detected": False,
                    "ambiguous_tokens": ambiguous_tokens,
                    "law_id_normalized": bool(normalized_to),
                    "law_id_normalized_to": normalized_to,
                    "citation_confidence": 1.0 if parsed["parse_ok"] else 0.7,
                    "needs_manual_review": not parsed["parse_ok"],
                },
                "retrieval_views": {"signal_text_de": None, "signal_text_en": None, "lexical_signature_de": None, "keyword_signature_de": None, "keyword_signature_en": None, "bm25_view_de": None, "reranker_view_de": None, "debug_excerpt_de": excerpt},
            }
            record["retrieval_views"] = _build_statute_views(record)

            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            lookup["citation_type"][citation_canon] = "statute"
            lookup["statute_parent"][citation_canon] = parent_article
            lookup["statute_law_id"][citation_canon] = law_id
            lookup["statute_must_match_paragraph"][citation_canon] = record["article_variant_structure"]["must_match_paragraph"]
            lookup["parent_to_statute_children"][parent_article].append(citation_canon)

            count += 1
            if limit is not None and count >= limit:
                break

    lookup["parent_to_statute_children"] = {parent: _unique_keep_order(children) for parent, children in lookup["parent_to_statute_children"].items()}
    lookup["law_meta"] = law_meta
    lookup["statute_count"] = count
    return lookup


def build_case_signals(
    law_meta: dict[str, dict[str, Any]],
    limit: int | None = None,
    chunksize: int = CASE_CHUNKSIZE,
) -> dict[str, Any]:
    lower_map = _load_lookup_lower_map()
    graph = _load_graph_data()
    case_to_articles = graph.get("case_to_articles", {}) or {}
    case_to_cases = graph.get("case_to_cases", {}) or {}
    cocitation = graph.get("cocitation", {}) or {}
    citation_doc_freq = graph.get("citation_doc_freq", {}) or {}
    fragment_counts = _build_case_fragment_counts(chunksize=chunksize)

    lookup = {
        "citation_type": {},
        "case_root": {},
        "case_family": {},
        "case_has_consideration": {},
        "case_count": 0,
    }

    count = 0
    seen: set[str] = set()
    CASE_SIGNALS_JSONL.parent.mkdir(parents=True, exist_ok=True)

    with open(CASE_SIGNALS_JSONL, "w", encoding="utf-8") as out:
        chunk_iter = pd.read_csv(COURT_CSV, usecols=["citation", "text"], chunksize=chunksize, encoding="utf-8")
        for chunk in chunk_iter:
            for row in chunk.itertuples(index=False):
                citation = _clean_text(getattr(row, "citation", ""))
                if not citation or citation in seen:
                    continue
                seen.add(citation)

                text = str(getattr(row, "text", "") or "")
                excerpt = _clean_text(text, limit=TEXT_SNIPPET_CHARS)
                parsed = _classify_case_citation(citation)
                extracted_refs = extract_all_citations(text) if text else {"articles": [], "bge_cases": [], "numbered_cases": []}

                outgoing_article_refs = _canonicalize_refs(case_to_articles.get(citation, []) or [], lower_map)
                if not outgoing_article_refs:
                    outgoing_article_refs = _canonicalize_refs(extracted_refs.get("articles", []), lower_map)

                outgoing_case_refs = []
                outgoing_case_refs.extend(extracted_refs.get("bge_cases", []))
                outgoing_case_refs.extend(extracted_refs.get("numbered_cases", []))
                outgoing_case_refs.extend((case_to_cases.get(citation, {}) or {}).keys())
                outgoing_case_refs = _canonicalize_refs(outgoing_case_refs, lower_map, self_citation=citation)

                law_ids_referenced = _unique_keep_order([_extract_law_id_from_article(art) for art in outgoing_article_refs])
                family_domain_prior_de = _unique_keep_order([law_meta.get(law_id, {}).get("group") for law_id in law_ids_referenced])
                family_domain_prior_en = _unique_keep_order([law_meta.get(law_id, {}).get("law_name_en") for law_id in law_ids_referenced])[:4]
                co_cited_with_top = [other for other, _ in sorted((cocitation.get(citation, {}) or {}).items(), key=lambda x: (-x[1], x[0]))[:GRAPH_NEIGHBOR_LIMIT]]
                parent_consideration = ".".join(parsed["consideration_segments"][:-1]) if len(parsed["consideration_segments"]) > 1 else None

                citation_confidence = 0.95
                if parsed["parse_mode"] == "legacy":
                    citation_confidence = 0.85
                elif parsed["parse_mode"] == "fallback":
                    citation_confidence = 0.65

                record = {
                    "schema_version": SCHEMA_VERSION,
                    "citation_canon": citation,
                    "citation_type": "case",
                    "language": "de",
                    "identity_structure": {
                        "citation_raw_variants": [citation, parsed["case_root"]],
                        "citation_match_keys": _match_keys_for_case(citation, parsed["case_root"], parsed["case_family_code"]),
                        "source_dataset": "court_considerations.csv",
                        "stable_group_id": _slugify(parsed["case_root"]),
                        "stable_leaf_id": _slugify(citation),
                        "law_id": None,
                        "law_abbrev": None,
                        "sr_number": None,
                        "law_name_de": None,
                        "law_name_en": None,
                    },
                    "heading_hierarchy_structure": {"law_title_raw": None, "law_title_clean": None, "heading_path_de": [], "heading_types": [], "heading_numbers": [], "heading_leaf_de": None, "heading_depth": 0},
                    "article_family_structure": {"parent_article_canon": None, "child_article_canon": None, "article_number": None, "article_number_base": None, "article_children": [], "article_siblings": [], "has_paragraph_children": False, "paragraph_count": 0},
                    "article_variant_structure": {"article_suffix": None, "paragraph_number": None, "letter_item": None, "ziff_number": None, "must_match_paragraph": False, "must_match_letter_item": False, "granularity_level": None},
                    "intra_article_structure": {"has_lettered_enumeration": False, "has_numbered_enumeration": False, "enumeration_items_de": [], "enumeration_trigger_terms_de": [], "sentence_spans": [], "conditional_clauses_de": [], "exception_clauses_de": []},
                    "case_tree_structure": {
                        "case_root": parsed["case_root"],
                        "consideration_path": parsed["consideration_path"],
                        "consideration_segments": parsed["consideration_segments"],
                        "consideration_depth": parsed["consideration_depth"],
                        "consideration_parent": parent_consideration,
                        "consideration_children": [],
                        "leaf_vs_internal": "leaf",
                    },
                    "case_family_structure": {
                        "case_family_code": parsed["case_family_code"],
                        "case_family_rank": None,
                        "case_year": parsed["case_year"],
                        "case_serial": parsed["case_serial"],
                        "family_domain_prior_de": family_domain_prior_de,
                        "family_domain_prior_en": family_domain_prior_en,
                    },
                    "bge_division_structure": {
                        "is_bge": parsed["is_bge"],
                        "bge_volume": parsed["bge_volume"],
                        "bge_division": parsed["bge_division"],
                        "bge_page": parsed["bge_page"],
                        "bge_division_prior_de": [family_domain_prior_de[0]] if parsed["is_bge"] and family_domain_prior_de else [],
                        "bge_division_prior_en": [family_domain_prior_en[0]] if parsed["is_bge"] and family_domain_prior_en else [],
                    },
                    "legacy_case_structure": {
                        "is_legacy_case": parsed["is_legacy_case"],
                        "legacy_prefix": parsed["legacy_prefix"],
                        "legacy_root": parsed["case_root"] if parsed["is_legacy_case"] else None,
                        "legacy_date": parsed["legacy_date"],
                        "legacy_consideration_token": parsed["consideration_path"],
                        "legacy_format_pattern": parsed["legacy_format_pattern"],
                        "legacy_normalized_root": parsed["case_root"] if parsed["is_legacy_case"] else None,
                    },
                    "reference_graph_structure": {
                        "outgoing_article_refs": outgoing_article_refs,
                        "outgoing_case_refs": outgoing_case_refs,
                        "incoming_ref_count": int(citation_doc_freq.get(citation, 0)),
                        "cited_by_cases_top": [],
                        "co_cited_with_top": co_cited_with_top,
                        "law_ids_referenced": law_ids_referenced,
                        "graph_degree": int(len(outgoing_article_refs) + len(outgoing_case_refs) + len(co_cited_with_top)),
                        "pmi_neighbors": [],
                    },
                    "fragment_aggregation_structure": {
                        "row_count_for_citation": int(fragment_counts.get(citation, 1)),
                        "exact_duplicate_count": max(int(fragment_counts.get(citation, 1)) - 1, 0),
                        "source_fragments_count": int(fragment_counts.get(citation, 1)),
                        "merged_text_length": len(text),
                        "fragment_positions": [],
                        "duplicate_titles": [],
                        "aggregation_strategy": "first-fragment text plus citation-count aggregation",
                    },
                    "normalization_structure": {
                        "normalized_from_raw": citation,
                        "normalization_rules_applied": ["collapse_whitespace", "parse_case_root", "canonicalize_outgoing_refs"],
                        "date_like_e_detected": bool(parsed["consideration_path"] and _DATE_LIKE_PATH_RE.match(parsed["consideration_path"])),
                        "ambiguous_tokens": [parsed["legacy_prefix"]] if parsed["is_legacy_case"] and parsed["legacy_prefix"] else [],
                        "law_id_normalized": False,
                        "law_id_normalized_to": None,
                        "citation_confidence": citation_confidence,
                        "needs_manual_review": parsed["parse_mode"] == "fallback" or bool(parsed["consideration_path"] and _DATE_LIKE_PATH_RE.match(parsed["consideration_path"])),
                    },
                    "retrieval_views": {"signal_text_de": None, "signal_text_en": None, "lexical_signature_de": None, "keyword_signature_de": None, "keyword_signature_en": None, "bm25_view_de": None, "reranker_view_de": None, "debug_excerpt_de": excerpt},
                }
                record["retrieval_views"] = _build_case_views(record)

                out.write(json.dumps(record, ensure_ascii=False) + "\n")
                lookup["citation_type"][citation] = "case"
                lookup["case_root"][citation] = parsed["case_root"]
                lookup["case_family"][citation] = parsed["case_family_code"]
                lookup["case_has_consideration"][citation] = bool(parsed["consideration_path"])

                count += 1
                if limit is not None and count >= limit:
                    lookup["case_count"] = count
                    return lookup

    lookup["case_count"] = count
    return lookup


def write_schema_json() -> None:
    CITATION_SIGNAL_SCHEMA_JSON.parent.mkdir(parents=True, exist_ok=True)
    with open(CITATION_SIGNAL_SCHEMA_JSON, "w", encoding="utf-8") as f:
        json.dump(build_signal_schema(), f, ensure_ascii=False, indent=2)


def build_all(
    statutes_only: bool = False,
    cases_only: bool = False,
    limit_statutes: int | None = None,
    limit_cases: int | None = None,
    case_chunksize: int = CASE_CHUNKSIZE,
) -> dict[str, Any]:
    if statutes_only and cases_only:
        raise ValueError("Choose at most one of statutes_only / cases_only.")

    t0 = time.time()
    write_schema_json()

    lookup: dict[str, Any] = {
        "schema_version": SCHEMA_VERSION,
        "signal_files": {
            "schema_json": str(CITATION_SIGNAL_SCHEMA_JSON),
            "statute_jsonl": str(STATUTE_SIGNALS_JSONL),
            "case_jsonl": str(CASE_SIGNALS_JSONL),
        },
    }

    if not cases_only:
        print("Building statute signals ...")
        statute_lookup = build_statute_signals(limit=limit_statutes)
        lookup.update(statute_lookup)
        print(f"  Statute signals: {statute_lookup['statute_count']:,}")

    law_meta = lookup.get("law_meta", {})
    if not law_meta and not statutes_only:
        law_meta, _ = _load_law_meta(_load_kb_records())
        lookup["law_meta"] = law_meta

    if not statutes_only:
        print("Building case signals ...")
        case_lookup = build_case_signals(
            law_meta=law_meta,
            limit=limit_cases,
            chunksize=case_chunksize,
        )
        for key, value in case_lookup.items():
            if key == "citation_type" and "citation_type" in lookup:
                lookup["citation_type"].update(value)
            else:
                lookup[key] = value
        print(f"  Case signals: {case_lookup['case_count']:,}")

    lookup["build_seconds"] = round(time.time() - t0, 2)

    CITATION_SIGNAL_LOOKUP_PKL.parent.mkdir(parents=True, exist_ok=True)
    with open(CITATION_SIGNAL_LOOKUP_PKL, "wb") as f:
        pickle.dump(lookup, f, protocol=pickle.HIGHEST_PROTOCOL)

    return lookup


### `stage0e_build_reference_graph_v2.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage0e_build_reference_graph_v2.py"

"""
stage0e_build_reference_graph_v2.py -- STAGE 0E: Build typed reference graph v2.

Builds a compact reference artifact around four primary typed edge sets:

  - article_to_articles: statute -> directly referenced statutes
  - article_to_cases:    statute -> cases citing that statute
  - case_to_articles:    case -> directly cited statutes
  - case_to_cases:       case -> directly cited cases

It also stores reverse views where they are not already implied by the primary
sets, plus lightweight case-root helpers for later graph expansion.

Inputs:
  - data/laws_de.csv
  - data/laws_knowledge_base.jsonl
  - index/citation_lookup.pkl
  - index/citation_graph.pkl

Output:
  - index/reference_graph_v2.pkl
"""

from __future__ import annotations

import argparse
import json
import pickle
import re
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pandas as pd

sys.path.insert(0, str(Path(__file__).parent))

from data.data_paths import (
    CITATION_GRAPH_PKL,
    KB_JSONL,
    LAWS_DE_CSV,
    LOOKUP_PKL,
    REFERENCE_GRAPH_V2_PKL,
)
from indexing.build_citation_graph import build_citation_graph, extract_all_citations
from indexing.build_lookup_tables import build_lookup_tables


SCHEMA_VERSION = "reference-graph-v2"
LAWS_CHUNKSIZE = 50_000

_WS_RE = re.compile(r"\s+")
_BGE_RE = re.compile(
    r"^(?P<root>BGE\s+(?P<volume>\d{2,3})\s+(?P<div>[IVX]+)\s+(?P<page>\d+))"
    r"(?:\s+E\.?\s+(?P<path>.+))?$"
)
_NUM_CASE_RE = re.compile(
    r"^(?P<root>(?P<family>\d+[A-Z])_(?P<serial>\d+)/(?P<year>\d{4}))"
    r"(?:\s+E\.?\s+(?P<path>.+))?$"
)
_LEGACY_CASE_RE = re.compile(
    r"^(?P<prefix>[A-Z0-9][A-Z0-9.]*)\s+"
    r"(?P<serial>\d+/\d{2,4})"
    r"(?:\s+(?P<date>\d{2}\.\d{2}\.\d{4}))?"
    r"(?:\s+E\.?\s+(?P<path>.+))?$"
)
_BARE_ARTICLE_REF_RE = re.compile(
    r"^(?:Art\.|Artikel)\s+"
    r"(?P<art>\d+[a-z]?)"
    r"(?:\s+Abs\.\s+(?P<abs>\d+))?"
    r"(?:\s+lit\.\s+(?P<lit>[a-z]))?"
    r"(?:\s+Ziff\.\s+(?P<ziff>\d+))?$",
    re.IGNORECASE,
)


def _clean_text(text: Any) -> str:
    return _WS_RE.sub(" ", str(text or "").strip())


def _load_lookup_data() -> dict[str, Any]:
    if not LOOKUP_PKL.exists():
        print("Lookup tables missing; building Stage 0C first ...")
        build_lookup_tables()
    with open(LOOKUP_PKL, "rb") as f:
        lookup = pickle.load(f)
    return lookup if isinstance(lookup, dict) else {}


def _load_citation_graph() -> dict[str, Any]:
    if not CITATION_GRAPH_PKL.exists():
        print("Citation graph missing; building Stage 0B first ...")
        build_citation_graph()
    with open(CITATION_GRAPH_PKL, "rb") as f:
        graph = pickle.load(f)
    return graph if isinstance(graph, dict) else {}


def _load_kb_records() -> dict[str, dict[str, Any]]:
    kb_map: dict[str, dict[str, Any]] = {}
    if not KB_JSONL.exists():
        return kb_map
    with open(KB_JSONL, encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except Exception:
                continue
            canon = _clean_text(rec.get("citation_canon"))
            if canon:
                kb_map[canon] = rec
    return kb_map


def _canonicalize_statute(citation: str, lower_map: dict[str, str]) -> str | None:
    citation = _clean_text(citation)
    if not citation or not citation.startswith("Art. "):
        return None
    return lower_map.get(citation.lower())


def _canonicalize_case(citation: str) -> str | None:
    citation = _clean_text(citation)
    return citation or None


def _classify_case_citation(citation: str) -> dict[str, Any]:
    citation = _clean_text(citation)

    m = _BGE_RE.match(citation)
    if m:
        path = _clean_text(m.group("path")) or None
        return {
            "case_root": m.group("root"),
            "consideration_path": path,
            "case_family_code": f"BGE_{m.group('div')}",
            "parse_mode": "bge",
        }

    m = _NUM_CASE_RE.match(citation)
    if m:
        path = _clean_text(m.group("path")) or None
        return {
            "case_root": m.group("root"),
            "consideration_path": path,
            "case_family_code": m.group("family"),
            "parse_mode": "modern",
        }

    m = _LEGACY_CASE_RE.match(citation)
    if m:
        root_parts = [m.group("prefix"), m.group("serial")]
        if m.group("date"):
            root_parts.append(m.group("date"))
        path = _clean_text(m.group("path")) or None
        return {
            "case_root": " ".join(root_parts),
            "consideration_path": path,
            "case_family_code": m.group("prefix"),
            "parse_mode": "legacy",
        }

    return {
        "case_root": citation,
        "consideration_path": None,
        "case_family_code": None,
        "parse_mode": "fallback",
    }


def _is_case_citation(citation: str) -> bool:
    citation = _clean_text(citation)
    if not citation or citation.startswith("Art. "):
        return False
    return bool(_BGE_RE.match(citation) or _NUM_CASE_RE.match(citation) or _LEGACY_CASE_RE.match(citation))


def _iter_kb_outgoing_refs(rec: dict[str, Any]) -> list[str]:
    refs = rec.get("references", {}) or {}
    raw_refs = []
    raw_refs.extend(refs.get("references_out_resolved") or [])
    raw_refs.extend(refs.get("references_out_raw") or [])
    out = []
    for ref in raw_refs:
        if isinstance(ref, dict):
            text = (
                ref.get("ref_citation_raw")
                or ref.get("citation")
                or ref.get("citation_canon")
                or ref.get("text")
            )
        else:
            text = ref
        text = _clean_text(text)
        if text:
            out.append(text)
    return out


def _source_law_id_from_statute(citation: str) -> str | None:
    parts = _clean_text(citation).split()
    if len(parts) >= 3 and parts[0] == "Art.":
        return parts[-1]
    return None


def _law_id_from_ref_law_id(ref_law_id: str | None) -> str | None:
    ref_law_id = _clean_text(ref_law_id)
    if not ref_law_id:
        return None
    m = re.search(r"-(?P<law>[^_]+)_art", ref_law_id)
    if m:
        return m.group("law")
    m = re.search(r"(?P<law>[^_]+)_art", ref_law_id)
    if m:
        return m.group("law")
    return None


def _resolve_kb_statute_targets(
    ref_entry: Any,
    source_citation: str,
    lower_map: dict[str, str],
    article_to_abs: dict[str, list[str]],
) -> list[str]:
    if isinstance(ref_entry, dict):
        raw_ref = (
            ref_entry.get("ref_citation_raw")
            or ref_entry.get("citation")
            or ref_entry.get("citation_canon")
            or ref_entry.get("text")
        )
        ref_law_id = ref_entry.get("ref_law_id")
    else:
        raw_ref = ref_entry
        ref_law_id = None

    raw_ref = _clean_text(raw_ref)
    if not raw_ref:
        return []

    direct = lower_map.get(raw_ref.lower())
    if direct:
        return [direct]

    normalized = raw_ref.replace("Artikel ", "Art. ")
    normalized = normalized.replace("artikel ", "Art. ")
    direct = lower_map.get(normalized.lower())
    if direct:
        return [direct]

    m = _BARE_ARTICLE_REF_RE.match(normalized)
    if not m:
        return []

    law_id = _law_id_from_ref_law_id(ref_law_id) or _source_law_id_from_statute(source_citation)
    if not law_id:
        return []

    parts = [f"Art. {m.group('art')}"]
    if m.group("abs"):
        parts.append(f"Abs. {m.group('abs')}")
    if m.group("lit"):
        parts.append(f"lit. {m.group('lit')}")
    if m.group("ziff"):
        parts.append(f"Ziff. {m.group('ziff')}")
    parts.append(law_id)
    candidate = " ".join(parts)

    direct = lower_map.get(candidate.lower())
    if direct:
        return [direct]

    if not m.group("abs") and not m.group("lit") and not m.group("ziff"):
        return article_to_abs.get(candidate, [])
    return []


def _reverse_counter_map(source_map: dict[str, Counter]) -> dict[str, Counter]:
    reverse: dict[str, Counter] = defaultdict(Counter)
    for src, targets in source_map.items():
        for tgt, count in targets.items():
            reverse[tgt][src] += int(count)
    return reverse


def _serialise_counter_map(source_map: dict[str, Counter]) -> dict[str, dict[str, int]]:
    return {
        src: {tgt: int(count) for tgt, count in targets.items() if int(count) > 0}
        for src, targets in source_map.items()
        if targets
    }


def _edge_stats(source_map: dict[str, Counter]) -> dict[str, int]:
    target_nodes = set()
    unique_edges = 0
    weighted_total = 0
    for targets in source_map.values():
        unique_edges += len(targets)
        weighted_total += sum(int(v) for v in targets.values())
        target_nodes.update(targets.keys())
    return {
        "source_nodes": len(source_map),
        "target_nodes": len(target_nodes),
        "unique_edges": unique_edges,
        "weighted_total": weighted_total,
    }


def _build_statute_reference_edges(
    lookup_data: dict[str, Any],
    limit_laws: int | None = None,
    chunksize: int = LAWS_CHUNKSIZE,
) -> tuple[dict[str, Counter], dict[str, Counter], dict[str, int]]:
    lower_map = lookup_data.get("citation_lower_map", {}) or {}
    article_to_abs = lookup_data.get("article_to_abs", {}) or {}
    article_to_articles: dict[str, Counter] = defaultdict(Counter)
    article_to_cases_direct_refs: dict[str, Counter] = defaultdict(Counter)
    kb_map = _load_kb_records()

    stats = {
        "law_rows_scanned": 0,
        "law_rows_with_article_refs": 0,
        "law_rows_with_case_refs": 0,
        "law_article_refs_from_text": 0,
        "law_case_refs_from_text": 0,
        "law_article_refs_from_kb": 0,
        "law_case_refs_from_kb": 0,
        "unknown_statute_targets_dropped": 0,
    }

    chunk_iter = pd.read_csv(
        LAWS_DE_CSV,
        usecols=["citation", "text"],
        dtype=str,
        chunksize=chunksize,
        on_bad_lines="skip",
    )

    processed = 0
    for chunk in chunk_iter:
        for row in chunk.itertuples(index=False):
            src = _canonicalize_statute(row.citation, lower_map)
            if not src:
                continue

            stats["law_rows_scanned"] += 1
            processed += 1
            extracted = extract_all_citations(_clean_text(row.text))

            article_hits = 0
            for raw_tgt in extracted["articles"]:
                tgt = _canonicalize_statute(raw_tgt, lower_map)
                if not tgt:
                    stats["unknown_statute_targets_dropped"] += 1
                    continue
                if tgt == src:
                    continue
                article_to_articles[src][tgt] += 1
                article_hits += 1
                stats["law_article_refs_from_text"] += 1

            case_hits = 0
            for raw_tgt in extracted["bge_cases"] + extracted["numbered_cases"]:
                tgt = _canonicalize_case(raw_tgt)
                if not tgt:
                    continue
                article_to_cases_direct_refs[src][tgt] += 1
                case_hits += 1
                stats["law_case_refs_from_text"] += 1

            if article_hits:
                stats["law_rows_with_article_refs"] += 1
            if case_hits:
                stats["law_rows_with_case_refs"] += 1

            if limit_laws is not None and processed >= limit_laws:
                break
        if limit_laws is not None and processed >= limit_laws:
            break

    for src, rec in kb_map.items():
        src_canon = _canonicalize_statute(src, lower_map)
        if not src_canon:
            continue
        refs = rec.get("references", {}) or {}
        raw_entries = []
        raw_entries.extend(refs.get("references_out_resolved") or [])
        raw_entries.extend(refs.get("references_out_raw") or [])
        for ref_entry in raw_entries:
            article_targets = _resolve_kb_statute_targets(
                ref_entry=ref_entry,
                source_citation=src_canon,
                lower_map=lower_map,
                article_to_abs=article_to_abs,
            )
            if article_targets:
                added = 0
                for tgt in article_targets:
                    if tgt == src_canon:
                        continue
                    article_to_articles[src_canon][tgt] += 1
                    added += 1
                stats["law_article_refs_from_kb"] += added
                continue

            if isinstance(ref_entry, dict):
                raw_ref = (
                    ref_entry.get("ref_citation_raw")
                    or ref_entry.get("citation")
                    or ref_entry.get("citation_canon")
                    or ref_entry.get("text")
                )
            else:
                raw_ref = ref_entry
            raw_ref = _clean_text(raw_ref)
            if _is_case_citation(raw_ref):
                tgt = _canonicalize_case(raw_ref)
                if tgt:
                    article_to_cases_direct_refs[src_canon][tgt] += 1
                    stats["law_case_refs_from_kb"] += 1
            elif raw_ref:
                stats["unknown_statute_targets_dropped"] += 1

    return article_to_articles, article_to_cases_direct_refs, stats


def _build_case_reference_edges(
    lower_map: dict[str, str],
) -> tuple[dict[str, Counter], dict[str, Counter], dict[str, Counter], dict[str, int]]:
    article_to_cases: dict[str, Counter] = defaultdict(Counter)
    case_to_articles: dict[str, Counter] = defaultdict(Counter)
    case_to_cases: dict[str, Counter] = defaultdict(Counter)

    graph = _load_citation_graph()
    raw_article_to_cases = graph.get("article_to_cases", {}) or {}
    raw_case_to_cases = graph.get("case_to_cases", {}) or {}

    dropped_article_keys = 0
    for raw_article, case_counts in raw_article_to_cases.items():
        article = _canonicalize_statute(raw_article, lower_map)
        if not article:
            dropped_article_keys += 1
            continue
        for raw_case, count in (case_counts or {}).items():
            case = _canonicalize_case(raw_case)
            if not case:
                continue
            edge_count = int(count)
            if edge_count <= 0:
                continue
            article_to_cases[article][case] += edge_count
            case_to_articles[case][article] += edge_count

    for raw_src, targets in raw_case_to_cases.items():
        src = _canonicalize_case(raw_src)
        if not src:
            continue
        for raw_tgt, count in (targets or {}).items():
            tgt = _canonicalize_case(raw_tgt)
            if not tgt or tgt == src:
                continue
            edge_count = int(count)
            if edge_count <= 0:
                continue
            case_to_cases[src][tgt] += edge_count

    stats = {
        "raw_article_keys_in_stage0b": len(raw_article_to_cases),
        "raw_case_keys_in_stage0b": len(raw_case_to_cases),
        "stage0b_article_keys_dropped": dropped_article_keys,
    }
    return article_to_cases, case_to_articles, case_to_cases, stats


def _build_case_root_helpers(case_maps: list[dict[str, Counter]]) -> tuple[dict[str, str], dict[str, list[str]], dict[str, str | None]]:
    case_leaf_to_root: dict[str, str] = {}
    root_to_case_leaves: dict[str, list[str]] = defaultdict(list)
    case_family: dict[str, str | None] = {}

    seen_cases = set()
    for mapping in case_maps:
        for src, targets in mapping.items():
            if not src.startswith("Art. "):
                seen_cases.add(src)
            for tgt in targets.keys():
                if not str(tgt).startswith("Art. "):
                    seen_cases.add(tgt)

    for case in sorted(seen_cases):
        parsed = _classify_case_citation(case)
        root = parsed["case_root"]
        case_leaf_to_root[case] = root
        case_family[case] = parsed["case_family_code"]
        root_to_case_leaves[root].append(case)

    root_to_case_leaves = {
        root: leaves for root, leaves in root_to_case_leaves.items()
    }
    return case_leaf_to_root, root_to_case_leaves, case_family


def build_all(
    limit_laws: int | None = None,
    laws_chunksize: int = LAWS_CHUNKSIZE,
) -> dict[str, Any]:
    t0 = time.time()
    lookup_data = _load_lookup_data()
    lower_map = lookup_data.get("citation_lower_map", {}) or {}

    print("Building statute-directed edges ...")
    article_to_articles, article_to_cases_direct_refs, statute_stats = _build_statute_reference_edges(
        lookup_data=lookup_data,
        limit_laws=limit_laws,
        chunksize=laws_chunksize,
    )

    print("Building case-directed edges ...")
    article_to_cases, case_to_articles, case_to_cases, case_stats = _build_case_reference_edges(
        lower_map=lower_map,
    )

    print("Building reverse views and case-root helpers ...")
    article_from_articles = _reverse_counter_map(article_to_articles)
    case_from_cases = _reverse_counter_map(case_to_cases)
    case_from_article_direct_refs = _reverse_counter_map(article_to_cases_direct_refs)
    case_leaf_to_root, root_to_case_leaves, case_family = _build_case_root_helpers(
        [article_to_cases, case_to_cases, case_from_article_direct_refs]
    )

    citation_type: dict[str, str] = {}
    for article_map in (article_to_articles, article_from_articles, article_to_cases, case_to_articles):
        for src, targets in article_map.items():
            if src.startswith("Art. "):
                citation_type[src] = "statute"
            for tgt in targets.keys():
                if tgt.startswith("Art. "):
                    citation_type[tgt] = "statute"

    for case_map in (article_to_cases, case_to_articles, case_to_cases, case_from_cases, article_to_cases_direct_refs):
        for src, targets in case_map.items():
            if not src.startswith("Art. "):
                citation_type[src] = "case"
            for tgt in targets.keys():
                if not tgt.startswith("Art. "):
                    citation_type[tgt] = "case"

    result = {
        "schema_version": SCHEMA_VERSION,
        "input_files": {
            "laws_de_csv": str(LAWS_DE_CSV),
            "kb_jsonl": str(KB_JSONL),
            "lookup_pkl": str(LOOKUP_PKL),
            "citation_graph_pkl": str(CITATION_GRAPH_PKL),
        },
        "article_to_articles": _serialise_counter_map(article_to_articles),
        "article_to_cases": _serialise_counter_map(article_to_cases),
        "case_to_articles": _serialise_counter_map(case_to_articles),
        "case_to_cases": _serialise_counter_map(case_to_cases),
        "article_from_articles": _serialise_counter_map(article_from_articles),
        "case_from_cases": _serialise_counter_map(case_from_cases),
        "article_to_cases_direct_refs": _serialise_counter_map(article_to_cases_direct_refs),
        "case_from_article_direct_refs": _serialise_counter_map(case_from_article_direct_refs),
        "case_leaf_to_root": case_leaf_to_root,
        "root_to_case_leaves": root_to_case_leaves,
        "case_family": case_family,
        "citation_type": citation_type,
        "build_stats": {
            "article_to_articles": _edge_stats(article_to_articles),
            "article_to_cases": _edge_stats(article_to_cases),
            "case_to_articles": _edge_stats(case_to_articles),
            "case_to_cases": _edge_stats(case_to_cases),
            "article_to_cases_direct_refs": _edge_stats(article_to_cases_direct_refs),
            "statute_scan": statute_stats,
            "stage0b_reuse": case_stats,
            "build_seconds": round(time.time() - t0, 2),
        },
    }

    REFERENCE_GRAPH_V2_PKL.parent.mkdir(parents=True, exist_ok=True)
    with open(REFERENCE_GRAPH_V2_PKL, "wb") as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"  article_to_articles edges: {result['build_stats']['article_to_articles']['unique_edges']:,}")
    print(f"  article_to_cases edges:    {result['build_stats']['article_to_cases']['unique_edges']:,}")
    print(f"  case_to_articles edges:    {result['build_stats']['case_to_articles']['unique_edges']:,}")
    print(f"  case_to_cases edges:       {result['build_stats']['case_to_cases']['unique_edges']:,}")
    print(f"Saved: {REFERENCE_GRAPH_V2_PKL}")
    return result


class ReferenceGraphV2:
    """Runtime helper for the typed reference graph v2 artifact."""

    def __init__(self, graph_pkl: Path = REFERENCE_GRAPH_V2_PKL):
        with open(graph_pkl, "rb") as f:
            data = pickle.load(f)
        self.article_to_articles = data.get("article_to_articles", {})
        self.article_to_cases = data.get("article_to_cases", {})
        self.case_to_articles = data.get("case_to_articles", {})
        self.case_to_cases = data.get("case_to_cases", {})
        self.article_from_articles = data.get("article_from_articles", {})
        self.case_from_cases = data.get("case_from_cases", {})
        self.article_to_cases_direct_refs = data.get("article_to_cases_direct_refs", {})
        self.case_from_article_direct_refs = data.get("case_from_article_direct_refs", {})
        self.case_leaf_to_root = data.get("case_leaf_to_root", {})
        self.root_to_case_leaves = data.get("root_to_case_leaves", {})
        self.case_family = data.get("case_family", {})
        self.citation_type = data.get("citation_type", {})
        self.build_stats = data.get("build_stats", {})

    @staticmethod
    def _top_n(edge_map: dict[str, int], top_n: int | None = None) -> list[tuple[str, int]]:
        items = sorted(edge_map.items(), key=lambda kv: (-int(kv[1]), kv[0]))
        return items if top_n is None else items[:top_n]

    def get_article_refs(self, citation: str, top_n: int | None = None) -> list[tuple[str, int]]:
        return self._top_n(self.article_to_articles.get(citation, {}), top_n=top_n)

    def get_cases_for_article(self, citation: str, top_n: int | None = None) -> list[tuple[str, int]]:
        return self._top_n(self.article_to_cases.get(citation, {}), top_n=top_n)

    def get_articles_for_case(self, citation: str, top_n: int | None = None) -> list[tuple[str, int]]:
        return self._top_n(self.case_to_articles.get(citation, {}), top_n=top_n)

    def get_case_refs(self, citation: str, top_n: int | None = None) -> list[tuple[str, int]]:
        return self._top_n(self.case_to_cases.get(citation, {}), top_n=top_n)


### `stage0f_build_gold_cocitation_prior.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage0f_build_gold_cocitation_prior.py"

"""
stage0f_build_gold_cocitation_prior.py

Build clean co-citation priors from gold citation sets.

Default runtime artifact is train-only and safe for development/evaluation.
An optional train+val artifact can also be built for competition-mode use.
"""

from __future__ import annotations

import argparse
import csv
import math
import pickle
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
from typing import Any

from data.data_paths import (
    GOLD_COCITATION_PRIOR_TRAIN_PKL,
    GOLD_COCITATION_PRIOR_TRAINVAL_PKL,
    TRAIN_CSV,
    VAL_CSV,
)


SCHEMA_VERSION = "gold-cocitation-prior-v1"


def _parse_gold(raw: str) -> list[str]:
    return [c.strip() for c in str(raw or "").split(";") if c and c.strip()]


def _citation_type(citation: str) -> str:
    return "law" if citation.startswith("Art. ") else "court"


def _pair_type(a: str, b: str) -> str:
    ta, tb = _citation_type(a), _citation_type(b)
    if ta == "law" and tb == "law":
        return "law-law"
    if ta == "court" and tb == "court":
        return "court-court"
    return "law-court"


def _safe_pmi(pair_support: int, count_a: int, count_b: int, total_queries: int) -> tuple[float | None, float | None]:
    if pair_support <= 0 or count_a <= 0 or count_b <= 0 or total_queries <= 0:
        return None, None
    p_ab = pair_support / total_queries
    p_a = count_a / total_queries
    p_b = count_b / total_queries
    if p_ab <= 0 or p_a <= 0 or p_b <= 0:
        return None, None
    pmi = math.log(p_ab / (p_a * p_b))
    denom = -math.log(p_ab)
    npmi = (pmi / denom) if denom > 0 else None
    return pmi, npmi


def build_prior(
    csv_paths: list[Path] | None = None,
    out_pkl: Path = GOLD_COCITATION_PRIOR_TRAIN_PKL,
    prior_name: str = "train",
) -> dict[str, Any]:
    csv_paths = csv_paths or [TRAIN_CSV]
    query_count = 0
    citation_query_count: Counter[str] = Counter()
    pair_support: Counter[tuple[str, str]] = Counter()

    input_files = []
    for csv_path in csv_paths:
        input_files.append(str(csv_path))
        with open(csv_path, encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                citations = sorted(set(_parse_gold(row.get("gold_citations", ""))))
                query_count += 1
                for citation in citations:
                    citation_query_count[citation] += 1
                for a, b in combinations(citations, 2):
                    pair_support[(a, b)] += 1

    neighbors: dict[str, list[dict[str, Any]]] = defaultdict(list)
    pair_stats: dict[tuple[str, str], dict[str, Any]] = {}

    for (a, b), support in pair_support.items():
        count_a = citation_query_count[a]
        count_b = citation_query_count[b]
        pmi, npmi = _safe_pmi(
            pair_support=support,
            count_a=count_a,
            count_b=count_b,
            total_queries=query_count,
        )
        jaccard = support / (count_a + count_b - support)
        pair_type = _pair_type(a, b)
        row = {
            "pair_type": pair_type,
            "support": int(support),
            "support_rate": support / query_count,
            "confidence_a_to_b": support / count_a if count_a else 0.0,
            "confidence_b_to_a": support / count_b if count_b else 0.0,
            "jaccard": jaccard,
            "pmi": pmi,
            "npmi": npmi,
        }
        pair_stats[(a, b)] = row

        neighbors[a].append(
            {
                "citation": b,
                "citation_type": _citation_type(b),
                "pair_type": pair_type,
                "support": int(support),
                "support_rate": support / query_count,
                "confidence": support / count_a if count_a else 0.0,
                "reverse_confidence": support / count_b if count_b else 0.0,
                "jaccard": jaccard,
                "pmi": pmi,
                "npmi": npmi,
            }
        )
        neighbors[b].append(
            {
                "citation": a,
                "citation_type": _citation_type(a),
                "pair_type": pair_type,
                "support": int(support),
                "support_rate": support / query_count,
                "confidence": support / count_b if count_b else 0.0,
                "reverse_confidence": support / count_a if count_a else 0.0,
                "jaccard": jaccard,
                "pmi": pmi,
                "npmi": npmi,
            }
        )

    for citation, items in neighbors.items():
        items.sort(
            key=lambda row: (
                -int(row["support"]),
                -(row["npmi"] if row["npmi"] is not None else -999.0),
                -float(row["confidence"]),
                row["citation"],
            )
        )

    pair_type_counts = Counter(_pair_type(a, b) for a, b in pair_support)

    result = {
        "schema_version": SCHEMA_VERSION,
        "prior_name": prior_name,
        "input_csvs": input_files,
        "query_count": query_count,
        "citation_query_count": dict(citation_query_count),
        "pair_count": len(pair_support),
        "pair_type_counts": dict(pair_type_counts),
        "neighbors": dict(neighbors),
        "pair_stats": {
            f"{a} || {b}": stats for (a, b), stats in pair_stats.items()
        },
        "suggested_thresholds": {
            "law-law": {"min_support": 3, "min_npmi": 0.45, "top_n": 3},
            "law-court": {"min_support": 2, "min_npmi": 0.70, "top_n": 1},
            "court-court": {"min_support": 3, "min_npmi": 0.90, "top_n": 1},
        },
    }

    out_pkl.parent.mkdir(parents=True, exist_ok=True)
    with open(out_pkl, "wb") as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"  Queries:    {query_count:,}")
    print(f"  Citations:  {len(citation_query_count):,}")
    print(f"  Pairs:      {len(pair_support):,}")
    print(f"  Pair types: {dict(pair_type_counts)}")
    print(f"  Prior:      {prior_name}")
    print(f"Saved: {out_pkl}")
    return result


class GoldCoCitationPrior:
    """Runtime helper around the train-only co-citation prior."""

    def __init__(self, prior_pkl: Path = GOLD_COCITATION_PRIOR_TRAIN_PKL):
        with open(prior_pkl, "rb") as f:
            data = pickle.load(f)
        self.prior_name = data.get("prior_name", "unknown")
        self.input_csvs = data.get("input_csvs", [])
        self.query_count = int(data.get("query_count", 0))
        self.citation_query_count = data.get("citation_query_count", {})
        self.pair_count = int(data.get("pair_count", 0))
        self.pair_type_counts = data.get("pair_type_counts", {})
        self.neighbors = data.get("neighbors", {})
        self.suggested_thresholds = data.get("suggested_thresholds", {})

    def get_companions(
        self,
        citation: str,
        pair_types: set[str] | None = None,
        min_support: int = 2,
        min_npmi: float = 0.0,
        top_n: int = 5,
    ) -> list[dict[str, Any]]:
        out = []
        for item in self.neighbors.get(citation, []):
            if pair_types and item["pair_type"] not in pair_types:
                continue
            if int(item["support"]) < min_support:
                continue
            npmi = item.get("npmi")
            if npmi is None or float(npmi) < min_npmi:
                continue
            out.append(item)
            if len(out) >= top_n:
                break
        return out


### Stage 0 runner

In [ ]:
# Run the full one-time Stage 0 build
run_stage0_all()

# Or run selected pieces instead:
# run_stage0_selected(lookup=True)
# run_stage0_selected(signals=True, refgraph=True)


## Stage 1 — Query Analysis

Corpus-grounded legal query understanding.

### `agent/prompts/query_analyzer_prompt.txt`

```text
You are a Swiss law examiner (Prüfungsexperte) analyzing a legal scenario.
Your task is to identify ALL relevant Swiss legal provisions and leading
cases (Leitentscheide) that a well-prepared candidate should cite.

Think step by step like a Swiss law professor:

1. CLASSIFY the legal domain(s): Civil law (ZGB/OR), Criminal law (StGB/StPO),
   Public law (BV/VwVG), Social insurance (ATSG/IVG/UVG/AVIG/BVG), International
   private law (IPRG), Debt collection (SchKG), Procedural (ZPO/StPO/BGG), etc.

2. IDENTIFY the specific legal issues (Rechtsfragen):
   - What are the main legal questions?
   - What sub-questions arise from each main question?
   - What procedural questions are relevant (standing, jurisdiction, appeal)?

3. For each issue, REASON about applicable provisions:
   - Which specific articles govern this issue?
   - Which Absatz (paragraph) is most directly relevant?
   - Are there related articles that modify, supplement, or define terms?
   - What are the leading BGE decisions (Leitentscheide) on this point?
   - Are there recent numbered decisions (e.g., 4A_xxx/20xx) on this?

4. Consider PROCEDURAL citations (often missed but always required):
   - Jurisdiction and competence articles (ZPO 1-12, StPO 1-15)
   - Procedural standing / Legitimation (BGG 76, 89, 115)
   - Appellate procedure provisions (BGG 72/78/82/113, StPO 393-428)
   - Cost and fee provisions (BGG 65-68, ZPO 106-107)
   - Time limits for appeal (BGG 100, ZPO 321)

5. Consider CROSS-REFERENCES:
   - Articles that reference other articles
   - General provisions applying alongside specific ones (e.g., OR 97 with specific liability)
   - Constitutional provisions underlying specific rules (BV 9/29/32 for fundamental rights)
   - ATSG general provisions alongside specific social insurance laws

6. Extract any EXPLICIT CITATIONS already in the query text:
   - If the query mentions "Art. 221 Abs. 1 lit. b StPO", cite it verbatim
   - These are guaranteed gold citations

7. Generate GERMAN SEARCH QUERIES: For each legal issue, create 3-5 focused
   German search queries using precise legal terminology that would appear in
   statutory articles and court decisions. These will be used for BM25 retrieval.

Output your analysis as JSON:
{
  "law_areas": ["Strafprozessrecht", "Strafrecht", ...],
  "explicit_citations": ["Art. 221 Abs. 1 StPO", ...],
  "legal_issues": [
    {
      "issue": "Conditions for pre-trial detention",
      "german_terms": ["Untersuchungshaft", "Kollusionsgefahr", "Fluchtgefahr"],
      "articles": ["Art. 221 Abs. 1 StPO", "Art. 222 StPO"],
      "cases": ["BGE 137 IV 122 E. 6.2"]
    }
  ],
  "procedural_articles": ["Art. 393 Abs. 1 StPO", "Art. 100 Abs. 1 BGG"],
  "candidate_articles": ["Art. 221 Abs. 1 StPO", "Art. 222 StPO", ...],
  "candidate_cases": ["BGE 137 IV 122 E. 6.2", ...],
  "search_queries_de": [
    "Verlängerung Untersuchungshaft Kollusionsgefahr StPO",
    "Haftentlassungsgesuch Verhältnismässigkeit",
    ...
  ],
  "search_queries_en": ["pre-trial detention collusion risk proportionality", ...]
}

```

### `stage1_query_analysis.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage1_query_analysis.py"

"""
stage1_query_analysis.py -- STAGE 1: Corpus-Grounded Legal Query Understanding

DESIGN PHILOSOPHY:
  A 4B model doesn't know Swiss law. But if you SHOW it evidence from the corpus,
  it becomes dramatically more accurate. This stage builds scaffolding around the
  LLM so it effectively acts as a human lawyer analyzing the query.

ARCHITECTURE (6-step pipeline, each step feeds the next):

  Step 1: EXPLICIT EXTRACTION (regex, instant)
    Extract any citation strings verbatim from the query text.
    These are free recall — always included regardless of everything else.

  Step 2: BM25 PROBE (CPU, <1s)
    Run the raw English query against BM25 to get initial signal.
    The top-10 results tell us WHICH law areas are relevant.
    This grounds all subsequent LLM reasoning in corpus evidence.

  Step 3: DOMAIN SCAFFOLDING (CPU, instant)
    From the BM25 probe results, extract:
      - Which law abbreviations appeared (StPO, ATSG, BGG, ...)
      - Their English names from the KB (Swiss Criminal Procedure Code, ...)
      - Their German keywords from the KB
    This creates a "cheat sheet" we inject into the LLM prompt.

  Step 4: LLM ANALYSIS (GPU, ~5-10s)
    With the domain scaffolding injected, the LLM:
      - Classifies legal domains (guided by BM25 evidence, not guessing)
      - Identifies legal issues with German Fachbegriffe
      - Suggests candidate citations it knows from training
      - Generates focused German search queries
    The LLM sees: "BM25 found articles from StPO (Criminal Procedure).
    The query mentions detention. What specific legal issues apply?"

  Step 5: HyDE — HYPOTHETICAL DOCUMENT EMBEDDING (GPU, ~5-10s)
    The LLM generates what a RELEVANT German legal article WOULD say.
    This is NOT a translation — it's a hypothetical document in the style
    of a Swiss statutory article. BM25 matches this against real articles
    far better than matching a short English query.
    
    Example: Query "Can pre-trial detention be extended?"
    HyDE output: "Die Untersuchungshaft kann vom Zwangsmassnahmengericht
    auf Antrag der Staatsanwaltschaft verlängert werden, wenn die
    Haftgründe nach Art. 221 weiterhin bestehen und die Verhältnismässigkeit
    gewahrt bleibt..."
    
    This 50-word German paragraph will match BM25 articles that a 5-word
    English query never could.

  Step 6: CROSS-LINGUAL TERM EXPANSION (CPU, instant)
    From the KB, expand English concepts to German equivalents:
      - "pre-trial detention" → "Untersuchungshaft" (from KB keyword mapping)
      - "disability insurance" → "Invalidenversicherung"
    These are exact corpus-grounded mappings, not LLM guesses.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python stage1_query_analysis.py                              # all val queries
#   python stage1_query_analysis.py --split test                 # test queries
#   python stage1_query_analysis.py --query "A claimant was..."  # single query debug
#   python stage1_query_analysis.py --backend anthropic          # use Claude API
#
# ─── OUTPUTS ────────────────────────────────────────────────────────────────
#   checkpoints/stage1_{split}.json
#   Per query: {
#     explicit_citations, law_areas, legal_issues, candidate_articles,
#     candidate_cases, search_queries_de, search_queries_en,
#     hyde_documents_de, bm25_probe_laws, domain_context,
#     procedural_articles, cross_lingual_terms
#   }
"""

import argparse
import json
import re
import sys
import time
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path(__file__).parent))

from data.data_paths import (
    TRAIN_CSV, VAL_CSV, TEST_CSV, CHECKPOINTS_DIR,
    STATUTORY_BM25_PKL, KB_JSONL,
)
from retrieval.explicit_citations import extract_explicit_citations

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# OFFLINE DOMAIN KNOWLEDGE (loaded once, shared across all queries)
# =============================================================================

_DOMAIN_KNOWLEDGE = None  # Lazy-loaded singleton


def _load_domain_knowledge() -> dict:
    """
    Build domain scaffolding from the corpus KB.
    This creates the "cheat sheet" that grounds LLM reasoning.

    Loads from laws_knowledge_base.jsonl:
      - law_taxonomy:       {abbrev: {name_en, name_de, sr_number}}
      - en_to_de_terms:     {"disability insurance": "Invalidenversicherung", ...}
      - law_area_groups:    {"Criminal": ["StGB", "StPO", ...], ...}
      - keyword_index:      {"Untersuchungshaft": ["Art. 221 StPO", ...]}
      - citation_to_text:   {citation_canon: "short article text snippet"}   ← NEW
      - valid_citations:    set of all citation_canon strings in corpus       ← NEW
    """
    global _DOMAIN_KNOWLEDGE
    if _DOMAIN_KNOWLEDGE is not None:
        return _DOMAIN_KNOWLEDGE

    print("  Loading domain knowledge from KB ...")

    law_taxonomy = {}
    en_to_de_terms = {}
    keyword_to_articles = {}
    law_to_en_name = {}
    law_to_de_name = {}
    citation_to_text = {}   # Fix 1: citation_canon -> article text snippet
    valid_citations = set() # Fix 3: all known citation_canon strings

    if KB_JSONL.exists():
        with open(KB_JSONL, encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except Exception:
                    continue

                canon = rec.get("citation_canon", "")
                if not canon:
                    continue

                valid_citations.add(canon)  # Fix 3

                law_info = rec.get("law", {})
                abbrev = law_info.get("law_abbreviation", "")
                name_en = law_info.get("law_name_en", "")
                # KB has no "law_name_de" — German name is in law_title_clean
                name_de = (law_info.get("law_name_de", "")
                           or law_info.get("law_title_clean", ""))
                sr = law_info.get("sr_number", "")

                if abbrev and abbrev not in law_taxonomy:
                    law_taxonomy[abbrev] = {
                        "name_en": name_en,
                        "name_de": name_de,
                        "sr_number": sr,
                    }

                if abbrev and name_en:
                    law_to_en_name[abbrev] = name_en
                if abbrev and name_de:
                    law_to_de_name[abbrev] = name_de

                # English→German term mapping from law names
                if name_en and name_de:
                    en_to_de_terms[name_en.lower()] = name_de

                # German keyword → article mapping
                kws = rec.get("keywords", {})
                for kw in (kws.get("provision_keywords_de") or []):
                    if isinstance(kw, str) and kw:
                        keyword_to_articles.setdefault(kw.lower(), []).append(canon)

                # Note: provision_keywords_en does not exist in the KB.
                # Cross-lingual terms come from law names + hardcoded dict below.

                # Fix 1: build article text snippets for BM25 probe injection
                text = (rec.get("search", {}).get("search_text_de", "")
                        or rec.get("content", {}).get("text_clean_de", "")
                        or "")
                if text:
                    citation_to_text[canon] = text[:250]

    LAW_AREA_MAP = {
        "Criminal law": ["StGB", "MStG", "BetmG", "SVG"],
        "Criminal procedure": ["StPO", "JStPO", "StBOG"],
        "Civil law": ["ZGB", "OR", "PrHG", "DSG"],
        "Civil procedure": ["ZPO"],
        "Public law": ["BV", "VwVG", "BGG", "RPG", "USG", "EnG"],
        "Social insurance": ["ATSG", "IVG", "UVG", "AVIG", "BVG", "AHV", "ELG", "KVG", "FamZG"],
        "Immigration": ["AIG", "AsylG"],
        "Debt enforcement": ["SchKG"],
        "Private international law": ["IPRG", "IRSG"],
        "Intellectual property": ["URG", "MSchG", "PatG"],
        "Federal court procedure": ["BGG", "BGerR"],
        "Tax law": ["LIFD", "LHID", "LTVA"],
    }
    law_to_areas = {}
    for area, abbrevs in LAW_AREA_MAP.items():
        for abbrev in abbrevs:
            law_to_areas.setdefault(abbrev, []).append(area)

    # Hardcoded EN→DE legal term dictionary for common query vocabulary.
    # The KB lacks provision_keywords_en, so this bridges the cross-lingual gap.
    _HARDCODED_EN_DE = {
        "pre-trial detention": "Untersuchungshaft",
        "pretrial detention": "Untersuchungshaft",
        "detention": "Haft",
        "collusion": "Kollusionsgefahr",
        "risk of flight": "Fluchtgefahr",
        "proportionality": "Verhältnismässigkeit",
        "disability insurance": "Invalidenversicherung",
        "invalidity": "Invalidität",
        "earning capacity": "Erwerbsfähigkeit",
        "incapacity for work": "Arbeitsunfähigkeit",
        "social insurance": "Sozialversicherung",
        "unemployment insurance": "Arbeitslosenversicherung",
        "accident insurance": "Unfallversicherung",
        "criminal law": "Strafrecht",
        "criminal procedure": "Strafprozessrecht",
        "civil procedure": "Zivilprozessrecht",
        "appeal": "Beschwerde",
        "cassation": "Kassation",
        "federal court": "Bundesgericht",
        "cantonal court": "Kantonsgericht",
        "statute of limitations": "Verjährung",
        "prescription": "Verjährung",
        "negligence": "Fahrlässigkeit",
        "intent": "Vorsatz",
        "self-defense": "Notwehr",
        "custody": "Sorgerecht",
        "visitation": "Besuchsrecht",
        "child welfare": "Kindeswohl",
        "child protection": "Kindesschutz",
        "divorce": "Scheidung",
        "maintenance": "Unterhalt",
        "alimony": "Unterhaltsbeitrag",
        "inheritance": "Erbrecht",
        "will": "Testament",
        "testamentary": "testamentarisch",
        "executor": "Willensvollstrecker",
        "heir": "Erbe",
        "contract": "Vertrag",
        "damages": "Schadenersatz",
        "liability": "Haftung",
        "tort": "Haftpflicht",
        "good faith": "Treu und Glauben",
        "abuse of rights": "Rechtsmissbrauch",
        "property": "Eigentum",
        "possession": "Besitz",
        "bona fide": "gutgläubig",
        "acquisition": "Erwerb",
        "mortgage": "Hypothek",
        "lease": "Miete",
        "tenancy": "Mietvertrag",
        "employment": "Arbeitsvertrag",
        "dismissal": "Kündigung",
        "termination": "Kündigung",
        "construction": "Werkvertrag",
        "mandate": "Auftrag",
        "asylum": "Asyl",
        "deportation": "Ausschaffung",
        "expulsion": "Landesverweisung",
        "residence permit": "Aufenthaltsbewilligung",
        "debt enforcement": "Schuldbetreibung",
        "bankruptcy": "Konkurs",
        "seizure": "Pfändung",
        "constitutional rights": "Grundrechte",
        "freedom of expression": "Meinungsfreiheit",
        "right to be heard": "rechtliches Gehör",
        "due process": "faires Verfahren",
        "recusal": "Ausstand",
        "legal aid": "unentgeltliche Rechtspflege",
        "power of attorney": "Vollmacht",
        "arbitration": "Schiedsgerichtsbarkeit",
        "recognition": "Anerkennung",
        "enforcement": "Vollstreckung",
        "acquittal": "Freispruch",
        "conviction": "Verurteilung",
        "sentencing": "Strafzumessung",
        "probation": "bedingte Strafe",
        "parole": "bedingte Entlassung",
        "confiscation": "Einziehung",
        "money laundering": "Geldwäscherei",
        "fraud": "Betrug",
        "theft": "Diebstahl",
        "robbery": "Raub",
        "embezzlement": "Veruntreuung",
        "forgery": "Urkundenfälschung",
        "sexual assault": "sexuelle Nötigung",
        "domestic violence": "häusliche Gewalt",
        "trafficking": "Menschenhandel",
        "tax evasion": "Steuerhinterziehung",
        "insolvency": "Zahlungsunfähigkeit",
        "surety": "Bürgschaft",
        "guarantee": "Garantie",
        "set-off": "Verrechnung",
        "assignment": "Zession",
        "unjust enrichment": "ungerechtfertigte Bereicherung",
        "servitude": "Dienstbarkeit",
        "easement": "Grunddienstbarkeit",
        "co-ownership": "Miteigentum",
        "condominium": "Stockwerkeigentum",
    }
    en_to_de_terms.update(_HARDCODED_EN_DE)

    # Common procedural articles (BGG) that appear in nearly every Federal Court case.
    # These are almost always cited but never in the candidate pool because they're
    # generic procedure, not substance. Inject them by default.
    PROCEDURAL_ARTICLES = [
        "Art. 42 BGG", "Art. 42 Abs. 1 BGG", "Art. 42 Abs. 2 BGG",
        "Art. 66 Abs. 1 BGG", "Art. 68 Abs. 2 BGG",
        "Art. 72 Abs. 1 BGG", "Art. 72 Abs. 2 BGG",
        "Art. 74 Abs. 1 BGG", "Art. 75 BGG", "Art. 75 Abs. 1 BGG",
        "Art. 76 Abs. 1 BGG",
        "Art. 82 BGG", "Art. 83 BGG",
        "Art. 90 BGG", "Art. 93 Abs. 1 BGG",
        "Art. 95 BGG", "Art. 97 Abs. 1 BGG",
        "Art. 100 Abs. 1 BGG", "Art. 105 Abs. 1 BGG", "Art. 106 Abs. 2 BGG",
        "Art. 107 Abs. 2 BGG", "Art. 113 BGG",
    ]

    _DOMAIN_KNOWLEDGE = {
        "law_taxonomy": law_taxonomy,
        "en_to_de_terms": en_to_de_terms,
        "keyword_to_articles": keyword_to_articles,
        "law_to_en_name": law_to_en_name,
        "law_to_de_name": law_to_de_name,
        "law_area_map": LAW_AREA_MAP,
        "law_to_areas": law_to_areas,
        "citation_to_text": citation_to_text,   # Fix 1
        "valid_citations": valid_citations,       # Fix 3
        "procedural_articles": PROCEDURAL_ARTICLES,
    }

    print(f"    Law abbreviations: {len(law_taxonomy):,}")
    print(f"    EN->DE term mappings: {len(en_to_de_terms):,}")
    print(f"    DE keywords indexed: {len(keyword_to_articles):,}")
    print(f"    Article texts indexed: {len(citation_to_text):,}")
    print(f"    Valid citations: {len(valid_citations):,}")
    return _DOMAIN_KNOWLEDGE


# =============================================================================
# STEP 2: BM25 PROBE — get corpus evidence before LLM runs
# =============================================================================

_SPARSE_RETRIEVER = None


def _get_sparse_retriever():
    """Lazy-load BM25 retriever (needed for probe + MAS stages)."""
    global _SPARSE_RETRIEVER
    if _SPARSE_RETRIEVER is None:
        if STATUTORY_BM25_PKL.exists():
            from retrieval.sparse_retriever import SparseRetriever
            _SPARSE_RETRIEVER = SparseRetriever()
        else:
            print("  WARNING: BM25 index not found. Probe step skipped.")
    return _SPARSE_RETRIEVER


def bm25_probe(query_text: str, top_k: int = 20) -> dict:
    """
    Quick BM25 search to get corpus-grounded signal BEFORE the LLM runs.
    
    Returns:
      - top_articles: [(citation, score), ...] top statutory BM25 hits
      - top_cases: [(citation, score), ...] top case-law BM25 hits
      - detected_laws: Counter of law abbreviations in top statutory results
      - detected_law_names: {abbrev: english_name} for detected laws
      - probe_context: formatted string to inject into LLM prompt
    """
    sparse = _get_sparse_retriever()
    if sparse is None:
        return {
            "top_articles": [],
            "top_cases": [],
            "detected_laws": Counter(),
            "detected_law_names": {},
            "probe_context": "",
            "top_citations": [],
            "top_case_citations": [],
            "top_with_text": [],
        }

    # Stage 1 should ground the LLM in statutory text, not let case-law
    # dominate the initial scaffold. We still keep a separate case probe for
    # downstream recall, but the LLM sees article evidence first.
    article_results = sparse.search_statutory([query_text], top_k=top_k)
    case_results = sparse.search_caselaw([query_text], top_k=min(top_k, 10))

    # Extract law abbreviations from top results
    # BM25 IDs use uppercase (STPO, STGB) but KB uses mixed-case (StPO, StGB).
    # Normalise via upper_to_abbrev so law name lookups work correctly.
    dk = _load_domain_knowledge()
    upper_to_abbrev = {ab.upper(): ab for ab in dk["law_taxonomy"]}

    detected_laws = Counter()
    for cite, _ in article_results:
        if cite.startswith("Art."):
            parts = cite.split()
            if parts:
                raw_abbrev = parts[-1]
                abbrev = upper_to_abbrev.get(raw_abbrev.upper(), raw_abbrev)
                detected_laws[abbrev] += 1

    # Map to English names
    detected_law_names = {}
    for abbrev in detected_laws:
        en_name = dk["law_to_en_name"].get(abbrev, "")
        if en_name:
            detected_law_names[abbrev] = en_name

    # Format context string for LLM injection
    probe_lines = []
    for abbrev, count in detected_laws.most_common(10):
        en_name = detected_law_names.get(abbrev, "unknown")
        probe_lines.append(f"  - {abbrev} ({en_name}): {count} statutory hits in top-{top_k}")

    probe_context = ""
    if probe_lines:
        probe_context = (
            "BM25 CORPUS EVIDENCE (top statutory articles matching your query):\n"
            + "\n".join(probe_lines)
            + "\n\nThese law areas are likely relevant. Use this as a starting point."
        )

    # Fix 1: enrich top results with article text for LLM injection
    citation_to_text = dk["citation_to_text"]
    top_with_text = []
    for i, (cite, _) in enumerate(article_results[:10]):
        text = citation_to_text.get(cite, "")
        # strip the law title prefix (everything up to " | Art.")
        if " | Art." in text:
            text = text[text.index(" | Art.") + 3:]
        elif " | BGE" in text:
            text = text[text.index(" | BGE") + 3:]
        top_with_text.append({
            "idx": i,
            "citation": cite,
            "text": text[:200].strip() if text else "",
        })

    top_citations = [cite for cite, _ in article_results[:10]]
    top_case_citations = [cite for cite, _ in case_results[:10]]

    return {
        "top_articles": article_results[:top_k],
        "top_cases": case_results[:min(top_k, 10)],
        "detected_laws": dict(detected_laws),
        "detected_law_names": detected_law_names,
        "probe_context": probe_context,
        "top_citations": top_citations,
        "top_case_citations": top_case_citations,
        "top_with_text": top_with_text,   # Fix 1
    }


# =============================================================================
# STEP 3: DOMAIN SCAFFOLDING — build a "cheat sheet" for the LLM
# =============================================================================

def build_domain_context(detected_laws: dict, query_text: str,
                         top_with_text: list | None = None) -> str:
    """
    Build a domain context string injected into the LLM prompt.

    Fix 1: now includes actual article text snippets from BM25 probe so the
    LLM reads corpus evidence rather than recalling from training memory.
    """
    dk = _load_domain_knowledge()
    sections = []

    # 1. Fix 1: BM25 probe results WITH article text — the core grounding signal.
    #    Each entry is numbered so the LLM can reference them by index.
    if top_with_text:
        probe_lines = []
        for entry in top_with_text:
            citation = entry["citation"]
            text = entry["text"]
            snippet = f"  [{entry['idx']}] {citation}"
            if text:
                snippet += f"\n      \"{text}\""
            probe_lines.append(snippet)
        sections.append(
            "BM25 CORPUS EVIDENCE — top matching articles (THESE EXIST IN THE CORPUS):\n"
            + "\n".join(probe_lines)
            + "\n\nThese are real articles. Use their terminology in your German search queries."
        )

    # 2. Detected law areas with full names
    if detected_laws:
        law_lines = []
        for abbrev, count in sorted(detected_laws.items(), key=lambda x: -x[1])[:15]:
            en = dk["law_to_en_name"].get(abbrev, "")
            de = dk["law_to_de_name"].get(abbrev, "")
            law_lines.append(f"  {abbrev} ({count} hits): {en} / {de}")
        sections.append("DETECTED LAW AREAS:\n" + "\n".join(law_lines))

    # 3. Cross-lingual term suggestions
    query_lower = query_text.lower()
    matched_terms = []
    for en_term, de_term in dk["en_to_de_terms"].items():
        if len(en_term) > 4 and en_term in query_lower:
            matched_terms.append((en_term, de_term))
    if matched_terms:
        term_lines = [f"  \"{en}\" → \"{de}\"" for en, de in matched_terms[:10]]
        sections.append(
            "CROSS-LINGUAL TERM MAPPINGS (English → German Fachbegriff):\n"
            + "\n".join(term_lines)
        )

    # 4. Swiss law area taxonomy (always included as reference)
    tax_lines = []
    for area, abbrevs in dk["law_area_map"].items():
        tax_lines.append(f"  {area}: {', '.join(abbrevs)}")
    sections.append("SWISS LAW AREA TAXONOMY:\n" + "\n".join(tax_lines))

    return "\n\n".join(sections)


def _dedupe_keep_order(items) -> list:
    seen = set()
    out = []
    for item in items:
        if not item or item in seen:
            continue
        seen.add(item)
        out.append(item)
    return out


def _extract_law_abbrev(citation: str) -> str:
    if not isinstance(citation, str) or not citation.startswith("Art."):
        return ""
    parts = citation.rsplit(" ", 1)
    if len(parts) != 2:
        return ""
    dk = _load_domain_knowledge()
    upper_to_abbrev = {ab.upper(): ab for ab in dk["law_taxonomy"]}
    raw = parts[1].strip()
    return upper_to_abbrev.get(raw.upper(), raw)


def _derive_law_hints(detected_laws: dict, explicit_citations: list[str] | set[str]) -> list[str]:
    hints = list(detected_laws.keys())
    for citation in explicit_citations:
        abbrev = _extract_law_abbrev(citation)
        if abbrev:
            hints.append(abbrev)
    return _dedupe_keep_order(hints)


def _build_crosslingual_queries(cross_lingual_terms: dict, law_hints: list[str]) -> list[str]:
    de_terms = _dedupe_keep_order(cross_lingual_terms.values())
    queries = []

    for de_term in de_terms[:5]:
        for law_hint in law_hints[:2]:
            queries.append(f"{de_term} {law_hint}")
        queries.append(de_term)

    for i in range(min(len(de_terms), 4)):
        for j in range(i + 1, min(len(de_terms), 4)):
            queries.append(f"{de_terms[i]} {de_terms[j]}")

    return _dedupe_keep_order(q.strip() for q in queries if isinstance(q, str) and len(q.strip()) > 3)[:8]


def _normalize_law_areas(
    raw_areas: list,
    law_hints: list[str],
    cross_lingual_terms: dict,
) -> list[str]:
    dk = _load_domain_knowledge()
    law_to_areas = dk.get("law_to_areas", {})

    alias_to_area = {
        "strafrecht": "Criminal law",
        "betrug": "Criminal law",
        "strafprozessrecht": "Criminal procedure",
        "strafverfahren": "Criminal procedure",
        "zivilrecht": "Civil law",
        "familienrecht": "Civil law",
        "erbrecht": "Civil law",
        "bürgerliches recht": "Civil law",
        "privatrecht": "Civil law",
        "privatrechtsverhältnisse": "Civil law",
        "eigentumrecht": "Civil law",
        "schuldrecht": "Civil law",
        "arbeitsrecht": "Civil law",
        "gesundheitsrecht": "Civil law",
        "handelsrecht": "Civil law",
        "wirtschaftsrecht": "Civil law",
        "produkthaftung": "Civil law",
        "konsumkreditrecht": "Civil law",
        "baulaw": "Civil law",
        "schadensrecht": "Civil law",
        "zivilprozessrecht": "Civil procedure",
        "handelsprozessrecht": "Civil procedure",
        "verwaltungsrecht": "Public law",
        "bundesrecht": "Public law",
        "recht auf gehör": "Public law",
        "verhältnismässigkeit": "Public law",
        "sozialversicherung": "Social insurance",
        "sozialversicherungsrecht": "Social insurance",
        "sozialversicherungrecht": "Social insurance",
        "wohlfahrtsrecht": "Social insurance",
        "immigrationrecht": "Immigration",
        "schuldbetreibung": "Debt enforcement",
        "schuldbeklagung": "Debt enforcement",
        "konkursrecht": "Debt enforcement",
        "internationales privatrecht": "Private international law",
        "privatinternationales recht": "Private international law",
        "privat- und internationales recht": "Private international law",
        "markenrecht": "Intellectual property",
        "wettbewerbsrecht": "Intellectual property",
        "schutz vor unlauterem wettbewerb": "Intellectual property",
        "bundesgerichtsrecht": "Federal court procedure",
        "bgg": "Federal court procedure",
    }

    normalized = []
    for raw_area in raw_areas or []:
        if not isinstance(raw_area, str):
            continue
        area = raw_area.strip()
        if not area:
            continue
        if area in dk["law_area_map"]:
            normalized.append(area)
            continue

        mapped = alias_to_area.get(area.lower())
        if mapped:
            normalized.append(mapped)
            continue

        normalized.extend(law_to_areas.get(area, []))
        normalized.extend(law_to_areas.get(area.upper(), []))

    if not normalized:
        for law_hint in law_hints:
            normalized.extend(law_to_areas.get(law_hint, []))

    if not normalized and cross_lingual_terms:
        term_to_area = [
            (("pre-trial detention", "pretrial detention", "detention", "collusion",
              "risk of flight", "self-defense", "acquittal", "conviction",
              "sentencing", "probation", "parole", "confiscation",
              "money laundering", "fraud", "theft", "robbery",
              "embezzlement", "forgery", "sexual assault",
              "domestic violence", "trafficking", "tax evasion"), "Criminal law"),
            (("appeal", "constitutional rights", "right to be heard",
              "due process", "recusal", "legal aid"), "Public law"),
            (("social insurance", "disability insurance", "invalidity",
              "earning capacity", "incapacity for work",
              "unemployment insurance", "accident insurance"), "Social insurance"),
            (("asylum", "deportation", "expulsion", "residence permit"), "Immigration"),
            (("debt enforcement", "bankruptcy", "seizure"), "Debt enforcement"),
            (("arbitration", "recognition"), "Private international law"),
            (("trademark", "unfair competition"), "Intellectual property"),
            (("maintenance", "alimony", "custody", "visitation", "child welfare",
              "child protection", "divorce", "inheritance", "will", "executor",
              "heir", "contract", "damages", "liability", "tort", "good faith",
              "abuse of rights", "property", "possession", "acquisition",
              "mortgage", "lease", "tenancy", "employment", "dismissal",
              "termination", "construction", "mandate", "power of attorney",
              "guarantee", "set-off", "assignment", "unjust enrichment",
              "servitude", "easement", "co-ownership", "condominium"), "Civil law"),
        ]
        for en_term in cross_lingual_terms:
            low = en_term.lower()
            for keywords, area in term_to_area:
                if low in keywords:
                    normalized.append(area)

    return _dedupe_keep_order(normalized)[:4]


def _build_fallback_hyde(de_terms: list[str], law_hints: list[str]) -> list[str]:
    if not de_terms:
        return []
    law_hint = law_hints[0] if law_hints else "dem einschlägigen Schweizer Recht"
    term_phrase = ", ".join(de_terms[:3])
    return [
        "Die einschlägigen Bestimmungen zu "
        f"{term_phrase} regeln die Anspruchsvoraussetzungen, mögliche Einwendungen "
        f"und die gerichtliche Durchsetzung nach {law_hint}. Massgeblich sind "
        "insbesondere die materiellen Voraussetzungen, die Beweisführung und die "
        "Verhältnismässigkeit der beantragten Rechtsfolge."
    ]


# =============================================================================
# STEP 4 + 5: LLM ANALYSIS + HyDE GENERATION (single LLM call)
# =============================================================================

# Fix 2: tightened system prompt — LLM is a READER of corpus evidence, not a knowledge source.
# No candidate_articles or candidate_cases in the schema (BM25 probe provides those).
# LLM only selects relevant probe indices + generates German terminology + HyDE text.
_SYSTEM_PROMPT_TEMPLATE = """You are a Swiss law classifier with access to corpus search results.
Your job is to READ the BM25 evidence below and help retrieve more relevant articles.

{domain_context}

RULES — read carefully:
1. DO NOT invent article numbers. The corpus evidence above shows REAL articles.
2. DO NOT cite BGE cases from memory — they are unreliable. Only extract cases verbatim from the query.
3. Your value is: classifying the legal domain, extracting German Fachbegriffe, and generating BM25 search queries.

TASK: Analyze the legal scenario and output JSON with exactly these fields:

STEP 1 — CLASSIFY: Which law areas apply? Choose only from "DETECTED LAW AREAS" and "SWISS LAW AREA TAXONOMY" shown above.
Output the English taxonomy labels exactly as written in "SWISS LAW AREA TAXONOMY".

STEP 2 — IDENTIFY ISSUES: For each distinct legal issue (Rechtsfrage), list:
  - A short issue description (English)
  - The German legal terms (Fachbegriffe) that appear in Swiss statutory text for this issue.
    Base these on the article texts shown in "BM25 CORPUS EVIDENCE" above.

STEP 3 — SELECT RELEVANT PROBE RESULTS: From the numbered BM25 results [0]..[9] above,
list the indices of results relevant to the query. Example: [0, 2, 4].

STEP 4 — GERMAN SEARCH QUERIES: Generate 5 focused German BM25 keyword queries.
Use EXACT German legal terms from the article texts shown above. These queries will
be used to retrieve more articles from the same corpus.

STEP 5 — HYPOTHETICAL GERMAN ARTICLE (HyDE): Write 1-2 short paragraphs in German
describing what a relevant Swiss statutory article would say about this scenario.
Use only terminology consistent with the law areas identified in STEP 1.
Stay strictly within the identified legal domain — do not blend in unrelated areas.

STEP 6 — EXTRACT any citations written verbatim in the query text (exact string match only).

Output ONLY valid JSON, no other text:
{{
  "law_areas": ["Civil law", "Civil procedure"],
  "legal_issues": [
    {{
      "issue": "Restriction of visitation rights",
      "german_terms": ["Besuchsrecht", "persönlicher Verkehr", "Kindesinteresse", "Kindeswohl"]
    }}
  ],
  "relevant_probe_indices": [0, 1, 3],
  "search_queries_de": [
    "Besuchsrecht Einschränkung Kindesinteresse ZGB",
    "persönlicher Verkehr Kind Elternteil Gefährdung",
    "Kindesschutz Besuchsrecht Einschränkung Gericht"
  ],
  "search_queries_en": ["restriction of visitation rights child welfare ZGB"],
  "hyde_documents_de": [
    "Das Gericht kann den persönlichen Verkehr zwischen dem Kind und dem nicht sorgeberechtigten Elternteil einschränken oder untersagen, wenn das Wohl des Kindes durch den Kontakt gefährdet wird. Voraussetzung ist, dass konkrete Anhaltspunkte für eine Gefährdung bestehen."
  ],
  "explicit_citations": []
}}"""


def analyze_single_query(
    query_text: str,
    backend: str = "local",
    model: str | None = None,
) -> dict:
    """
    Run the full 6-step query understanding pipeline.
    """
    t0 = time.time()
    results = {}

    # ── Step 1: Explicit citation extraction (regex) ─────────────────────
    explicit = extract_explicit_citations(query_text)
    results["explicit_citations_regex"] = sorted(explicit)

    # ── Step 2: BM25 probe — corpus evidence ─────────────────────────────
    probe = bm25_probe(query_text, top_k=20)
    results["bm25_probe"] = {
        "detected_laws": probe["detected_laws"],
        "detected_law_names": probe["detected_law_names"],
        "top_citations": probe.get("top_citations", []),
        "top_case_citations": probe.get("top_case_citations", []),
    }

    # ── Step 3: Domain scaffolding (Fix 1: inject article texts) ─────────
    top_with_text = probe.get("top_with_text", [])
    domain_context = build_domain_context(
        probe["detected_laws"], query_text, top_with_text=top_with_text
    )
    results["domain_context"] = domain_context

    # ── Step 4 + 5: LLM analysis + HyDE (single call) ───────────────────
    # Fix 2: new prompt schema — no candidate_articles/candidate_cases
    # Fix 4: enable_thinking=False, max_new_tokens=2048 (LLM classifies, not reasons)
    system_prompt = _SYSTEM_PROMPT_TEMPLATE.format(domain_context=domain_context)

    if backend == "local":
        from agent.llm_backend import generate_json
        analysis = generate_json(
            system_prompt=system_prompt,
            user_prompt=query_text,
            max_new_tokens=2048,
            enable_thinking=False,
        )
    elif backend == "anthropic":
        from agent.llm_backend import call_anthropic, parse_json_response
        raw = call_anthropic(system_prompt, query_text,
                             model=model or "claude-sonnet-4-6")
        analysis = parse_json_response(raw)
    elif backend == "openai":
        from agent.llm_backend import call_openai, parse_json_response
        raw = call_openai(system_prompt, query_text, model=model or "gpt-4o")
        analysis = parse_json_response(raw)
    else:
        raise ValueError(f"Unknown backend: {backend}")

    # ── LLM FAILURE FALLBACK ─────────────────────────────────────────────
    # If LLM returned empty/broken JSON (val_010 case), build minimal analysis
    # from BM25 probe + cross-lingual terms so downstream stages aren't starved.
    _llm_failed = (
        not analysis.get("law_areas")
        and not analysis.get("search_queries_de")
        and not analysis.get("hyde_documents_de")
    )
    if _llm_failed:
        print("  WARNING: LLM returned empty analysis — using BM25 probe fallback")
        dk_fallback = _load_domain_knowledge()
        # Derive fallback signals from BM25 + explicit query citations.
        query_lower_fb = query_text.lower()
        fb_cross_lingual_terms = {}
        for en_t, de_t in dk_fallback["en_to_de_terms"].items():
            if len(en_t) > 4 and en_t in query_lower_fb:
                fb_cross_lingual_terms[en_t] = de_t
        fb_terms = _dedupe_keep_order(fb_cross_lingual_terms.values())
        law_hints = _derive_law_hints(probe["detected_laws"], explicit)
        analysis["law_areas"] = _normalize_law_areas([], law_hints, fb_cross_lingual_terms)

        fb_queries = _build_crosslingual_queries(fb_cross_lingual_terms, law_hints)
        # Also add detected law keywords from probe top texts
        for entry in top_with_text[:5]:
            text = entry.get("text", "")
            if text:
                # Use first 40 chars as a keyword query
                fb_queries.append(text[:40].strip())
        analysis["search_queries_de"] = _dedupe_keep_order(fb_queries)[:8]
        analysis["search_queries_en"] = [query_text[:100]]
        analysis["legal_issues"] = [{"issue": "See BM25 probe", "german_terms": fb_terms[:5]}]
        analysis["hyde_documents_de"] = _build_fallback_hyde(fb_terms, law_hints)

    # ── Step 6: Cross-lingual term expansion ─────────────────────────────
    dk = _load_domain_knowledge()
    cross_lingual_terms = {}
    query_lower = query_text.lower()
    for en_term, de_term in dk["en_to_de_terms"].items():
        if len(en_term) > 4 and en_term in query_lower:
            cross_lingual_terms[en_term] = de_term
    results["cross_lingual_terms"] = cross_lingual_terms

    # ── Merge explicit citations (regex + LLM) ────────────────────────────
    # Only accept LLM-added citations that actually appear verbatim in the query
    llm_explicit = {
        c for c in analysis.get("explicit_citations", [])
        if c in query_text
    }
    results["explicit_citations"] = sorted(explicit | llm_explicit)
    law_hints = _derive_law_hints(probe["detected_laws"], results["explicit_citations"])

    # ── Fix 2: build candidate_articles from probe selection + probe top ──
    # LLM selects relevant probe indices; we map back to citation strings.
    relevant_indices = analysis.get("relevant_probe_indices", [])
    selected_citations = []
    for idx in relevant_indices:
        if isinstance(idx, int) and 0 <= idx < len(top_with_text):
            selected_citations.append(top_with_text[idx]["citation"])

    # candidate_articles = LLM-selected statutory probe results + statutory top-10
    probe_top_articles = [c for c in probe.get("top_citations", []) if c.startswith("Art.")]
    probe_top_cases = [c for c in probe.get("top_case_citations", []) if not c.startswith("Art.")]
    candidate_articles = list(dict.fromkeys(selected_citations + probe_top_articles))

    # Fix 3: post-validation — strip any LLM-hallucinated citations not in corpus
    valid_citations = dk["valid_citations"]
    # Only validate law citations (Art. ...) — court citations not in KB
    def _is_valid(c: str) -> bool:
        if c.startswith("Art."):
            return c in valid_citations
        return True  # court citations pass through

    candidate_articles = [c for c in candidate_articles if c.startswith("Art.") and _is_valid(c)]
    results["candidate_articles"] = candidate_articles

    # candidate_cases come from the separate case-law BM25 probe.
    results["candidate_cases"] = probe_top_cases

    # Standard fields from LLM analysis
    for k, default in [
        ("law_areas", []),
        ("legal_issues", []),
        ("search_queries_de", []),
        ("search_queries_en", []),
        ("hyde_documents_de", []),
    ]:
        results[k] = analysis.get(k, default)

    # Fix 3: strip hallucinated law citations from explicit_citations too.
    # Regex-derived citations come from literal query references and may use
    # alternate abbreviations (e.g. CO -> OR), so we keep them here and let the
    # verifier normalize or drop them later if needed.
    # Only validate the LLM-added ones.
    results["explicit_citations"] = sorted(
        explicit | {c for c in llm_explicit if _is_valid(c)}
    )
    law_hints = _derive_law_hints(probe["detected_laws"], results["explicit_citations"])
    results["law_areas"] = _normalize_law_areas(
        results.get("law_areas", []),
        law_hints,
        cross_lingual_terms,
    )

    # BM25 probe top for downstream stages
    results["bm25_probe_top"] = probe_top_articles
    results["bm25_probe_cases"] = probe_top_cases

    # ── Procedural articles: inject common BGG articles ────────────────
    # Nearly every Federal Court case cites these. They appear in ~60% of
    # gold citations but BM25 never retrieves them (too generic).
    dk_proc = _load_domain_knowledge()
    # Filter procedural articles to those that exist in the corpus
    proc_valid = dk_proc.get("valid_citations", set())
    procedural = [a for a in dk_proc.get("procedural_articles", [])
                  if a in proc_valid]
    results["procedural_articles"] = procedural

    # ── Normalize BM25 candidate casing (STPO→StPO) ───────────────────
    upper_to_abbrev = {ab.upper(): ab for ab in dk_proc["law_taxonomy"]}
    def _norm_casing(cite: str) -> str:
        if cite.startswith("Art."):
            parts = cite.rsplit(" ", 1)
            if len(parts) == 2:
                canon_ab = upper_to_abbrev.get(parts[1].upper(), parts[1])
                return f"{parts[0]} {canon_ab}"
        return cite
    results["candidate_articles"] = [_norm_casing(c) for c in results["candidate_articles"]]
    results["candidate_cases"] = [_norm_casing(c) for c in results.get("candidate_cases", [])]

    # Generate additional German queries from cross-lingual terms
    results["search_queries_de_crosslingual"] = _build_crosslingual_queries(
        cross_lingual_terms,
        law_hints,
    )

    # Combine all German search queries for stage 2
    all_de_queries = _dedupe_keep_order(
        results["search_queries_de"]
        + results.get("hyde_documents_de", [])
        + results["search_queries_de_crosslingual"]
    )
    results["all_search_queries_de"] = all_de_queries

    elapsed = time.time() - t0
    results["analysis_time_s"] = round(elapsed, 1)

    return results


# =============================================================================
# BATCH RUNNER
# =============================================================================

def run_batch(split: str, backend: str = "local", model: str | None = None,
              resume: bool = True) -> dict:
    """Run Stage 1 on all queries in a split with incremental saving."""
    import pandas as pd

    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    df = pd.read_csv(csv_map[split])

    out_path = CHECKPOINTS_DIR / f"stage1_{split}.json"

    # Pre-load domain knowledge and BM25
    _load_domain_knowledge()
    _get_sparse_retriever()

    # Resume from checkpoint (re-run entries that had errors)
    results = {}
    if resume and out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            results = json.load(f)
        # Remove entries with errors so they get re-run
        failed = [qid for qid, r in results.items() if "error" in r]
        for qid in failed:
            del results[qid]
        if failed:
            print(f"  Resuming: {len(results)} done, {len(failed)} failed entries will be re-run: {failed}")
        else:
            print(f"  Resuming: {len(results)} queries already done")

    for i, (_, row) in enumerate(df.iterrows()):
        qid = row["query_id"]
        if qid in results:
            continue

        query = row["query"]
        print(f"\n[{i+1}/{len(df)}] {qid}")
        print(f"  Query: {query[:100]}...")

        # Clear VRAM before each query to prevent fragmentation buildup
        if backend == "local":
            try:
                import torch
                torch.cuda.empty_cache()
            except Exception:
                pass

        t0 = time.time()
        try:
            analysis = analyze_single_query(query, backend=backend, model=model)
        except Exception as e:
            print(f"  ERROR: {e}")
            import traceback
            traceback.print_exc()
            # Recover CUDA context and retry once with gc
            try:
                import gc
                import torch
                gc.collect()
                torch.cuda.empty_cache()
                print("  Retrying after CUDA cleanup...")
                analysis = analyze_single_query(query, backend=backend, model=model)
            except Exception as e2:
                print(f"  RETRY FAILED: {e2}")
                try:
                    import gc
                    import torch
                    gc.collect()
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                analysis = {"error": str(e), "explicit_citations": list(extract_explicit_citations(query))}

        elapsed = time.time() - t0

        # Print diagnostics
        print(f"  Done in {elapsed:.1f}s")
        print(f"  BM25 probe laws: {analysis.get('bm25_probe', {}).get('detected_laws', {})}")
        print(f"  Explicit citations: {analysis.get('explicit_citations', [])}")
        print(f"  Law areas: {analysis.get('law_areas', [])}")
        print(f"  DE queries: {analysis.get('search_queries_de', [])[:3]}")
        print(f"  HyDE docs: {len(analysis.get('hyde_documents_de', []))} generated")
        print(f"  XL terms: {analysis.get('cross_lingual_terms', {})}")
        print(f"  Candidates: {len(analysis.get('candidate_articles', []))} art + "
              f"{len(analysis.get('candidate_cases', []))} cases")

        results[qid] = analysis

        # Save incrementally
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\nStage 1 complete. Saved: {out_path}")
    return results


# =============================================================================
# CLI
# =============================================================================


### Stage 1 runner

In [ ]:
stage1_outputs = run_stage1(split=SPLIT, backend=BACKEND)

## Stage 2 — Multi-Agent Sparse Retrieval

MAS retrieval over the indexed corpus.

### `agent/prompts/mas_prompts.py`

```python
"""
mas_prompts.py -- System prompts for the Multi-Agent Sparse retrieval system.

Adapted from LegalMALR Table 2 for Swiss law with BM25 retrieval.
All agents use the SAME Qwen3-4B model — only the system prompt differs.

Each agent outputs German search queries that will be fed to BM25.
"""

# ─── PLANNER AGENT ──────────────────────────────────────────────────────────
# Decides which agent to invoke next, or whether to terminate.
PLANNER_PROMPT = """You are a Swiss law retrieval planner. You coordinate a team of
specialist agents to find ALL relevant legal provisions for a given scenario.

You have already retrieved some candidate provisions. Your job is to decide
what to do next to improve recall (find more relevant provisions).

Available agents:
1. REWRITE — rewrites colloquial English into precise German legal terminology
2. SUPPLEMENT — makes implicit legal conditions explicit (thresholds, standing, procedural)
3. DECOMPOSE — splits complex multi-issue queries into focused sub-queries
4. SUPPORTIVE — generates queries for procedural, interpretive, and auxiliary provisions
5. CROSSREF — generates queries for constitutional and cross-referenced articles

Based on the current state, decide:
- Which agent would most likely find NEW relevant provisions not yet in the pool?
- Or should we TERMINATE because further searching is unlikely to help?

Output JSON only:
{
  "reasoning": "Brief explanation of what's missing and why this agent helps",
  "action": "REWRITE" | "SUPPLEMENT" | "DECOMPOSE" | "SUPPORTIVE" | "CROSSREF" | "TERMINATE"
}
"""

# ─── SINGLE-ELEMENT REWRITE AGENT ──────────────────────────────────────────
# Rewrites colloquial EN terms into precise DE legal terminology for BM25.
REWRITE_PROMPT = """You are a Swiss law terminology expert. Given an English legal scenario,
generate precise German search queries using the exact legal terms that would appear
in Swiss statutory articles and court decisions.

CRITICAL: Your queries will be used for BM25 keyword search, so use the EXACT German
legal terms (Fachbegriffe) that appear in the law text, not paraphrases.

Examples of good translations:
- "pre-trial detention" → "Untersuchungshaft"
- "collusion risk" → "Kollusionsgefahr"
- "disability insurance" → "Invalidenversicherung"
- "medical expert opinion" → "medizinisches Gutachten"
- "right to be heard" → "rechtliches Gehör"
- "proportionality" → "Verhältnismässigkeit"

Generate 3-5 focused German search queries. Each query should:
- Target a SPECIFIC legal concept from the scenario
- Use 2-4 precise German legal terms
- Include relevant law abbreviations (StPO, ATSG, BGG, etc.)

Output JSON:
{
  "queries": ["Untersuchungshaft Kollusionsgefahr StPO", "Haftprüfung Verhältnismässigkeit", ...]
}
"""

# ─── SUPPLEMENTARY-ELEMENT AGENT ───────────────────────────────────────────
# Makes implicit legal conditions explicit.
SUPPLEMENT_PROMPT = """You are a Swiss law expert who identifies IMPLICIT legal requirements.

Given a legal scenario, identify conditions, thresholds, and requirements that are
implied but not explicitly stated. For each, generate a German search query.

Think about:
- Standing requirements (Legitimation, Beschwerdebefugnis)
- Jurisdictional prerequisites (örtliche/sachliche Zuständigkeit)
- Time limits and deadlines (Fristen, Verwirkung)
- Burden of proof rules (Beweislast)
- Exhaustion of remedies requirements
- Threshold amounts or severity levels
- Formal requirements (Schriftlichkeit, Begründungspflicht)

Generate 2-4 German search queries targeting these implicit requirements.

Output JSON:
{
  "implicit_issues": [
    {"issue": "...", "query": "Beschwerdelegitimation BGG Strafverfahren"}
  ]
}
"""

# ─── MULTI-ELEMENT DECOMPOSITION AGENT ─────────────────────────────────────
# Splits complex queries into focused sub-queries.
DECOMPOSE_PROMPT = """You are a Swiss law analyst who breaks down complex legal scenarios
into individual legal questions (Rechtsfragen).

A typical exam scenario may involve 3-8 distinct legal issues, each requiring
different statutory articles and case law. Your job is to identify each issue
and create a focused German search query for it.

For each sub-issue, generate ONE precise German search query targeting:
- The specific legal provision(s) that govern this issue
- The relevant law abbreviation

Example decomposition:
  Scenario about wrongful arrest + appeal + costs
  → "Untersuchungshaft Voraussetzungen Art 221 StPO"
  → "Beschwerde gegen Haftanordnung Art 222 StPO"
  → "Kostenregelung Strafverfahren Art 428 StPO"
  → "Beschwerdefrist BGG Strafverfahren"

Generate 3-6 sub-queries covering ALL distinct legal issues.

Output JSON:
{
  "sub_issues": [
    {"issue": "...", "query": "..."}
  ]
}
"""

# ─── SUPPORTIVE-LAW AGENT ─────────────────────────────────────────────────
# Targets procedural, interpretive, and auxiliary provisions.
SUPPORTIVE_PROMPT = """You are a Swiss procedural law specialist. Given a legal scenario
and the substantive provisions already found, identify SUPPORTING provisions that
a well-prepared exam candidate must also cite.

These are often missed but always required:
1. PROCEDURAL provisions:
   - Which court has jurisdiction? (BGG 72-89 for Federal Tribunal)
   - What is the appeal mechanism? (Beschwerde in Strafsachen, Zivilsachen, öff. Recht)
   - Filing deadlines (BGG 100 Abs. 1: 30 Tage)
   - Cost allocation (BGG 65-68, ZPO 106-107)
   - Legal aid (unentgeltliche Rechtspflege, BGG 64)

2. INTERPRETIVE provisions:
   - Definition articles (e.g., StGB 110 Definitionen)
   - General clauses (OR 2 Treu und Glauben, ZGB 4 Richterliches Ermessen)

3. AUXILIARY provisions:
   - Transitional provisions (Übergangsrecht)
   - Scope of application articles

Generate 2-4 German search queries for supporting provisions.

Output JSON:
{
  "supporting_queries": [
    {"type": "procedural", "query": "Beschwerde Bundesgericht Strafsachen BGG 78"},
    {"type": "cost", "query": "Gerichtskosten Parteientschädigung BGG 65 66"}
  ]
}
"""

# ─── CROSS-REFERENCE AGENT ────────────────────────────────────────────────
# NEW agent not in LegalMALR — targets constitutional and cross-referenced articles.
CROSSREF_PROMPT = """You are a Swiss constitutional and cross-reference law expert.
Given a legal scenario, identify:

1. CONSTITUTIONAL provisions (BV articles) that underlie the specific rules:
   - Art. 9 BV (Willkürverbot / prohibition of arbitrariness)
   - Art. 29 BV (Verfahrensgarantien / procedural guarantees)
   - Art. 32 BV (Unschuldsvermutung / presumption of innocence)
   - Art. 10 BV (Recht auf persönliche Freiheit)
   - Art. 13 BV (Schutz der Privatsphäre)
   - Art. 36 BV (Einschränkung von Grundrechten)

2. GENERAL PROVISIONS that apply alongside specific rules:
   - ATSG provisions alongside IVG/UVG/AVIG
   - OR general part alongside specific contracts
   - ZGB Einleitungsartikel (ZGB 1-10)

3. CROSS-REFERENCED articles:
   - If Art. X refers to Art. Y, both should be cited
   - "sinngemäss anwendbar" references

Generate 2-4 German search queries for these cross-references.

Output JSON:
{
  "crossref_queries": [
    {"type": "constitutional", "query": "Willkürverbot Art 9 BV Grundrechtseingriff"},
    {"type": "general", "query": "ATSG Allgemeiner Teil Invalidenversicherung"}
  ]
}
"""

# Map agent names to their prompts
AGENT_PROMPTS = {
    "PLANNER":    PLANNER_PROMPT,
    "REWRITE":    REWRITE_PROMPT,
    "SUPPLEMENT":  SUPPLEMENT_PROMPT,
    "DECOMPOSE":  DECOMPOSE_PROMPT,
    "SUPPORTIVE": SUPPORTIVE_PROMPT,
    "CROSSREF":   CROSSREF_PROMPT,
}

```

### `stage2_mas_retrieval.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage2_mas_retrieval.py"

"""
stage2_mas_retrieval.py -- STAGE 2: Multi-Agent Sparse Retrieval (LegalMALR-Adapted)

This is the CORE INNOVATION of the revised pipeline, directly adapted from
LegalMALR's MAS architecture but using BM25 instead of dense retrieval.

Architecture:
  - Planner Agent:       Decides which specialist to invoke next, or terminate
  - Rewrite Agent:       EN colloquial -> precise DE legal terminology
  - Supplement Agent:    Makes implicit conditions explicit
  - Decompose Agent:     Splits multi-issue queries into sub-queries
  - Supportive Agent:    Targets procedural/auxiliary provisions
  - Cross-Reference Agent: Finds constitutional + cross-referenced articles

Each agent generates German search queries -> BM25 retrieval -> merge into pool.
The loop runs 2-4 iterations, stopping when no new candidates are found.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python stage2_mas_retrieval.py                              # all val queries
#   python stage2_mas_retrieval.py --split test                 # test queries
#   python stage2_mas_retrieval.py --query "A claimant was..."  # single query
#
# ─── PREREQUISITES ──────────────────────────────────────────────────────────
#   index/bm25_v2_index.pkl   (from stage0)
#   index/bm25_v2_ids.pkl     (from stage0)
#   checkpoints/stage1_{split}.json  (from stage1)
#   GPU: Qwen3-4B loaded (~2.8 GB VRAM)
#
# ─── OUTPUTS ────────────────────────────────────────────────────────────────
#   checkpoints/stage2_{split}.json
#   Per query: {query_id: {candidate_pool: [...], retrieval_log: [...]}}
#
# ─── EXPECTED TIMING ────────────────────────────────────────────────────────
#   ~20-40 seconds per query (2-4 iterations × LLM + BM25)
#   ~40 queries × 30s = ~20 min for full val set
"""

import argparse
import json
import os
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

from data.data_paths import (
    TRAIN_CSV, VAL_CSV, TEST_CSV,
    CHECKPOINTS_DIR, STATUTORY_BM25_PKL,
)
from agent.prompts.mas_prompts import AGENT_PROMPTS

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

FORCE_FIRST_ACTION = os.getenv("SWISS_STAGE2_FORCE_FIRST_ACTION", "").strip().upper()
if FORCE_FIRST_ACTION not in {"REWRITE", "SUPPLEMENT", "DECOMPOSE", "SUPPORTIVE", "CROSSREF"}:
    FORCE_FIRST_ACTION = ""


def mas_retrieve_single(
    query_text: str,
    stage1_analysis: dict,
    sparse_retriever,
    llm_backend,
    max_iterations: int = 4,
    bm25_top_k: int = 30,
) -> dict:
    """
    Run MAS iterative retrieval for a single query.

    Returns {
        'candidate_pool': set of citation strings,
        'retrieval_log':  list of iteration records,
        'sources': dict mapping citation -> set of source names
    }
    """
    candidate_pool: set[str] = set()
    sources: dict[str, set[str]] = {}  # cite -> {'explicit', 'bm25_initial', 'mas_rewrite', ...}
    bm25_scores: dict[str, float] = {}  # cite -> max BM25 score across all queries

    # Build abbreviation normalization map (STPO->StPO, STGB->StGB, etc.)
    _upper_to_canon: dict[str, str] = {}
    try:
        from data.data_paths import KB_JSONL
        with open(KB_JSONL, encoding="utf-8") as _f:
            import json as _json
            for _line in _f:
                _rec = _json.loads(_line)
                _ab = (_rec.get("law") or {}).get("law_abbreviation") or ""
                if _ab:
                    _upper_to_canon[_ab.upper()] = _ab
    except Exception:
        pass

    def _norm(cite: str) -> str:
        """Normalize BM25 citation casing: 'Art. 222 STPO' -> 'Art. 222 StPO'."""
        if not cite.startswith("Art.") or not _upper_to_canon:
            return cite
        parts = cite.rsplit(" ", 1)
        if len(parts) == 2:
            return f"{parts[0]} {_upper_to_canon.get(parts[1].upper(), parts[1])}"
        return cite

    def add_candidates(cites, source_name):
        for c, score in cites:
            c = _norm(c)
            candidate_pool.add(c)
            if c not in sources:
                sources[c] = set()
            sources[c].add(source_name)
            # Track max BM25 score per citation
            if score > 0:
                bm25_scores[c] = max(bm25_scores.get(c, 0.0), score)

    retrieval_log = []

    # ── Initial: explicit citations from query ───────────────────────────
    explicit = set(stage1_analysis.get("explicit_citations", []))
    for c in explicit:
        candidate_pool.add(c)
        sources[c] = {"explicit_from_query"}

    # ── Initial: BM25 with original query + all DE queries from Stage 1 ──
    de_queries = stage1_analysis.get("search_queries_de", [])
    hyde_docs = stage1_analysis.get("hyde_documents_de", [])
    xl_queries = stage1_analysis.get("search_queries_de_crosslingual", [])
    en_queries = [query_text] + stage1_analysis.get("search_queries_en", [])
    # Include HyDE docs and cross-lingual queries in initial retrieval —
    # these were computed in Stage 1 but previously unused until MAS loop.
    all_initial_queries = en_queries + de_queries + hyde_docs + xl_queries

    for q in all_initial_queries:
        results = sparse_retriever.search_combined([q], top_k=50)
        add_candidates(results, "bm25_initial")

    # Track top-10 for confidence scoring
    initial_results = sparse_retriever.search_combined(all_initial_queries, top_k=10)
    bm25_top10 = {c for c, _ in initial_results}
    for c in bm25_top10:
        if c in sources:
            sources[c].add("bm25_top10")

    retrieval_log.append({
        "iteration": 0,
        "action": "INITIAL",
        "queries": all_initial_queries,
        "new_candidates": len(candidate_pool),
        "total_pool": len(candidate_pool),
    })

    # ── LLM-suggested candidates from Stage 1 ───────────────────────────
    for c in stage1_analysis.get("candidate_articles", []):
        candidate_pool.add(c)
        sources.setdefault(c, set()).add("llm_stage1")
    for c in stage1_analysis.get("candidate_cases", []):
        candidate_pool.add(c)
        sources.setdefault(c, set()).add("llm_stage1")
    for c in stage1_analysis.get("procedural_articles", []):
        candidate_pool.add(c)
        sources.setdefault(c, set()).add("llm_stage1_procedural")

    # ── MAS Iterative Loop ───────────────────────────────────────────────
    # Optional force-schedule for the first iteration. Leave unset by default so
    # the planner can pick the best first specialist for the query.
    _forced_actions = {1: FORCE_FIRST_ACTION} if FORCE_FIRST_ACTION else {}

    for iteration in range(1, max_iterations + 1):
        prev_size = len(candidate_pool)

        # Check if this iteration has a forced action
        action = _forced_actions.get(iteration)
        planner_response = {"reasoning": f"forced {action}", "action": action} if action else None

        if not action:
            # Planner decides next action
            planner_context = (
                f"SCENARIO:\n{query_text}\n\n"
                f"LEGAL ANALYSIS:\n{json.dumps(stage1_analysis.get('legal_issues', []), indent=2)}\n\n"
                f"CURRENT CANDIDATE POOL: {len(candidate_pool)} citations\n"
                f"ITERATION: {iteration}/{max_iterations}\n"
                f"PREVIOUS ACTIONS: {[r['action'] for r in retrieval_log]}\n\n"
                f"CURRENT CANDIDATE LIST:\n{json.dumps(sorted(candidate_pool), ensure_ascii=False)}"
            )

            planner_response = llm_backend.generate_json(
                system_prompt=AGENT_PROMPTS["PLANNER"],
                user_prompt=planner_context,
                max_new_tokens=256,
                enable_thinking=False,
            )

            action = str(planner_response.get("action", "TERMINATE")).strip().upper()

        if action == "TERMINATE" or action not in AGENT_PROMPTS:
            retrieval_log.append({
                "iteration": iteration,
                "action": "TERMINATE",
                "reasoning": planner_response.get("reasoning", ""),
                "total_pool": len(candidate_pool),
            })
            break

        # Selected agent generates reformulations
        agent_context = (
            f"SCENARIO:\n{query_text}\n\n"
            f"LEGAL ANALYSIS:\n{json.dumps(stage1_analysis.get('legal_issues', []), indent=2)}\n\n"
            f"ALREADY FOUND ({len(candidate_pool)} citations):\n"
            f"{json.dumps(sorted(candidate_pool), ensure_ascii=False)}\n\n"
            f"Generate search queries to find provisions NOT YET in the pool."
        )

        agent_response = llm_backend.generate_json(
            system_prompt=AGENT_PROMPTS[action],
            user_prompt=agent_context,
            max_new_tokens=512,
            enable_thinking=False,
        )

        # Extract queries from agent response (format varies by agent)
        reformulated_queries = _extract_queries(agent_response)

        if not reformulated_queries:
            print(
                f"    Iter {iteration}: {action} returned 0 queries "
                f"(response keys: {sorted(agent_response.keys()) if isinstance(agent_response, dict) else type(agent_response).__name__})"
            )

        # Each reformulation triggers BM25 retrieval
        new_count = 0
        for rq in reformulated_queries:
            results = sparse_retriever.search_combined([rq], top_k=bm25_top_k)
            for c, _ in results:
                if c not in candidate_pool:
                    new_count += 1
            add_candidates(results, f"mas_{action.lower()}")

        retrieval_log.append({
            "iteration": iteration,
            "action": action,
            "reasoning": planner_response.get("reasoning", ""),
            "queries": reformulated_queries,
            "new_candidates": new_count,
            "total_pool": len(candidate_pool),
        })

        print(f"    Iter {iteration}: {action} → {len(reformulated_queries)} queries → +{new_count} new ({len(candidate_pool)} total)")

        # Early termination: no new candidates found
        if len(candidate_pool) == prev_size:
            retrieval_log.append({
                "iteration": iteration + 1,
                "action": "TERMINATE (no new candidates)",
                "total_pool": len(candidate_pool),
            })
            break

    return {
        "candidate_pool": sorted(candidate_pool),
        "sources": {c: sorted(s) for c, s in sources.items()},
        "bm25_scores": bm25_scores,  # cite -> max BM25 score (for Stage 5 scoring)
        "retrieval_log": retrieval_log,
    }


def _extract_queries(agent_response: dict) -> list[str]:
    """Extract search queries from common agent JSON response shapes."""
    if not isinstance(agent_response, dict):
        return []

    queries: list[str] = []

    def _push(value):
        if isinstance(value, str):
            queries.append(value)
            return
        if isinstance(value, dict):
            for key in ("query", "search_query", "search", "text"):
                if isinstance(value.get(key), str):
                    queries.append(value[key])
                    return
            return
        if isinstance(value, (list, tuple)):
            for item in value:
                _push(item)

    for key in (
        "queries",
        "sub_issues",
        "implicit_issues",
        "supporting_queries",
        "crossref_queries",
        "search_queries",
        "retrieval_queries",
    ):
        _push(agent_response.get(key))

    seen = set()
    cleaned = []
    for q in queries:
        q = q.strip()
        if len(q) <= 3 or q in seen:
            continue
        seen.add(q)
        cleaned.append(q)
    return cleaned


def run_batch(split: str, backend: str = "local", max_iterations: int = 4,
              resume: bool = True) -> dict:
    """Run Stage 2 on all queries in a split."""
    import pandas as pd
    from retrieval.sparse_retriever import SparseRetriever
    import agent.llm_backend as llm_backend

    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    df = pd.read_csv(csv_map[split])

    # Load Stage 1 results
    stage1_path = CHECKPOINTS_DIR / f"stage1_{split}.json"
    if not stage1_path.exists():
        print(f"ERROR: Stage 1 results not found: {stage1_path}")
        print("Run first: python stage1_query_analysis.py")
        sys.exit(1)
    with open(stage1_path, encoding="utf-8") as f:
        stage1_results = json.load(f)

    # Load BM25 retriever
    print("Loading BM25 retriever...")
    sparse = SparseRetriever()

    # Load LLM (stays in memory)
    if backend == "local":
        print("Loading Qwen3-4B...")
        llm_backend.get_model_and_tokenizer()

    out_path = CHECKPOINTS_DIR / f"stage2_{split}.json"
    results = {}
    if resume and out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            results = json.load(f)
        print(f"  Resuming: {len(results)} queries already done")

    for i, (_, row) in enumerate(df.iterrows()):
        qid = row["query_id"]
        if qid in results:
            continue

        query = row["query"]
        analysis = stage1_results.get(qid, {})

        print(f"\n[{i+1}/{len(df)}] {qid}")

        # Clear VRAM before each query
        if backend == "local":
            try:
                import torch
                torch.cuda.empty_cache()
            except Exception:
                pass

        t0 = time.time()
        try:
            result = mas_retrieve_single(
                query_text=query,
                stage1_analysis=analysis,
                sparse_retriever=sparse,
                llm_backend=llm_backend,
                max_iterations=max_iterations,
            )
        except Exception as e:
            print(f"  ERROR: {e}")
            result = {"candidate_pool": [], "sources": {}, "retrieval_log": [{"error": str(e)}]}

        elapsed = time.time() - t0
        print(f"  Done in {elapsed:.1f}s — {len(result['candidate_pool'])} candidates")

        results[qid] = result

        # Save incrementally
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\nStage 2 complete. Saved: {out_path}")
    return results


### Stage 2 runner

In [ ]:
stage2_outputs = run_stage2(split=SPLIT, backend=BACKEND, max_iterations=STAGE2_MAX_ITERATIONS)

## Stage 3 — Citation Graph Expansion

Graph-based candidate expansion.

### `stage3_graph_expansion.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage3_graph_expansion.py"

"""
stage3_graph_expansion.py -- STAGE 3: Citation Graph Traversal + Expansion

After MAS produces the candidate pool, expand via the citation graph:
  - Statute found → find citing court decisions (top 5 per article)
  - Court decision found → find cited statutes

This is the RECALL MULTIPLIER for case law:
  - 59.4% of val citations are statutory, 40.6% are case law
  - Train data is 98.8% statutory → model can't learn case law patterns
  - The citation graph bridges this gap mechanically

CPU only, runs in <1 second per query.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python stage3_graph_expansion.py                    # all val queries
#   python stage3_graph_expansion.py --split test       # test queries
#
# ─── PREREQUISITES ──────────────────────────────────────────────────────────
#   index/citation_graph.pkl       (from stage0)
#   checkpoints/stage2_{split}.json (from stage2)
#
# ─── OUTPUTS ────────────────────────────────────────────────────────────────
#   checkpoints/stage3_{split}.json
#   Per query: {query_id: {expanded_pool: [...], sources: {...},
#               graph_stats: {articles_expanded, cases_found, ...}}}
#
# ─── EXPECTED TIMING ────────────────────────────────────────────────────────
#   <1 second per query, ~40 queries = ~30 seconds total
"""

import argparse
import json
import re
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

from data.data_paths import (
    CHECKPOINTS_DIR,
    CITATION_GRAPH_PKL,
    GOLD_COCITATION_PRIOR_TRAIN_PKL,
    GOLD_COCITATION_PRIOR_TRAINVAL_PKL,
    LOOKUP_PKL,
)

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)


def is_law_citation(c: str) -> bool:
    return bool(re.match(r"^Art\.\s+\d", c.strip()))


def is_case_citation(c: str) -> bool:
    return bool(re.match(r"^BGE\s+\d+", c.strip()) or re.match(r"^\d+[A-Z]_\d+/\d{4}", c.strip()))


def _extract_base_article(cite: str) -> str | None:
    """'Art. 221 Abs. 1 lit. b StPO' -> 'Art. 221 StPO'"""
    m = re.match(r"(Art\.\s+\d+[a-z]*)\s+(?:Abs\.|lit\.).*?\s+(\S+)$", cite, re.IGNORECASE)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    m = re.match(r"(Art\.\s+\d+[a-z]*)\s+(\S+)$", cite, re.IGNORECASE)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    return None


# Sources considered "strong" for targeted sibling expansion
_STRONG_SOURCES = {
    "explicit_from_query", "bm25_top10", "bm25_initial",
    "llm_stage1", "llm_stage1_procedural",
    "mas_rewrite", "mas_supplement", "mas_decompose",
    "mas_supportive", "mas_crossref",
}

_GOLD_PRIOR_RULES = {
    "law-law": {"min_support": 3, "min_npmi": 0.45, "top_n": 3},
    "law-court": {"min_support": 2, "min_npmi": 0.70, "top_n": 1},
    "court-court": {"min_support": 3, "min_npmi": 0.90, "top_n": 1},
}


def expand_single_query(
    candidate_pool: list[str],
    sources: dict[str, list[str]],
    graph_retriever,
    gold_prior=None,
    gold_prior_rules: dict[str, dict[str, float | int]] | None = None,
    top_cases_per_article: int = 3,
    top_related_cases: int = 2,
    a2a_lower: dict[str, list[str]] | None = None,
) -> dict:
    """
    Expand a candidate pool via the citation graph.

    Three expansion directions:
      1. Statute  → top citing court decisions   (article_to_cases)
      2. Case     → all cited statutes            (case_to_articles)
      3. Case     → top related cases             (case_to_cases)  ← NEW

    Returns {
        'expanded_pool': list of all citations (original + graph-found),
        'sources': updated sources dict,
        'graph_stats': {...}
    }
    """
    expanded = set(candidate_pool)
    src = {c: set(s) for c, s in sources.items()}

    articles_expanded      = 0
    cases_found            = 0
    reverse_articles_found = 0
    related_cases_found    = 0

    for cite in list(candidate_pool):
        if is_law_citation(cite):
            # Statute → top citing court decisions
            citing_cases = graph_retriever.articles_to_cases(
                [cite], top_k_per_article=top_cases_per_article
            )
            for case_cite, score in citing_cases[:top_cases_per_article]:
                if case_cite not in expanded:
                    cases_found += 1
                expanded.add(case_cite)
                src.setdefault(case_cite, set()).add("citation_graph")
            if citing_cases:
                articles_expanded += 1

        elif is_case_citation(cite):
            # Court decision → cited statutes
            cited_articles = graph_retriever.cases_to_articles([cite])
            for art in cited_articles:
                if art not in expanded:
                    reverse_articles_found += 1
                expanded.add(art)
                src.setdefault(art, set()).add("citation_graph")

            # Court decision → related cases (case-to-case graph)
            related = graph_retriever.graph.get_related_cases(
                cite, top_n=top_related_cases
            )
            for rel_cite in related:
                if rel_cite not in expanded:
                    related_cases_found += 1
                expanded.add(rel_cite)
                src.setdefault(rel_cite, set()).add("citation_graph")

    # ── Targeted sibling Abs. expansion ─────────────────────────────────
    # For articles with strong retrieval signals, add all sibling Abs.
    # paragraphs from the same base article. E.g., if Art. 277 Abs. 2 ZGB
    # has a bm25_initial hit, also add Art. 277 Abs. 1 ZGB.
    sibling_found = 0
    if a2a_lower:
        pool_lower = {c.lower() for c in expanded}
        for cite in list(candidate_pool):
            if not cite.startswith("Art."):
                continue
            cite_sources = src.get(cite, set())
            if not cite_sources & _STRONG_SOURCES:
                continue
            base = _extract_base_article(cite)
            if not base:
                continue
            children = a2a_lower.get(base.lower(), [])
            for sib in children:
                if sib.lower() not in pool_lower:
                    expanded.add(sib)
                    pool_lower.add(sib.lower())
                    src.setdefault(sib, set()).add("sibling_expansion")
                    sibling_found += 1

    # ── Co-citation expansion: use PMI-ranked co-citations ──────────────
    # PMI filters out ubiquitous articles (Art. 66 BGG) that co-occur with
    # everything but carry no relevance signal. Tighter params than before:
    # top_n=3 (was 5), only expand from Stage 2 original pool (not graph-found).
    co_cited_found = 0
    _original_pool = set(candidate_pool)  # only expand from retrieval hits
    for cite in list(_original_pool):
        companions = graph_retriever.graph.get_co_cited_pmi(
            cite, top_n=3
        )
        for companion, pmi_score in companions:
            if pmi_score < 2.0:  # skip low-PMI (generic) co-citations
                continue
            if companion not in expanded:
                co_cited_found += 1
            expanded.add(companion)
            src.setdefault(companion, set()).add("co_citation")

    # ── Train-only gold co-citation prior expansion ──────────────────────
    # This uses train supervision only, so it is safe for runtime. We keep it
    # conservative and pair-type-aware because court-court co-occurrences are
    # much sparser and noisier than statute bundles.
    gold_prior_found = 0
    gold_prior_rules = gold_prior_rules or _GOLD_PRIOR_RULES
    if gold_prior:
        for cite in list(_original_pool):
            cite_sources = src.get(cite, set())
            if not cite_sources & _STRONG_SOURCES:
                continue

            pair_types = {"law-law", "law-court"} if is_law_citation(cite) else {"law-court", "court-court"}
            for pair_type in pair_types:
                rule = gold_prior_rules[pair_type]
                companions = gold_prior.get_companions(
                    cite,
                    pair_types={pair_type},
                    min_support=rule["min_support"],
                    min_npmi=rule["min_npmi"],
                    top_n=rule["top_n"],
                )
                for item in companions:
                    companion = item["citation"]
                    if companion not in expanded:
                        gold_prior_found += 1
                    expanded.add(companion)
                    src.setdefault(companion, set()).add(f"gold_cocitation_prior:{pair_type}")

    return {
        "expanded_pool": sorted(expanded),
        "sources": {c: sorted(s) for c, s in src.items()},
        "gold_prior_rules": gold_prior_rules if gold_prior else None,
        "graph_stats": {
            "original_pool":           len(candidate_pool),
            "expanded_pool":           len(expanded),
            "articles_expanded":       articles_expanded,
            "cases_found_via_graph":   cases_found,
            "reverse_articles_found":  reverse_articles_found,
            "related_cases_found":     related_cases_found,
            "co_cited_found":          co_cited_found,
            "gold_prior_found":        gold_prior_found,
            "sibling_found":           sibling_found,
        },
    }


def _load_gold_prior(prior_mode: str):
    if prior_mode == "off":
        print("  Gold co-citation prior disabled")
        return None

    from stage0f_build_gold_cocitation_prior import GoldCoCitationPrior

    if prior_mode == "trainval":
        prior_path = GOLD_COCITATION_PRIOR_TRAINVAL_PKL
        label = "train+val"
    else:
        prior_path = GOLD_COCITATION_PRIOR_TRAIN_PKL
        label = "train-only"

    if not prior_path.exists():
        print(f"  Gold co-citation prior not found at {prior_path}; skipping that expansion channel")
        return None

    print(f"Loading {label} gold co-citation prior...")
    prior = GoldCoCitationPrior(prior_path)
    print(f"  Loaded prior '{prior.prior_name}' for {len(prior.neighbors):,} citations")
    return prior


def run_batch(
    split: str,
    resume: bool = True,
    gold_prior_mode: str = "train",
    gold_prior_rules: dict[str, dict[str, float | int]] | None = None,
) -> dict:
    """Run Stage 3 on all queries in a split."""
    from retrieval.graph_retriever import GraphRetriever

    # Load Stage 2 results
    stage2_path = CHECKPOINTS_DIR / f"stage2_{split}.json"
    if not stage2_path.exists():
        print(f"ERROR: Stage 2 results not found: {stage2_path}")
        print("Run first: python stage2_mas_retrieval.py")
        sys.exit(1)
    with open(stage2_path, encoding="utf-8") as f:
        stage2_results = json.load(f)

    # Load citation graph
    if not CITATION_GRAPH_PKL.exists():
        print(f"ERROR: Citation graph not found: {CITATION_GRAPH_PKL}")
        print("Run first: python stage0_build_index.py --graph-only")
        sys.exit(1)
    print("Loading citation graph...")
    graph = GraphRetriever()

    gold_prior = _load_gold_prior(gold_prior_mode)

    # Load article_to_abs lookup for sibling expansion
    import pickle
    a2a_lower = {}
    if LOOKUP_PKL.exists():
        with open(LOOKUP_PKL, "rb") as f:
            lookup = pickle.load(f)
        a2a = lookup.get("article_to_abs", {})
        a2a_lower = {k.lower(): v for k, v in a2a.items()}
        print(f"  Loaded article_to_abs: {len(a2a_lower)} entries for sibling expansion")

    out_path = CHECKPOINTS_DIR / f"stage3_{split}.json"
    results = {}
    if resume and out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            results = json.load(f)
        print(f"  Resuming: {len(results)} queries already done")

    total_original = 0
    total_expanded = 0

    for qid, stage2 in stage2_results.items():
        if qid in results:
            stats = results[qid].get("graph_stats", {})
            total_original += stats.get("original_pool", 0)
            total_expanded += stats.get("expanded_pool", 0)
            continue

        t0 = time.time()
        result = expand_single_query(
            candidate_pool=stage2["candidate_pool"],
            sources=stage2.get("sources", {}),
            graph_retriever=graph,
            gold_prior=gold_prior,
            gold_prior_rules=gold_prior_rules,
            a2a_lower=a2a_lower,
        )
        elapsed = time.time() - t0

        stats = result["graph_stats"]
        total_original += stats["original_pool"]
        total_expanded += stats["expanded_pool"]

        print(f"  {qid}: {stats['original_pool']} → {stats['expanded_pool']} "
              f"(+{stats['cases_found_via_graph']} cases, +{stats['reverse_articles_found']} arts) "
              f"[{elapsed:.2f}s]")

        results[qid] = result

    # Save
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\nStage 3 complete. Saved: {out_path}")
    print(f"  Total: {total_original} → {total_expanded} citations "
          f"(+{total_expanded - total_original} from graph)")
    return results


### Stage 3 runner

In [ ]:
stage3_outputs = run_stage3(split=SPLIT, gold_prior_mode=STAGE3_GOLD_PRIOR_MODE)

## Stage 4 — Reranking and Direct Generation

LLM reranking and cross-encoder scoring.

### `agent/prompts/reranker_prompt.txt`

```text
You are a Swiss law expert evaluating candidate legal provisions for relevance.

SCENARIO:
{query_text}

CANDIDATE PROVISIONS (with text excerpts):
{numbered_candidates_with_text}

TASK: Rate each candidate's relevance to the SPECIFIC scenario above.
Use this scale:
  3 = DIRECTLY APPLICABLE: the provision's conditions or legal rule are met by
      the facts in the scenario, or the case directly addresses this legal question
  2 = CLOSELY RELATED: same legal issue area, procedural basis, cost/fee articles,
      constitutional rights, general definitions, or leading cases on related questions
  1 = TANGENTIAL: same law area but addresses a different situation or sub-issue
  0 = NOT RELEVANT: different legal domain or clearly inapplicable

RULES — be inclusive:
- Rate 3 if the provision TEXT directly matches the scenario facts.
- Rate 2 for procedural provisions (appeal rights, costs, legal aid, admissibility),
  constitutional provisions, general-part articles, and leading cases from the same area.
- A court decision typically cites 15-40 provisions. When in doubt, rate 2 not 1.
- From a batch of 30, expect 3-8 rated as 3, and 5-15 rated as 2.

Output a JSON object mapping each selected provision (rating >= 2) to its score.
Use the EXACT citation strings from the CANDIDATES list above.
Omit provisions rated 0 or 1.

Format: {{"selected": {{"<citation from list>": <rating>, ...}}}}

```

### `agent/prompts/direct_gen_prompt.txt`

```text
You are drafting the citation list for a Swiss Federal Tribunal decision.

SCENARIO:
{query_text}

CITATIONS ALREADY FOUND:
{selected_citations}

Your task: generate ADDITIONAL Swiss legal citations that a court would cite
in its decision on this scenario but are MISSING from the list above.

A typical Federal Tribunal decision cites 15-40 provisions. Generate citations
in these categories that are NOT yet in the list:

1. PROCEDURAL (almost always cited):
   - BGG admissibility: Art. 82/83/90/93/95/97/100/105/106/107/113 BGG
   - Party standing: Art. 76 Abs. 1 BGG, Art. 382 Abs. 1 StPO
   - Costs: Art. 66/68 BGG, Art. 422/428 StPO, Art. 135 StPO
   - Cantonal court: Art. 37/39 StBOG, Art. 80 Abs. 1 BGG

2. CONSTITUTIONAL (when fundamental rights are at issue):
   - Art. 29 Abs. 2 BV (right to be heard)
   - Art. 10 Abs. 2 BV (personal liberty)
   - Art. 31 BV (deprivation of liberty)

3. SUBSTANTIVE: Leading BGE decisions and statutory articles on the
   specific legal issues. Use the format:
   - Articles: "Art. [number] [Abs. X] [lit. x] [law abbreviation]"
   - BGE: "BGE [volume] [part] [page] E. [section]"
   - Case: "[court]_[number]/[year] E. [section]"

Generate 10-20 additional citations. Better to over-generate — verification
will filter out any that don't exist in the corpus.

Output ONLY a JSON object:
{{"additional": ["Art. 100 Abs. 1 BGG", "Art. 66 Abs. 1 BGG", "BGE 137 IV 122 E. 6.2", ...]}}

```

### `stage4_llm_reranker.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage4_llm_reranker.py"

"""
stage4_llm_reranker.py -- STAGE 4: LLM Reranker + Direct Citation Generation

Two functions using the same Qwen3-4B model:

  4A. RERANKING: Given the expanded candidate pool from Stage 3, ask the LLM
      to select which provisions are actually relevant. This is the precision
      stage — filtering the high-recall pool down to what should be cited.

      Following LegalMALR Section 3.4: no chain-of-thought in output.
      The LLM reasons internally but outputs only the final selection.

      Each candidate is shown with its actual article text from the KB so the
      LLM can make informed relevance judgments, not just guess from IDs.

  4B. DIRECT GENERATION: After reranking, ask the LLM to generate any
      additional citations from memory that aren't in the candidate pool.
      These go through existence verification in Stage 5.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python stage4_llm_reranker.py                     # all val queries
#   python stage4_llm_reranker.py --split test        # test queries
#   python stage4_llm_reranker.py --skip-direct-gen   # reranking only
#
# ─── PREREQUISITES ──────────────────────────────────────────────────────────
#   checkpoints/stage1_{split}.json  (from stage1)
#   checkpoints/stage3_{split}.json  (from stage3)
#   index/laws_knowledge_base.jsonl  (from stage0, for article texts)
#   GPU: Qwen3-4B loaded (~2.8 GB VRAM)
#
# ─── OUTPUTS ────────────────────────────────────────────────────────────────
#   checkpoints/stage4_{split}.json
#   Per query: {query_id: {reranked: [...], direct_gen: [...], sources: {...}}}
#
# ─── EXPECTED TIMING ────────────────────────────────────────────────────────
#   Pre-filter to ~1200 candidates (skip co-citation-only noise)
#   batch_size=50 with text -> ~24 batches/query -> ~35-60 min for 10 val queries
"""

import argparse
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

from data.data_paths import CHECKPOINTS_DIR

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# Load prompt templates
_PROMPTS_DIR = Path(__file__).parent / "agent" / "prompts"
_RERANKER_PROMPT = (_PROMPTS_DIR / "reranker_prompt.txt").read_text(encoding="utf-8")
_DIRECT_GEN_PROMPT = (_PROMPTS_DIR / "direct_gen_prompt.txt").read_text(encoding="utf-8")

# Sources that indicate real retrieval signal (includes co_citation)
_SIGNAL_SOURCES = {
    "explicit_from_query", "bm25_top10", "bm25_initial",
    "citation_graph", "llm_stage1", "llm_stage1_procedural",
    "mas_rewrite", "mas_supplement", "mas_decompose",
    "mas_supportive", "mas_crossref",
    "co_citation",
}

# Singleton citation text cache
_CITATION_TEXTS: dict[str, str] | None = None


def _load_citation_texts() -> dict[str, str]:
    """
    Load article text snippets from KB JSONL once.
    Returns {citation_canon: text_snippet} with base-article fallbacks.

    For 'Art. 221 Abs. 1 StPO' with no direct entry, falls back to 'Art. 221 StPO'.
    For BGE/case citations, uses court decision summary if present.
    """
    global _CITATION_TEXTS
    if _CITATION_TEXTS is not None:
        return _CITATION_TEXTS

    from data.data_paths import KB_JSONL

    texts: dict[str, str] = {}
    base_texts: dict[str, str] = {}  # "Art. 221 StPO" -> text (fallback)

    print("  Loading citation texts from KB...")
    try:
        with open(KB_JSONL, encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except Exception:
                    continue
                canon = rec.get("citation_canon", "")
                if not canon:
                    continue
                text = (rec.get("search", {}).get("search_text_de", "")
                        or rec.get("content", {}).get("text_clean_de", "") or "")
                if not text:
                    continue
                texts[canon] = text[:200]
                # Build base-article fallback: "Art. 221 Abs. 1 StPO" -> "Art. 221 StPO"
                if canon.startswith("Art."):
                    parts = canon.split()
                    if len(parts) >= 3:
                        art_num = parts[1]          # "221"
                        law_abbrev = parts[-1]      # "StPO"
                        base = f"Art. {art_num} {law_abbrev}"
                        if base not in base_texts:
                            base_texts[base] = text[:200]
    except Exception as e:
        print(f"  WARNING: Could not load citation texts: {e}")

    # Merge base fallbacks (don't override specific ones)
    for base, text in base_texts.items():
        if base not in texts:
            texts[base] = text

    _CITATION_TEXTS = texts
    print(f"  Loaded {len(_CITATION_TEXTS):,} citation texts")
    return _CITATION_TEXTS


def _get_text(citation: str, texts: dict[str, str]) -> str:
    """
    Look up article text for a citation, with Abs./lit. fallback.

    'Art. 221 Abs. 1 StPO' -> tries exact, then 'Art. 221 StPO'.
    """
    if citation in texts:
        return texts[citation]
    if citation.startswith("Art."):
        parts = citation.split()
        if len(parts) >= 3:
            base = f"Art. {parts[1]} {parts[-1]}"
            if base in texts:
                return texts[base]
    return ""


def rerank_candidates(
    query_text: str,
    candidates: list[str],
    llm_backend,
    citation_texts: dict[str, str],
    batch_size: int = 30,
) -> dict[str, int]:
    """
    4A. Use LLM to score relevant citations from the candidate pool.

    Each candidate is shown with its article text so the LLM can judge
    relevance from content, not just from citation IDs.

    Processes in batches of 30 (with text, ~2K tokens/batch).
    Returns dict of {citation: relevance_tier} where tier is 2 or 3.
    Tier 3 = directly applicable, Tier 2 = closely related.
    """
    all_scored: dict[str, int] = {}

    total_batches = (len(candidates) + batch_size - 1) // batch_size
    for batch_num, start in enumerate(range(0, len(candidates), batch_size), 1):
        batch = candidates[start:start + batch_size]
        print(f"      batch {batch_num}/{total_batches} ({start+1}-{start+len(batch)})...", end=" ", flush=True)

        # Format each candidate with its text snippet
        lines = []
        for i, c in enumerate(batch):
            text = _get_text(c, citation_texts)
            if text:
                snippet = text[:180].replace("\n", " ")
                lines.append(f"{i+1}. {c} — {snippet}")
            else:
                lines.append(f"{i+1}. {c}")
        numbered = "\n".join(lines)

        prompt = _RERANKER_PROMPT.format(
            query_text=query_text,
            numbered_candidates_with_text=numbered,
        )

        response = llm_backend.generate_json(
            system_prompt="You are a Swiss law expert. Output only JSON.",
            user_prompt=prompt,
            max_new_tokens=1024,
            enable_thinking=False,
        )

        selected = response.get("selected", {})
        batch_count = 0
        if isinstance(selected, dict):
            for cite, tier in selected.items():
                cite = cite.strip()
                try:
                    tier = int(tier)
                except (ValueError, TypeError):
                    tier = 2
                if tier >= 2:
                    all_scored[cite] = max(all_scored.get(cite, 0), tier)
                    batch_count += 1
        elif isinstance(selected, list):
            for s in selected:
                if isinstance(s, str):
                    all_scored[s.strip()] = max(all_scored.get(s.strip(), 0), 2)
                    batch_count += 1

        if batch_count == 0 and batch_num <= 3:
            print(f"+0 [DEBUG keys: {list(response.keys())}, "
                  f"type: {type(selected).__name__}, len: {len(selected) if hasattr(selected, '__len__') else '?'}]")
        else:
            print(f"+{batch_count} selected ({len(all_scored)} total)")

    return all_scored


def generate_direct_citations(
    query_text: str,
    already_selected: list[str],
    llm_backend,
) -> list[str]:
    """
    4B. Ask LLM to generate additional citations from memory.

    These are citations the LLM knows are relevant but weren't in the
    retrieval pool. They go through existence verification in Stage 5.
    """
    prompt = _DIRECT_GEN_PROMPT.format(
        query_text=query_text,
        selected_citations="\n".join(already_selected[:30]),
    )

    response = llm_backend.generate_json(
        system_prompt="You are a Swiss law expert. Output only JSON.",
        user_prompt=prompt,
        max_new_tokens=512,
        enable_thinking=False,
    )

    additional = response.get("additional", [])
    if isinstance(additional, list):
        raw = [c.strip() for c in additional if isinstance(c, str)]
        # Filter out non-Swiss or malformed citations:
        # - "ff." notation (not a specific article)
        # - Non-Swiss law abbreviations (CPLR, USC, etc.)
        # - Empty or too-short strings
        import re
        _SWISS_ART = re.compile(r"^Art\.\s+\d+")
        _SWISS_CASE = re.compile(r"^(BGE\s+\d+|\d+[A-Z]_\d+/\d{4})")
        filtered = []
        for c in raw:
            if " ff." in c or " ff " in c:
                continue  # skip "Art. 101 ff. ZGB"
            if _SWISS_ART.match(c) or _SWISS_CASE.match(c):
                filtered.append(c)
        return filtered
    return []


def process_single_query(
    query_text: str,
    stage1_analysis: dict,
    stage3_result: dict,
    llm_backend,
    batch_size: int = 30,
    skip_direct_gen: bool = False,
) -> dict:
    """
    Run Stage 4 for a single query.

    Pre-filters candidates to those with real retrieval signal before reranking
    (skips co-citation-only entries which have no BM25 or graph backing).
    This reduces the pool from ~2500 to ~1200 candidates, cutting LLM calls by ~half.
    """
    candidates = stage3_result.get("expanded_pool", [])
    sources_raw = stage3_result.get("sources", {})
    sources = {c: set(s) for c, s in sources_raw.items()}

    # Pre-filter: only rerank candidates with at least one real retrieval signal.
    # Candidates found ONLY via co-citation (score=0.00) are the weakest signal
    # and contribute mostly noise. Explicit citations always pass through.
    explicit_set = set(stage1_analysis.get("explicit_citations", []))
    rerank_pool = [
        c for c in candidates
        if c in explicit_set
        or any(s in _SIGNAL_SOURCES for s in sources_raw.get(c, []))
    ]

    print(f"    Pre-filter: {len(candidates)} -> {len(rerank_pool)} candidates for reranking")

    # Load citation texts (KB lazy-loaded once across all queries)
    citation_texts = _load_citation_texts()

    # 4A: Reranking — returns {citation: tier} with tier 2 or 3
    reranked_scored = rerank_candidates(query_text, rerank_pool, llm_backend,
                                        citation_texts, batch_size)
    reranked_list = sorted(reranked_scored.keys())
    for c in reranked_scored:
        tier = reranked_scored[c]
        sources.setdefault(c, set()).add("llm_reranker")
        if tier >= 3:
            sources[c].add("llm_reranker_tier3")

    # 4B: Direct citation generation
    direct_gen = []
    if not skip_direct_gen:
        direct_gen = generate_direct_citations(query_text, reranked_list, llm_backend)
        for c in direct_gen:
            sources.setdefault(c, set()).add("llm_direct_gen")

    # Combine: reranked + multi-signal pass-through + explicit + procedural + direct_gen
    import re
    _SWISS_PATTERN = re.compile(
        r"^(Art\.\s+\d+|BGE\s+\d+|\d+[A-Z]_\d+/\d{4})"
    )
    explicit = {c for c in stage1_analysis.get("explicit_citations", [])
                if _SWISS_PATTERN.match(c)}
    procedural = set(stage1_analysis.get("procedural_articles", []))

    # Single-signal pass-through: candidates with 1+ independent retrieval signal
    # survive even if the LLM reranker rejected them. These low-signal candidates
    # have low base scores (~0.10) but can be rescued by cross-encoder scoring
    # in stage4b if the CE rates them as relevant.
    _MULTI_SIGNAL_SOURCES = {
        "explicit_from_query", "bm25_top10", "bm25_initial",
        "citation_graph", "llm_stage1", "llm_stage1_procedural",
        "mas_rewrite", "mas_supplement", "mas_decompose",
        "mas_supportive", "mas_crossref",
        "co_citation",
    }
    multi_signal = []
    for c in rerank_pool:
        if c in reranked_scored:
            continue  # already selected by LLM
        c_sources = sources.get(c, set())
        signal_count = len(c_sources & _MULTI_SIGNAL_SOURCES)
        if signal_count >= 1:
            multi_signal.append(c)

    # Also pass through LLM stage1 candidates (the LLM already thought these
    # were relevant during query analysis)
    stage1_candidates = set(
        stage1_analysis.get("candidate_articles", [])
        + stage1_analysis.get("candidate_cases", [])
    )
    stage1_passthrough = [
        c for c in rerank_pool
        if c in stage1_candidates and c not in reranked_scored
    ]

    all_citations = list(dict.fromkeys(
        reranked_list
        + sorted(multi_signal)
        + sorted(stage1_passthrough)
        + sorted(explicit)
        + sorted(procedural)
        + direct_gen
    ))

    print(f"    Multi-signal pass-through: {len(multi_signal)}, Stage1 pass-through: {len(stage1_passthrough)}")

    return {
        "reranked": reranked_list,
        "reranked_tiers": reranked_scored,   # {cite: 2 or 3} for Stage 5 scoring
        "direct_gen": direct_gen,
        "explicit": sorted(explicit),
        "procedural": sorted(procedural),
        "multi_signal": sorted(multi_signal),
        "stage1_passthrough": sorted(stage1_passthrough),
        "all_citations": all_citations,
        "sources": {c: sorted(s) for c, s in sources.items()
                    if c in set(all_citations)},
        "stats": {
            "candidates_in": len(candidates),
            "rerank_pool": len(rerank_pool),
            "reranked_out": len(reranked_scored),
            "reranked_tier3": sum(1 for t in reranked_scored.values() if t >= 3),
            "reranked_tier2": sum(1 for t in reranked_scored.values() if t == 2),
            "multi_signal": len(multi_signal),
            "stage1_pass": len(stage1_passthrough),
            "direct_gen": len(direct_gen),
            "explicit": len(explicit),
            "procedural": len(procedural),
            "total_out": len(all_citations),
        },
    }


def run_batch(split: str, backend: str = "local",
              skip_direct_gen: bool = False, resume: bool = True) -> dict:
    """Run Stage 4 on all queries in a split."""
    import pandas as pd
    import agent.llm_backend as llm_backend
    from data.data_paths import TRAIN_CSV, VAL_CSV, TEST_CSV

    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    df = pd.read_csv(csv_map[split])

    # Load prerequisites
    stage1_path = CHECKPOINTS_DIR / f"stage1_{split}.json"
    stage3_path = CHECKPOINTS_DIR / f"stage3_{split}.json"
    for p, stage in [(stage1_path, 1), (stage3_path, 3)]:
        if not p.exists():
            print(f"ERROR: Stage {stage} results not found: {p}")
            print(f"Run first: python stage{stage}_*.py")
            sys.exit(1)

    with open(stage1_path, encoding="utf-8") as f:
        stage1_results = json.load(f)
    with open(stage3_path, encoding="utf-8") as f:
        stage3_results = json.load(f)

    # Load LLM
    if backend == "local":
        print("Loading Qwen3-4B...")
        llm_backend.get_model_and_tokenizer()

    out_path = CHECKPOINTS_DIR / f"stage4_{split}.json"
    results = {}
    if resume and out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            results = json.load(f)
        # Remove entries with errors so they get re-run
        failed = [qid for qid, r in results.items()
                  if "error" in r.get("stats", {})]
        for qid in failed:
            del results[qid]
        if failed:
            print(f"  Resuming: {len(results)} done, {len(failed)} failed will be re-run: {failed}")
        else:
            print(f"  Resuming: {len(results)} queries already done")

    for i, (_, row) in enumerate(df.iterrows()):
        qid = row["query_id"]
        if qid in results:
            continue

        query = row["query"]
        analysis = stage1_results.get(qid, {})
        stage3 = stage3_results.get(qid, {"expanded_pool": [], "sources": {}})

        print(f"\n[{i+1}/{len(df)}] {qid}")

        # Clear VRAM before each query — Stage 4 is the heaviest LLM user
        # (~50 batches per query). Without this, VRAM fragments and OOM hits
        # mid-pipeline after several queries.
        if backend == "local":
            try:
                import torch
                torch.cuda.empty_cache()
            except Exception:
                pass

        t0 = time.time()
        try:
            result = process_single_query(
                query, analysis, stage3, llm_backend,
                skip_direct_gen=skip_direct_gen,
            )
        except Exception as e:
            print(f"  ERROR: {e}")
            try:
                import torch
                torch.cuda.empty_cache()
            except Exception:
                pass
            result = {"all_citations": [], "sources": {}, "stats": {"error": str(e)}}

        elapsed = time.time() - t0
        stats = result.get("stats", {})
        print(f"  Done in {elapsed:.1f}s — "
              f"{stats.get('candidates_in', '?')} candidates / "
              f"{stats.get('rerank_pool', '?')} reranked -> "
              f"{stats.get('reranked_out', '?')} LLM-selected "
              f"(T3:{stats.get('reranked_tier3', '?')} T2:{stats.get('reranked_tier2', '?')}) + "
              f"{stats.get('multi_signal', '?')} multi-sig + "
              f"{stats.get('stage1_pass', '?')} s1-pass + "
              f"{stats.get('direct_gen', '?')} direct = "
              f"{stats.get('total_out', '?')} total")

        results[qid] = result

        # Save incrementally
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\nStage 4 complete. Saved: {out_path}")
    return results


### Stage 4 runner

In [ ]:
stage4_outputs = run_stage4(split=SPLIT, backend=BACKEND, skip_direct_gen=STAGE4_SKIP_DIRECT_GEN)

### `stage4b_cross_encoder.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage4b_cross_encoder.py"

"""
stage4b_cross_encoder.py -- STAGE 4b: Cross-Encoder Relevance Scoring

Scores the top-500 candidates (by base composite score) per query using
Qwen3-Reranker-4B as a cross-encoder. Outputs P(yes) probability for each
(query, citation) pair.

These scores are consumed by Stage 5 (Strategy C): the binary llm_reranker/
tier3 signals are replaced with continuous cross-encoder scores x 0.18,
improving macro-F1 from 0.5400 to 0.6520 on val.

# --- HOW TO RUN ---------------------------------------------------------------
#   cd E:\swiss-law-pipeline
#   python stage4b_cross_encoder.py                    # val split
#   python stage4b_cross_encoder.py --split test       # test split
#   python stage4b_cross_encoder.py --top-n 300        # score more candidates
#
# --- PREREQUISITES ------------------------------------------------------------
#   checkpoints/stage4_{split}.json    (from stage4)
#   checkpoints/bm25_scores_{split}.json  (from stage5 first run, or precomputed)
#   index/citation_lookup.pkl          (from stage0, for verification)
#   data/laws_knowledge_base.jsonl     (article texts)
#   data/court_considerations.csv      (court decision texts)
#   GPU: ~2.8 GB VRAM for Qwen3-Reranker-4B NF4
#
# --- OUTPUTS -------------------------------------------------------------------
#   checkpoints/reranker_scores_{split}.json
#     Format: {qid: {citation: float_score, ...}, ...}
#
# --- EXPECTED TIMING -----------------------------------------------------------
#   ~60 min for 10 val queries (500 candidates each, batch_size=8)
"""

import argparse
import csv
import gc
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

from data.data_paths import (
    CHECKPOINTS_DIR, LOOKUP_PKL, KB_JSONL, COURT_CSV,
    VAL_CSV, TEST_CSV, TRAIN_CSV,
)
from agent.verifier import Verifier
from stage5_verify_and_score import verify_and_score_single

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Configuration ─────────────────────────────────────────────────────────────
TOP_N = 500          # Score top-N candidates per query by base composite score
BATCH_SIZE = 8       # Cross-encoder GPU batch size
RERANKER_ID = "Qwen/Qwen3-Reranker-4B"


def _load_texts():
    """Load article and court decision texts for document representation."""
    import csv as csv_mod

    kb_texts = {}
    kb_base_texts = {}
    with open(KB_JSONL, encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except Exception:
                continue
            canon = rec.get("citation_canon", "")
            if not canon:
                continue
            text = (rec.get("search", {}).get("search_text_de", "")
                    or rec.get("content", {}).get("text_clean_de", "") or "")
            if text:
                kb_texts[canon] = text[:500]
                if canon.startswith("Art."):
                    parts = canon.split()
                    if len(parts) >= 3:
                        base = f"Art. {parts[1]} {parts[-1]}"
                        if base not in kb_base_texts:
                            kb_base_texts[base] = text[:500]

    for base, text in kb_base_texts.items():
        if base not in kb_texts:
            kb_texts[base] = text

    return kb_texts


def _load_court_texts(needed_citations: set[str], kb_texts: dict) -> dict[str, str]:
    """Load court consideration texts for non-article citations."""
    court_texts = {}
    matched = 0
    with open(COURT_CSV, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if i % 500000 == 0 and i > 0:
                print(f"    ...{i:,} rows, {matched} matched")
            cite = row.get("citation", "")
            if cite in needed_citations:
                text = row.get("text", "")
                if text:
                    if cite in court_texts:
                        court_texts[cite] += " " + text
                    else:
                        court_texts[cite] = text
                        matched += 1

    for c in court_texts:
        court_texts[c] = court_texts[c][:500]

    return court_texts


def _get_text(citation: str, kb_texts: dict, court_texts: dict) -> str:
    """Look up text for a citation (article or court decision)."""
    if citation in kb_texts:
        return kb_texts[citation]
    if citation.startswith("Art."):
        parts = citation.split()
        if len(parts) >= 3:
            base = f"Art. {parts[1]} {parts[-1]}"
            if base in kb_texts:
                return kb_texts[base]
    if citation in court_texts:
        return court_texts[citation]
    return ""


def _load_reranker():
    """Load Qwen3-Reranker-4B cross-encoder model (NF4 quantized)."""
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    # Unload Qwen3-4B generative model if loaded (free GPU memory)
    try:
        from agent import llm_backend
        if llm_backend._MODEL is not None:
            del llm_backend._MODEL
            del llm_backend._TOKENIZER
            llm_backend._MODEL = None
            llm_backend._TOKENIZER = None
            gc.collect()
            torch.cuda.empty_cache()
            print("  Unloaded Qwen3-4B to free GPU memory.")
    except Exception:
        pass

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        RERANKER_ID, padding_side='left', trust_remote_code=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        RERANKER_ID, quantization_config=quant_config,
        device_map="auto", trust_remote_code=True,
    ).eval()

    token_true_id = tokenizer.convert_tokens_to_ids("yes")
    token_false_id = tokenizer.convert_tokens_to_ids("no")

    prefix = (
        "<|im_start|>system\nJudge whether the Document meets the requirements "
        "based on the Query and the Instruct provided. Note that the answer can "
        "only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
    )
    suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
    prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
    suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)

    return model, tokenizer, token_true_id, token_false_id, prefix_tokens, suffix_tokens


def _format_pair(query: str, doc: str) -> str:
    """Format a (query, document) pair for the cross-encoder."""
    instruction = (
        "Given a Swiss law query describing a legal scenario, determine if the "
        "cited legal provision or court decision is relevant to resolving the query"
    )
    return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}"


def _score_batch(pairs: list[str], model, tokenizer, token_true_id, token_false_id,
                 prefix_tokens, suffix_tokens, batch_size: int = BATCH_SIZE) -> list[float]:
    """Score a batch of (query, doc) pairs with the cross-encoder."""
    import torch

    all_scores = []
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        inputs = tokenizer(
            batch, padding=False, truncation='longest_first',
            return_attention_mask=False,
            max_length=8192 - len(prefix_tokens) - len(suffix_tokens),
        )
        for i, ids in enumerate(inputs['input_ids']):
            inputs['input_ids'][i] = prefix_tokens + ids + suffix_tokens
        inputs = tokenizer.pad(inputs, padding=True, return_tensors="pt")
        for key in inputs:
            inputs[key] = inputs[key].to(model.device)
        with torch.no_grad():
            logits = model(**inputs).logits[:, -1, :]
            true_s = logits[:, token_true_id]
            false_s = logits[:, token_false_id]
            stacked = torch.stack([false_s, true_s], dim=1)
            probs = torch.nn.functional.log_softmax(stacked, dim=1)
            scores = probs[:, 1].exp().tolist()
        all_scores.extend(scores)
        del inputs, logits, true_s, false_s, stacked, probs
    torch.cuda.empty_cache()
    return all_scores


def run_batch(split: str, top_n: int = TOP_N):
    """
    Run Stage 4b: cross-encoder scoring on top-N candidates per query.

    Loads stage4 results, computes base scores (same as stage5), selects
    top-N candidates, scores them with the cross-encoder, and saves the
    result to checkpoints/reranker_scores_{split}.json.
    """
    t_total = time.time()

    # ── Check for existing scores ─────────────────────────────────────────
    out_path = CHECKPOINTS_DIR / f"reranker_scores_{split}.json"
    existing_scores = {}
    if out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            existing_scores = json.load(f)
        print(f"Found existing scores for {len(existing_scores)} queries at {out_path}")

    # ── Load stage4 results ───────────────────────────────────────────────
    stage4_path = CHECKPOINTS_DIR / f"stage4_{split}.json"
    if not stage4_path.exists():
        print(f"ERROR: Stage 4 results not found: {stage4_path}")
        sys.exit(1)
    with open(stage4_path, encoding="utf-8") as f:
        stage4_results = json.load(f)

    # ── Load BM25 cache (for base scoring) ────────────────────────────────
    bm25_cache = {}
    bm25_path = CHECKPOINTS_DIR / f"bm25_scores_{split}.json"
    if bm25_path.exists():
        with open(bm25_path, encoding="utf-8") as f:
            bm25_cache = json.load(f)

    # ── Load verifier ─────────────────────────────────────────────────────
    verifier = Verifier() if LOOKUP_PKL.exists() else None

    # ── Load query texts ──────────────────────────────────────────────────
    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    query_texts = {}
    with open(csv_map[split], encoding="utf-8") as f:
        for row in csv.DictReader(f):
            query_texts[row["query_id"]] = row["query"]

    # ── Compute base scores to select top-N candidates ────────────────────
    print(f"\nComputing base scores for top-{top_n} selection...")
    base_scored_all = {}
    for qid, stage4 in stage4_results.items():
        bm25_scores = bm25_cache.get(qid)
        result = verify_and_score_single(stage4, verifier, bm25_scores=bm25_scores)
        base_scored_all[qid] = result["scored"]

    # ── Determine which queries need scoring ──────────────────────────────
    queries_to_score = []
    for qid in sorted(stage4_results.keys()):
        if qid not in existing_scores:
            queries_to_score.append(qid)
        elif len(existing_scores[qid]) < top_n:
            # Re-score if we now want more candidates than previously scored
            n_avail = len(base_scored_all.get(qid, []))
            n_existing = len(existing_scores[qid])
            if n_avail > n_existing:
                queries_to_score.append(qid)
                print(f"  {qid}: re-scoring ({n_existing} -> top-{min(top_n, n_avail)})")

    if not queries_to_score:
        print(f"\nAll {len(stage4_results)} queries already scored. Nothing to do.")
        elapsed = time.time() - t_total
        print(f"\nStage 4b done in {elapsed:.1f}s")
        return

    print(f"\n{len(queries_to_score)} queries to score.")

    # ── Load document texts ───────────────────────────────────────────────
    print("\nLoading article texts...")
    kb_texts = _load_texts()
    print(f"  {len(kb_texts):,} article texts loaded.")

    # Collect court citations needing text
    needed_court = set()
    for qid in queries_to_score:
        for c, _ in base_scored_all[qid][:top_n]:
            if not c.startswith("Art."):
                needed_court.add(c)

    court_texts = {}
    if needed_court:
        print(f"\nLoading court texts for {len(needed_court):,} citations...")
        court_texts = _load_court_texts(needed_court, kb_texts)
        print(f"  {len(court_texts):,} court texts loaded.")

    # ── Load cross-encoder model ──────────────────────────────────────────
    print("\nLoading Qwen3-Reranker-4B...")
    model, tokenizer, token_true_id, token_false_id, prefix_tokens, suffix_tokens = _load_reranker()
    print("  Reranker loaded.")

    # ── Score each query ──────────────────────────────────────────────────
    print(f"\nScoring {len(queries_to_score)} queries (top-{top_n} each)...")
    reranker_scores = dict(existing_scores)  # start from existing

    for qi, qid in enumerate(queries_to_score):
        query = query_texts.get(qid, "")
        top_cands = base_scored_all[qid][:top_n]

        # Build (query, doc) pairs
        pairs = []
        cite_list = []
        for c, _ in top_cands:
            text = _get_text(c, kb_texts, court_texts)
            doc = f"{c}: {text}" if text else c
            pairs.append(_format_pair(query, doc))
            cite_list.append(c)

        t_start = time.time()
        scores = _score_batch(
            pairs, model, tokenizer, token_true_id, token_false_id,
            prefix_tokens, suffix_tokens,
        )
        t_score = time.time() - t_start

        reranker_scores[qid] = dict(zip(cite_list, scores))

        has_text = sum(1 for c in cite_list if _get_text(c, kb_texts, court_texts))
        print(f"  [{qi+1}/{len(queries_to_score)}] {qid}: "
              f"{len(cite_list)} scored in {t_score:.0f}s ({has_text} with text)")

        # Save incrementally (resume-safe)
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(reranker_scores, f, ensure_ascii=False)

    elapsed = (time.time() - t_total) / 60
    n_total = sum(len(v) for v in reranker_scores.values())
    print(f"\nStage 4b complete: {len(reranker_scores)} queries, {n_total} citations scored.")
    print(f"  Saved to {out_path}")
    print(f"  Time: {elapsed:.1f} min")


### Stage 4b runner

In [ ]:
stage4b_outputs = run_stage4b(split=SPLIT, top_n=STAGE4B_TOP_N) if STAGE4B_TOP_N is not None else run_stage4b(split=SPLIT)

## Stage 5 — Verification, Confidence, and Thresholding

Final normalization, confidence scoring, and submission writing.

### `stage5_verify_and_score.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/stage5_verify_and_score.py"

"""
stage5_verify_and_score.py -- STAGE 5: Verification + Confidence Scoring + F1 Tuning

Three sub-stages:

  5A. VERIFICATION: Normalize citation formatting and check existence against
      the corpus. Drop hallucinated citations. Expand Art.-level to Abs.-level.

  5B. CONFIDENCE SCORING: Each citation gets a composite score combining:
      - Binary signal weights (explicit, graph, reranker tier, etc.)
      - Continuous BM25 relevance score (per-query normalized, 0-0.20)
      - Graduated multi-signal bonus (2+, 3+, 4+ independent signals)

  5C. THRESHOLD TUNING (val only): Sweep thresholds to maximize macro-F1.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#   python stage5_verify_and_score.py                    # val: verify + tune threshold
#   python stage5_verify_and_score.py --split test       # test: verify + apply threshold
#   python stage5_verify_and_score.py --threshold 0.20   # override threshold
#
# ─── PREREQUISITES ──────────────────────────────────────────────────────────
#   index/citation_lookup.pkl        (from stage0)
#   index/bm25_v2_index.pkl         (from stage0, for continuous BM25 scoring)
#   checkpoints/stage4_{split}.json  (from stage4)
#   For val: data/val.csv with gold_citations column
#
# ─── OUTPUTS ────────────────────────────────────────────────────────────────
#   checkpoints/stage5_{split}.json  (scored predictions)
#   submissions/submission_{split}.csv  (final submission file)
#
# ─── EXPECTED TIMING ────────────────────────────────────────────────────────
#   ~30s index load + <1 second per query + ~5 seconds for threshold sweep
"""

import argparse
import json
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).parent))

from data.data_paths import (
    CHECKPOINTS_DIR, VAL_CSV, TEST_CSV, TRAIN_CSV,
    LOOKUP_PKL, OUT_DIR, STATUTORY_BM25_PKL,
)
from agent.verifier import Verifier, normalize_citation
from scoring.confidence import score_all_citations, compute_f1, optimize_threshold
from retrieval.sparse_retriever import SparseRetriever, tokenise

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)


def _compute_bm25_scores(
    qid: str,
    query_text: str,
    all_citations: list[str],
    stage1_analysis: dict,
    sparse: SparseRetriever,
    verbose: bool = False,
) -> dict[str, float]:
    """
    Compute per-query normalized BM25 scores for candidate citations ONLY.

    FAST approach:
      1. Combine all query texts into one merged token list (no PMI enhancement)
      2. Call bm25.get_scores() ONCE (single pass over 2.15M docs)
      3. Look up only the indices of our candidate citations
      4. Normalize to [0, 1]

    Skipping PMI enhancement (which adds 50+ tokens) makes this ~6x faster.
    Using a single combined call instead of per-query calls makes it ~5x faster.
    Net: ~30x faster than the naive approach.
    """
    import numpy as np

    # Gather query texts: original + top DE translations
    de_queries = stage1_analysis.get("search_queries_de", [])[:3]
    en_queries = [query_text] + stage1_analysis.get("search_queries_en", [])[:2]
    all_queries = en_queries + de_queries

    # Merge all query tokens into one set (deduplicated), NO PMI enhancement
    merged_tokens = []
    seen = set()
    for q in all_queries:
        for t in tokenise(q):
            if t not in seen:
                merged_tokens.append(t)
                seen.add(t)

    if not merged_tokens:
        return {}

    if verbose:
        print(f"      {len(all_queries)} queries → {len(merged_tokens)} unique tokens")

    # Pre-resolve citation indices
    cite_indices: dict[str, int] = {}
    for cite in all_citations:
        idx = sparse._id_to_idx.get(cite, -1)
        if idx >= 0:
            cite_indices[cite] = idx
        else:
            cl = cite.lower()
            for canon_cite, canon_idx in sparse._id_to_idx.items():
                if canon_cite.lower() == cl:
                    cite_indices[cite] = canon_idx
                    break

    if not cite_indices:
        return {}

    # Single BM25 call with merged tokens — one pass over 2.15M docs
    if verbose:
        print(f"      Scoring {len(cite_indices)} citations (single BM25 call)...")

    raw = sparse._bm25.get_scores(merged_tokens)

    # Extract only our candidate citations' scores
    cite_names = list(cite_indices.keys())
    idx_array = np.array([cite_indices[c] for c in cite_names])
    scores = raw[idx_array]

    if verbose:
        print(f"      BM25 call done")

    # Normalize by max score → [0, 1]
    max_val = scores.max()
    if max_val <= 0:
        return {}

    normalized = {}
    for i, cite in enumerate(cite_names):
        if scores[i] > 0:
            normalized[cite] = float(scores[i] / max_val)

    return normalized


def _compute_and_cache_all_bm25_scores(
    split: str,
    stage4_results: dict,
    stage1_results: dict,
    query_texts: dict[str, str],
    sparse: SparseRetriever,
) -> dict[str, dict[str, float]]:
    """
    Compute BM25 scores for ALL queries and cache to disk.
    On subsequent runs, loads from cache instantly.

    Returns dict: qid -> {citation: normalized_score}
    """
    cache_path = CHECKPOINTS_DIR / f"bm25_scores_{split}.json"

    # Try loading from cache
    if cache_path.exists():
        print(f"  Loading cached BM25 scores from {cache_path}")
        import time
        t0 = time.time()
        with open(cache_path, encoding="utf-8") as f:
            cached = json.load(f)
        print(f"  Loaded {len(cached)} queries in {time.time()-t0:.1f}s")
        # Verify cache covers all queries
        missing = set(stage4_results.keys()) - set(cached.keys())
        if not missing:
            return cached
        print(f"  Cache missing {len(missing)} queries, computing them...")
    else:
        cached = {}

    # Compute missing queries
    import time
    total = len(stage4_results)
    for i, (qid, stage4) in enumerate(stage4_results.items()):
        if qid in cached:
            continue
        t0 = time.time()
        bm25_scores = _compute_bm25_scores(
            qid, query_texts.get(qid, ""),
            stage4.get("all_citations", []),
            stage1_results.get(qid, {}),
            sparse,
            verbose=False,
        )
        elapsed = time.time() - t0
        cached[qid] = bm25_scores
        n_scored = len(bm25_scores)
        n_total = len(stage4.get("all_citations", []))
        print(f"    [{i+1}/{total}] {qid}: {n_scored}/{n_total} scored in {elapsed:.1f}s")

        # Save incrementally
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(cached, f, ensure_ascii=False)

    print(f"  Cached to {cache_path}")
    return cached


def verify_and_score_single(
    stage4_result: dict,
    verifier: Verifier | None,
    confidence_weights: dict[str, float] | None = None,
    bm25_scores: dict[str, float] | None = None,
) -> dict:
    """
    Verify, normalize, and score all citations for a single query.

    Returns {
        'scored': [(citation, score), ...],  # sorted by score desc
        'verified': [...],
        'dropped': [...],
        'stats': {...}
    }
    """
    all_citations = stage4_result.get("all_citations", [])
    sources_raw = stage4_result.get("sources", {})
    reranked_tiers = stage4_result.get("reranked_tiers", {})

    # Convert source lists to sets
    source_sets: dict[str, set[str]] = {}
    for source_name in ["explicit_from_query", "bm25_top10", "bm25_initial",
                        "citation_graph", "co_citation",
                        "llm_reranker", "llm_reranker_tier3", "llm_direct_gen",
                        "llm_stage1", "llm_stage1_procedural",
                        "mas_rewrite", "mas_supplement", "mas_decompose",
                        "mas_supportive", "mas_crossref"]:
        source_sets[source_name] = set()

    for cite, cite_sources in sources_raw.items():
        for s in cite_sources:
            if s in source_sets:
                source_sets[s].add(cite)
            # Map MAS sources to generic "bm25_initial" for scoring
            if s.startswith("mas_"):
                source_sets.setdefault("bm25_initial", set()).add(cite)

    # Inject reranker tier data from Stage 4's tiered output
    # Handles both old format (tier 2/3) and new format (0-10 scores)
    for cite, score_val in reranked_tiers.items():
        if not isinstance(score_val, (int, float)):
            continue
        score_val = int(score_val)
        # Map 0-10 scores: >=8 → tier3, >=4 → tier2 (also works for old 2/3 format)
        if score_val >= 3:  # tier 3 in old format, or 3+ in new
            source_sets.setdefault("llm_reranker_tier3", set()).add(cite)
        if score_val >= 2:  # tier 2+ in old format, or 2+ in new
            source_sets.setdefault("llm_reranker", set()).add(cite)

    # 5A: Verification
    if verifier:
        verified, dropped = verifier.verify_and_normalize(all_citations)
    else:
        verified = all_citations
        dropped = []

    # Remap source sets: verifier may normalize citation strings (e.g. STPO→StPO),
    # so we need to ensure the sources dict uses the post-verification forms.
    _verified_lower_to_verified = {c.lower(): c for c in verified}
    for source_name, cite_set in source_sets.items():
        remapped = set()
        for c in cite_set:
            cl = c.lower()
            if cl in _verified_lower_to_verified:
                remapped.add(_verified_lower_to_verified[cl])
            else:
                remapped.add(c)
        source_sets[source_name] = remapped

    # Remap BM25 scores to use post-verification citation forms
    bm25_remapped = None
    if bm25_scores:
        bm25_remapped = {}
        for c, s in bm25_scores.items():
            cl = c.lower()
            if cl in _verified_lower_to_verified:
                bm25_remapped[_verified_lower_to_verified[cl]] = max(
                    bm25_remapped.get(_verified_lower_to_verified[cl], 0.0), s
                )
            else:
                bm25_remapped[c] = max(bm25_remapped.get(c, 0.0), s)

    # 5B: Confidence scoring
    scored = score_all_citations(verified, source_sets, weights=confidence_weights,
                                 bm25_scores=bm25_remapped)

    return {
        "scored": scored,
        "verified": verified,
        "dropped": dropped,
        "stats": {
            "input_citations": len(all_citations),
            "verified": len(verified),
            "dropped": len(dropped),
        },
    }


def run_batch(
    split: str,
    threshold: float | None = None,
    confidence_weights: dict[str, float] | None = None,
) -> pd.DataFrame:
    """
    Run Stage 5 on all queries in a split.

    For val: tunes threshold on gold data.
    For test: applies provided threshold (or default).
    """
    # Load Stage 4 results
    stage4_path = CHECKPOINTS_DIR / f"stage4_{split}.json"
    if not stage4_path.exists():
        print(f"ERROR: Stage 4 results not found: {stage4_path}")
        print("Run first: python stage4_llm_reranker.py")
        sys.exit(1)
    with open(stage4_path, encoding="utf-8") as f:
        stage4_results = json.load(f)

    # Load verifier
    verifier = None
    if LOOKUP_PKL.exists():
        print("Loading citation verifier...")
        verifier = Verifier()
    else:
        print("WARNING: Lookup tables not found. Skipping verification.")

    # Load BM25 index for continuous scoring
    sparse = None
    if STATUTORY_BM25_PKL.exists():
        print("Loading BM25 index for continuous scoring...")
        sparse = SparseRetriever()
    else:
        print("WARNING: BM25 index not found. Using binary scoring only.")

    # Load query data
    csv_map = {"train": TRAIN_CSV, "val": VAL_CSV, "test": TEST_CSV}
    df = pd.read_csv(csv_map[split])

    # Load Stage 1 results (for query text → BM25 scoring)
    stage1_path = CHECKPOINTS_DIR / f"stage1_{split}.json"
    stage1_results = {}
    if stage1_path.exists():
        with open(stage1_path, encoding="utf-8") as f:
            stage1_results = json.load(f)

    # Build query_id -> query_text mapping
    query_texts = {row["query_id"]: row["query"] for _, row in df.iterrows()}

    # ── Precompute BM25 scores (cached to disk) ─────────────────────────
    all_bm25_scores = {}
    if sparse:
        print("\nPrecomputing BM25 scores (cached to disk)...")
        all_bm25_scores = _compute_and_cache_all_bm25_scores(
            split, stage4_results, stage1_results, query_texts, sparse,
        )

    # ── Load cross-encoder scores from Stage 4b (Strategy C) ──────────
    all_ce_scores: dict[str, dict[str, float]] = {}
    ce_path = CHECKPOINTS_DIR / f"reranker_scores_{split}.json"
    if ce_path.exists():
        print(f"\nLoading cross-encoder scores from {ce_path}...")
        with open(ce_path, encoding="utf-8") as f:
            all_ce_scores = json.load(f)
        n_scored = sum(len(v) for v in all_ce_scores.values())
        print(f"  Loaded CE scores for {len(all_ce_scores)} queries ({n_scored} citations)")
    else:
        print(f"\nNo cross-encoder scores found at {ce_path} — using binary reranker only.")

    # ── Process all queries ──────────────────────────────────────────────
    print(f"\nScoring {len(stage4_results)} queries...")
    all_scored: dict[str, list[tuple[str, float]]] = {}
    all_results: dict[str, dict] = {}

    total_verified = 0
    total_dropped = 0

    for qid, stage4 in stage4_results.items():
        bm25_scores = all_bm25_scores.get(qid)
        result = verify_and_score_single(stage4, verifier, confidence_weights,
                                          bm25_scores=bm25_scores)
        all_scored[qid] = result["scored"]
        all_results[qid] = result
        total_verified += result["stats"]["verified"]
        total_dropped += result["stats"]["dropped"]

    print(f"  Total verified: {total_verified}, dropped: {total_dropped}")

    # ── 5B+: Cross-encoder Strategy C (post-processing) ────────────────
    # Exactly replicates the validated test (test_reranker_all10.py Strategy C):
    #   1. Subtract binary reranker weight (0.25 tier3, 0.15 tier2)
    #   2. Add cross-encoder P(yes) * weight
    # Uses sources_raw (pre-verification keys) for subtraction — this matches
    # the test behavior where key normalization mismatches are expected.
    if all_ce_scores:
        from scoring.confidence import DEFAULT_WEIGHTS

        # Build case-insensitive CE lookup per query
        ce_lower: dict[str, dict[str, float]] = {}
        for qid, ce_dict in all_ce_scores.items():
            ce_lower[qid] = {k.lower(): v for k, v in ce_dict.items()}

        def _apply_strategy_c(base_scored, stage4_data, ce_scores_lower, w_ce):
            """Apply Strategy C: subtract binary reranker, add CE * weight."""
            adjusted_scored = {}
            for qid, scored_list in base_scored.items():
                sources_raw = stage4_data[qid].get("sources", {})
                ce_q = ce_scores_lower.get(qid, {})
                new_list = []
                for c, base_s in scored_list:
                    cite_sources = set(sources_raw.get(c, []))
                    adjusted = base_s
                    if "llm_reranker_tier3" in cite_sources:
                        adjusted -= DEFAULT_WEIGHTS.get("llm_reranker_tier3", 0.25)
                    elif "llm_reranker" in cite_sources:
                        adjusted -= DEFAULT_WEIGHTS.get("llm_reranker", 0.15)
                    adjusted = max(0.0, adjusted)
                    rk_s = ce_q.get(c.lower(), 0.0)
                    adjusted += rk_s * w_ce
                    new_list.append((c, min(adjusted, 1.0)))
                new_list.sort(key=lambda x: -x[1])
                adjusted_scored[qid] = new_list
            return adjusted_scored

        # Save base scores for sweeping (don't mutate yet)
        base_scored_copy = {qid: list(scored) for qid, scored in all_scored.items()}

        if split == "val":
            print("\nSweeping cross-encoder Strategy C weight...")

            val_gold_sweep: dict[str, set[str]] = {}
            for _, row in df.iterrows():
                qid_r = row["query_id"]
                raw = str(row.get("gold_citations", "") or "")
                gold_s = {c.strip() for c in raw.split(";") if c.strip()}
                if gold_s:
                    val_gold_sweep[qid_r] = gold_s

            base_thresh, base_f1 = optimize_threshold(all_scored, val_gold_sweep)
            print(f"  Base (no CE): macro-F1={base_f1:.4f} @ thresh={base_thresh:.2f}")

            best_ce_w, best_ce_f1, best_ce_thresh = 0.0, base_f1, base_thresh
            for w_int in range(5, 61):
                w_ce = w_int / 100.0
                adjusted = _apply_strategy_c(
                    base_scored_copy, stage4_results, ce_lower, w_ce)
                t, f1 = optimize_threshold(adjusted, val_gold_sweep)
                if f1 > best_ce_f1:
                    best_ce_f1 = f1
                    best_ce_w = w_ce
                    best_ce_thresh = t

            print(f"  Best CE weight: {best_ce_w:.2f}, macro-F1={best_ce_f1:.4f} "
                  f"@ thresh={best_ce_thresh:.2f} (delta={best_ce_f1 - base_f1:+.4f})")

            if best_ce_w > 0:
                all_scored = _apply_strategy_c(
                    base_scored_copy, stage4_results, ce_lower, best_ce_w)

                # Diagnostic: count how many citations got CE boost vs subtracted
                n_ce_hit = 0
                n_sub = 0
                for qid in all_scored:
                    sources_raw = stage4_results[qid].get("sources", {})
                    ce_q = ce_lower.get(qid, {})
                    for c, _ in base_scored_copy[qid]:
                        if ce_q.get(c.lower(), 0.0) > 0:
                            n_ce_hit += 1
                        cite_src = set(sources_raw.get(c, []))
                        if "llm_reranker_tier3" in cite_src or "llm_reranker" in cite_src:
                            n_sub += 1
                print(f"  Diagnostic: {n_ce_hit} citations got CE boost, "
                      f"{n_sub} had reranker subtracted (via sources_raw key match)")

                ce_config = {
                    "ce_weight": best_ce_w, "ce_threshold": best_ce_thresh,
                    "strategy": "C",
                }
                ce_config_path = CHECKPOINTS_DIR / "ce_config.json"
                with open(ce_config_path, "w") as f:
                    json.dump(ce_config, f)
                print(f"  Saved CE config to {ce_config_path}")
        else:
            ce_config_path = CHECKPOINTS_DIR / "ce_config.json"
            if ce_config_path.exists():
                with open(ce_config_path) as f:
                    ce_config = json.load(f)
                w_ce = ce_config["ce_weight"]
                print(f"\nApplying CE Strategy C with weight={w_ce:.2f} (from val tuning)")
                all_scored = _apply_strategy_c(
                    base_scored_copy, stage4_results, ce_lower, w_ce)
            else:
                print(f"\nWARNING: CE scores found but no ce_config.json — "
                      f"run val split first to tune CE weight.")

    # ── 5C: Threshold tuning (val only) ─────────────────────────────────
    if split == "val" and threshold is None:
        print("\nTuning threshold on val gold...")
        val_gold: dict[str, set[str]] = {}
        for _, row in df.iterrows():
            qid = row["query_id"]
            raw = str(row.get("gold_citations", "") or "")
            gold = {c.strip() for c in raw.split(";") if c.strip()}
            if gold:
                val_gold[qid] = gold

        best_thresh, best_f1 = optimize_threshold(all_scored, val_gold)
        print(f"  Best threshold: {best_thresh:.2f}")
        print(f"  Best macro-F1:  {best_f1:.4f}")
        threshold = best_thresh

        # Also show per-query F1 at best threshold
        print(f"\n  Per-query F1 @ threshold={threshold:.2f}:")
        f1s = []
        for qid, scored in all_scored.items():
            preds = [c for c, s in scored if s >= threshold]
            gold = val_gold.get(qid, set())
            if gold:
                f = compute_f1(preds, gold)
                f1s.append(f)
                print(f"    {qid}: F1={f:.4f} (pred={len(preds)}, gold={len(gold)})")
        macro = sum(f1s) / len(f1s) if f1s else 0.0
        print(f"\n  Macro F1: {macro:.4f}")

    if threshold is None:
        threshold = 0.15  # Default: low threshold for high recall
        print(f"\nUsing default threshold: {threshold:.2f}")

    # ── Generate submission ──────────────────────────────────────────────
    print(f"\nGenerating submission with threshold={threshold:.2f}...")
    rows = []
    for _, row in df.iterrows():
        qid = row["query_id"]
        scored = all_scored.get(qid, [])
        preds = [c for c, s in scored if s >= threshold]
        rows.append({
            "query_id": qid,
            "predicted_citations": ";".join(preds),
        })

    out_df = pd.DataFrame(rows)
    out_csv = OUT_DIR / f"submission_{split}.csv"
    out_df.to_csv(out_csv, index=False)
    print(f"  Saved: {out_csv}")

    # Save detailed results
    out_detail = CHECKPOINTS_DIR / f"stage5_{split}.json"
    detail = {
        "threshold": threshold,
        "results": {qid: {
            "scored": r["scored"],
            "stats": r["stats"],
        } for qid, r in all_results.items()},
    }
    with open(out_detail, "w", encoding="utf-8") as f:
        json.dump(detail, f, ensure_ascii=False, indent=2)
    print(f"  Saved: {out_detail}")

    # Summary stats
    n_preds = out_df["predicted_citations"].apply(
        lambda x: len([c for c in str(x).split(";") if c.strip()]) if pd.notna(x) else 0
    )
    print(f"\n  Submission summary:")
    print(f"    Queries: {len(out_df)}")
    print(f"    Avg citations: {n_preds.mean():.1f}")
    print(f"    Max citations: {n_preds.max()}")
    print(f"    Queries with 0: {(n_preds == 0).sum()}")

    return out_df


### Stage 5 runner

In [ ]:
stage5_outputs = run_stage5(split=SPLIT, threshold=STAGE5_THRESHOLD)

submission_path = get_submission_path(SPLIT)
print("Generated submission:", submission_path)


## Pipeline Orchestration

Optional one-shot entrypoint if you prefer not to run stage cells manually.

### `pipeline.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/pipeline.py"

"""
pipeline.py -- Master orchestrator for the full 5-stage pipeline.

Runs all stages sequentially for a given split (val/test).
Each stage reads from the previous stage's checkpoint file.

# ─── HOW TO RUN ─────────────────────────────────────────────────────────────
#   cd E:\\swiss-law-pipeline
#
#   # First time: build indexes (one-time, ~4-7 hours)
#   python stage0_build_index.py
#
#   # Then run the full pipeline:
#   python pipeline.py --split val             # evaluate on val
#   python pipeline.py --split test            # generate test submission
#   python pipeline.py --split val --stage 2   # resume from stage 2
#
# ─── STAGES ─────────────────────────────────────────────────────────────────
#   Stage 0: Offline index building (BM25 + graph + lookup) — run separately
#   Stage 1: LLM query analysis (GPU, ~7 min for 40 queries)
#   Stage 2: MAS multi-agent BM25 retrieval (GPU+CPU, ~20 min)
#   Stage 3: Citation graph expansion (CPU, ~30 sec)
#   Stage 4: LLM reranking + direct citation gen (GPU, ~10 min)
#   Stage 4b: Cross-encoder scoring with Qwen3-Reranker-4B (GPU, ~25 min)
#   Stage 5: Verification + scoring + F1 threshold (CPU, ~30 sec)
#   Total: ~65-105 min for 40 queries
#
# ─── PREREQUISITES ──────────────────────────────────────────────────────────
#   index/bm25_v2_index.pkl + bm25_v2_ids.pkl    (from stage0)
#   index/citation_graph.pkl                       (from stage0)
#   index/citation_lookup.pkl                      (from stage0)
#   GPU: RTX 4050 6GB with Qwen3-4B Q4 (~2.8 GB VRAM)
"""

import argparse
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

from data.data_paths import (
    STATUTORY_BM25_PKL, CITATION_GRAPH_PKL, LOOKUP_PKL,
    CHECKPOINTS_DIR, OUT_DIR,
)


def check_prerequisites():
    """Check that all Stage 0 outputs exist."""
    missing = []
    for path, name in [
        (STATUTORY_BM25_PKL, "BM25 index"),
        (CITATION_GRAPH_PKL, "Citation graph"),
        (LOOKUP_PKL, "Lookup tables"),
    ]:
        if not path.exists():
            missing.append(f"  {name}: {path}")

    if missing:
        print("ERROR: Missing prerequisites (run stage0_build_index.py first):")
        for m in missing:
            print(m)
        return False
    return True


def run_pipeline(
    split: str,
    start_stage: int = 1,
    backend: str = "local",
    stage3_gold_prior_mode: str = "off",
):
    """Run the full pipeline from start_stage through stage 5."""
    t_total = time.time()

    if not check_prerequisites():
        sys.exit(1)

    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # ── Stage 1: Query Analysis ──────────────────────────────────────────
    if start_stage <= 1:
        print("\n" + "=" * 70)
        print("  STAGE 1: LLM Query Analysis")
        print("=" * 70)
        t0 = time.time()
        from stage1_query_analysis import run_batch as run_stage1
        run_stage1(split, backend=backend)
        print(f"  Stage 1 done in {(time.time()-t0)/60:.1f} min")

    # ── Stage 2: MAS Multi-Agent Retrieval ───────────────────────────────
    if start_stage <= 2:
        print("\n" + "=" * 70)
        print("  STAGE 2: Multi-Agent Sparse Retrieval")
        print("=" * 70)
        t0 = time.time()
        from stage2_mas_retrieval import run_batch as run_stage2
        run_stage2(split, backend=backend)
        print(f"  Stage 2 done in {(time.time()-t0)/60:.1f} min")

    # ── Stage 3: Citation Graph Expansion ────────────────────────────────
    if start_stage <= 3:
        print("\n" + "=" * 70)
        print("  STAGE 3: Citation Graph Expansion")
        print("=" * 70)
        t0 = time.time()
        from stage3_graph_expansion import run_batch as run_stage3
        run_stage3(split, gold_prior_mode=stage3_gold_prior_mode)
        print(f"  Stage 3 done in {time.time()-t0:.1f}s")

    # ── Stage 4: LLM Reranker + Direct Gen ───────────────────────────────
    if start_stage <= 4:
        print("\n" + "=" * 70)
        print("  STAGE 4: LLM Reranker + Direct Citation Generation")
        print("=" * 70)
        t0 = time.time()
        from stage4_llm_reranker import run_batch as run_stage4
        run_stage4(split, backend=backend)
        print(f"  Stage 4 done in {(time.time()-t0)/60:.1f} min")

    # ── Stage 4b: Cross-Encoder Scoring ───────────────────────────────────
    if start_stage <= 4:
        print("\n" + "=" * 70)
        print("  STAGE 4b: Cross-Encoder Scoring (Qwen3-Reranker-4B)")
        print("=" * 70)
        t0 = time.time()
        from stage4b_cross_encoder import run_batch as run_stage4b
        run_stage4b(split)
        print(f"  Stage 4b done in {(time.time()-t0)/60:.1f} min")

    # ── Stage 5: Verification + Scoring + Threshold ──────────────────────
    if start_stage <= 5:
        print("\n" + "=" * 70)
        print("  STAGE 5: Verification + Confidence Scoring + Threshold")
        print("=" * 70)
        t0 = time.time()
        from stage5_verify_and_score import run_batch as run_stage5
        run_stage5(split)
        print(f"  Stage 5 done in {time.time()-t0:.1f}s")

    # ── Summary ──────────────────────────────────────────────────────────
    elapsed = (time.time() - t_total) / 60
    print("\n" + "=" * 70)
    print(f"  PIPELINE COMPLETE — {split} set — {elapsed:.1f} min total")
    print("=" * 70)
    sub = OUT_DIR / f"submission_{split}.csv"
    if sub.exists():
        print(f"  Submission: {sub}")


### Optional pipeline runner

In [ ]:
# Alternative to the stage-by-stage runner cells above:
# run_full_pipeline(
#     split=SPLIT,
#     start_stage=START_STAGE,
#     backend=BACKEND,
#     stage3_gold_prior_mode=STAGE3_GOLD_PRIOR_MODE,
# )


## Submission Creation

Submission utilities and validation.

### `submit.py`

In [ ]:
__file__ = r"/mnt/data/swiss_law_pipeline_unzipped/swiss_law_pipeline/submit.py"

"""
submit.py -- Generate and validate a competition submission CSV.

# HOW TO RUN:
#   cd E:\\swiss-law-pipeline
#   python submit.py                                    # run pipeline on test
#   python submit.py --validate submissions/submission_val.csv  # validate only
#   python submit.py --from-checkpoint                  # build from stage5 checkpoint
"""

import argparse
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).parent))
from data.data_paths import TEST_CSV, OUT_DIR, CHECKPOINTS_DIR

OUT_DIR.mkdir(parents=True, exist_ok=True)


def validate_submission(submission_path: Path) -> bool:
    """Validate submission format against test query IDs."""
    print(f"Validating: {submission_path}")

    try:
        sub = pd.read_csv(submission_path)
    except Exception as e:
        print(f"  ERROR: Cannot read CSV: {e}")
        return False

    required_cols = {"query_id", "predicted_citations"}
    missing = required_cols - set(sub.columns)
    if missing:
        print(f"  ERROR: Missing columns: {missing}")
        return False

    test_df = pd.read_csv(TEST_CSV)
    test_ids = set(test_df["query_id"].astype(str))
    sub_ids = set(sub["query_id"].astype(str))

    missing_ids = test_ids - sub_ids
    extra_ids = sub_ids - test_ids

    if missing_ids:
        print(f"  ERROR: Missing {len(missing_ids)} test query IDs")
        return False
    if extra_ids:
        print(f"  WARNING: {len(extra_ids)} extra query IDs")

    sub["n_citations"] = sub["predicted_citations"].apply(
        lambda x: len([c for c in str(x).split(";") if c.strip()]) if pd.notna(x) else 0
    )
    print(f"  Queries:        {len(sub)}")
    print(f"  Avg citations:  {sub['n_citations'].mean():.1f}")
    print(f"  Max citations:  {sub['n_citations'].max()}")
    print(f"  Queries with 0: {(sub['n_citations'] == 0).sum()}")
    print("  VALID")
    return True


### Submission validation

In [ ]:
submission_path = get_submission_path(SPLIT)
print("Submission path:", submission_path)

if submission_path.exists():
    submission_valid = validate_generated_submission(SPLIT)
    print("Submission valid:", submission_valid)
else:
    print("Submission file does not exist yet.")
